# OCR trọn một PDF · đối chiếu bốn model · tải kết quả offline

Điền đường dẫn ở bước 1 rồi **Run All**. Bốn model dùng cùng tập trang và chạy lần lượt.
Mặc định một PDF, toàn bộ trang. Mỗi model nạp một lần, xử lý hết sách rồi giải phóng GPU.
Kết quả từng trang được lưu ngay; chạy lại chỉ bổ sung trang thiếu. Render ảnh được dùng chung.
Sau OCR: so sánh, tạo HTML theo trang, cắt vùng bất đồng và đóng gói ZIP.
Chỉ worker OCR được ghi ra một file Python để chạy ở tiến trình riêng và giải phóng GPU.


## 1. Cấu hình

Gắn 3 Dataset riêng và chỉ điền 3 đường dẫn gốc: `PACKAGES_DIR`, `MODELS_DIR`, `DATA_DIR`.
Packages chứa `paddle_manifest.json`; models chứa `hunyuan`, `ppocr_det`, `tonggu`...;
data chứa các PDF. Notebook tự ghép `WEIGHTS` từ `MODELS_DIR` theo cấu trúc đã chuẩn bị.
Nếu Dataset có thêm folder bọc `packages/` hoặc `models/`, notebook cũng nhận được.
Đặt `RUN_OCR=False` để đọc checkpoint ở `WORK_DIR`.
Hoặc điền `EXPORTED_LOG_DIR` trỏ tới `data/log_notebook_3` (hay thư mục một sách có `raw/`).
Log gốc được giữ nguyên; notebook tạo bản checkpoint riêng trong `WORK_DIR/imported_logs`.
`DATA_DIR` chứa PDF gốc để xác minh SHA-256 và cắt ảnh, không cần model cho chế độ này.
`BOOKS=["1"]`, `MAX_BOOKS=1`, `PAGES=""`, `LIMIT=None`: toàn bộ một PDF.
Thử nhanh bằng `PAGES="1-2"`, `LIMIT=2`. Đặt `EXPORT_RESULTS=True` để xuất từ checkpoint.
`IMAGE_PREPARATION` kiểm soát tách vùng và kích thước ảnh OCR. Preview không cần model:
```python
cfg = OCRConfig(engine="hunyuan", **BOOK_PROFILES["1.json"], **IMAGE_PREPARATION)
preview, audit = preview_preparation("data/1.pdf", 50, cfg)
display(preview)
display(audit)
```
Chọn trang có chữ/chú thích; kiểm tra đường cắt trước khi chạy cả sách.
Ảnh nguồn DPI nhỏ không đồng nghĩa thiếu pixel: xem `configs/ocr_books/README.md`.

Để dùng Dataset ảnh đã chuẩn bị, điền `PREPARED_IMAGES`, chọn `PREPARED_VARIANT`
và `PREPARED_CROP_TYPE`. Trang có ảnh hợp lệ không render PDF. Mặc định thiếu trang
báo lỗi; fallback phải bật riêng. Hash snapshot/variant/crop/model tách checkpoint.
Nguồn, mapping và raw output từng vùng vẫn được ghi. Dataset giữ pixel nguồn;
giới hạn processor chỉ áp dụng lúc inference. Xem `docs/image_dataset.md` để chuẩn
bị/preview/resume/đóng gói Kaggle; thử 1-2 trang trước khi chuẩn bị cả tập.


In [ ]:
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
PACKAGES_DIR = ""  # điền thư mục chứa paddle_manifest.json trong Dataset packages
MODELS_DIR = ""  # Dataset models: thư mục chứa hunyuan, ppocr_det, tonggu...
DATA_DIR = "" if ON_KAGGLE else "data"  # điền thư mục PDF trong Dataset đã upload
WORK_DIR = Path("/kaggle/working/four_model_consensus") if ON_KAGGLE else Path("work/four_model_consensus")

# Điền đường dẫn thực tế; để chuỗi rỗng cho tới khi gắn Dataset models.
# Tự ghép đường dẫn theo cấu trúc Dataset models đã chuẩn bị.
_model_folders = {
    "hunyuan": "hunyuan",
    "ppocr_det": "ppocr_det",
    "ppocr_rec": "ppocr_rec",
    "paddle_vl": "paddle_vl",
    "paddle_layout": "paddle_layout",
    "tonggu_ocr": "tonggu/ocr",
    "tonggu_layout": "tonggu/layout/finetune_infer.pth",
}
_models_root = Path(MODELS_DIR) if MODELS_DIR else None
if _models_root and not (_models_root / "hunyuan").is_dir() and (_models_root / "models/hunyuan").is_dir():
    _models_root = _models_root / "models"
WEIGHTS = {name: _models_root / folder for name, folder in _model_folders.items()} if _models_root else {}
BOOKS = ["1"]
MAX_BOOKS = 1
PAGES = ""  # toàn bộ PDF; dùng "1-2" để thử nhanh
LIMIT = None
RUN_OCR = ON_KAGGLE

REVIEW_BOOK = None  # None: chọn trang đầu cần rà soát
REVIEW_PAGE = None  # số trang PDF, bắt đầu từ 1
REVIEW_VIEW = "regions"  # regions: cụm bất đồng; characters: từng chữ
MAX_REVIEW_ROWS = 20
ADVANCED = {}  # tùy chọn nâng cao: xem docs/four_model_consensus.md

# Profile theo sách; ADVANCED["use_book_profiles"] = False để bỏ qua.
BOOK_PROFILES = {'1.json': {'books': ['1.pdf'],
            'dpi': 225,
            'max_side': 750,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '2.json': {'books': ['2.pdf'],
            'dpi': 226,
            'max_side': 754,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '3.json': {'books': ['3.pdf'],
            'dpi': 226,
            'max_side': 754,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '4.json': {'books': ['4.pdf'],
            'dpi': 181,
            'max_side': 704,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '5.json': {'books': ['5.pdf'],
            'dpi': 181,
            'max_side': 704,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '6.json': {'books': ['6.pdf'],
            'dpi': 226,
            'max_side': 653,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 'bien.json': {'books': ['bien.pdf'],
               'dpi': 200,
               'max_side': 2333,
               'max_new_tokens': 6000,
               'prompt': '',
               'reading_direction': 'horizontal_ltr'},
 'kinh.json': {'books': ['kinh.pdf'],
               'dpi': 45,
               'max_side': 2206,
               'max_new_tokens': 6000,
               'prompt': '',
               'reading_direction': 'vertical_rtl'}}

# Xuất sau khi bốn model hoàn tất. Giữ checkpoint riêng để tiếp tục khi bị ngắt.
OUTPUT_DIR = WORK_DIR / "output"
EXPORT_RESULTS = RUN_OCR  # đặt True khi cần xuất từ checkpoint với RUN_OCR=False
EXPORT_CROPS = True
CREATE_ZIP = True
PREVIEW_CONFLICTS = False  # HTML đã có ảnh; bật nếu muốn xem thêm ngay trong notebook

# So sánh lại log đã tải về; để "" nếu đọc checkpoint ở WORK_DIR.
# Không chạy OCR, không cần weights/packages khi dùng thư mục log này.
EXPORTED_LOG_DIR = ""  # ví dụ: "data/log_notebook_3"; dùng với RUN_OCR=False

# Ảnh OCR: crop nguyên pixel; không tăng nét/nhị phân hóa ảnh nguồn.
# auto: Hunyuan tách nhóm cột dọc; Paddle-VL/TongGu dùng layout gốc;
# PP-OCR chỉ tách thêm khi vượt ngân sách pixel/cạnh dài.
IMAGE_PREPARATION = {
    "segmentation": "auto",  # none: đối chứng cả trang; columns: ép tách cột
    "segmentation_args": {"columns_per_region": 3, "max_regions": 64},
    "region_upscale": 1.0,  # nội suy 2.0 không tạo thêm chi tiết thật
    "region_max_pixels": 0,  # 0: ngân sách riêng theo engine
    "region_max_side": 0,    # 0: không thêm cap cạnh dài cho crop
    "resolution_strict": False,  # True: lỗi trang nếu không giữ được pixel
}
# Giới hạn processor có thể đổi qua ADVANCED["model_overrides"]. Ví dụ:
# ADVANCED = {"model_overrides": {"hunyuan": {"model_max_pixels": 4194304}}}
# Tách thủ công trang 50 tại 1/2 chiều ngang:
# IMAGE_PREPARATION["segmentation_args"]["page_column_cuts"] = {"50": [0.5]}

# Dataset ảnh đã chuẩn bị offline. Để rỗng để dùng workflow render PDF hiện tại.
# Có thể trỏ tới dataset.json hoặc snapshots/<sha256>.json trong Kaggle input.
PREPARED_IMAGES = ""
PREPARED_VARIANT = "original"  # original | clahe | bicubic_x2 (phải có trong Dataset)
PREPARED_CROP_TYPE = "group"  # page | group | column: cùng crop cho bốn model
# native: dùng cả trang, giữ layout riêng của Paddle/TongGu.
PREPARED_MISSING_FALLBACK = False  # True: cho phép render riêng trang chưa chuẩn bị


## 2. Các hàm xử lý trực tiếp trong notebook

Ô đầu định nghĩa cấu hình model, chọn trang, điều phối worker, đọc checkpoint và so sánh.
Ô tiếp theo chứa các hàm setup dependency offline của Paddle/TongGu. Không tải repo hay
cài wheel mã dự án. Bạn có thể sửa trực tiếp các hàm và tham số trong notebook.


In [ ]:
from __future__ import annotations
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
from types import SimpleNamespace
from IPython.display import display, FileLink

NOTEBOOK_CODE_ROOT = Path(globals().get("NOTEBOOK_CODE_ROOT_OVERRIDE",
    "/tmp/vtm_notebook_worker" if ON_KAGGLE else "work/notebook_worker")).resolve()
NOTEBOOK_CODE_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_WORKER_FILE = NOTEBOOK_CODE_ROOT / "ocr_worker.py"
NOTEBOOK_PROFILE_DIR = NOTEBOOK_CODE_ROOT / "profiles"

import dataclasses
import hashlib
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
HUNYUAN_PROMPT = '提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，表格用html格式表达，文档中公式用latex格式表示，按照阅读顺序组织进行解析。'
LAYOUT_PROMPT = '按照阅读顺序解析图中的版式信息。'

def _json_default(o: Any):
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

def config_hash(cfg: Any, exclude: set[str]) -> str:
    data = {k: v for k, v in dataclasses.asdict(cfg).items() if k not in exclude}
    raw = json.dumps(data, ensure_ascii=False, sort_keys=True, default=_json_default)
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()

@dataclass
class OCRConfig:
    """Cấu hình bước OCR. Mọi trường đều có thể sửa từ notebook."""
    engine: str = 'hunyuan'
    input_dir: str = 'data'
    books: list[str] = field(default_factory=list)
    workdir: str = 'work'
    run_name: str = ''
    pages: str = ''
    dpi: int = 200
    max_side: int = 1280
    segmentation: str = 'none'
    reading_direction: str = 'vertical_rtl'
    segmentation_args: dict = field(default_factory=dict)
    region_max_pixels: int = 0
    region_max_side: int = 0
    region_upscale: float = 1.0
    resolution_strict: bool = False
    model_min_pixels: int | None = None
    model_max_pixels: int | None = None
    image_dataset: str = ''
    image_dataset_key: str = ''
    image_variant: str = 'original'
    image_crop_type: str = 'group'
    image_dataset_fallback: bool = False
    max_new_tokens: int = 4096
    dtype: str = 'fp16'
    model: str = 'tencent/HunyuanOCR'
    model_revision: str = ''
    prompt: str = ''
    repetition_penalty: float = 1.08
    device: str = 'auto'
    paddle_version: str = 'v1.6'
    paddle_args: dict = field(default_factory=dict)
    tonggu_repo: str = ''
    tonggu_config: str = ''
    limit: int | None = None
    page_timeout: int = 300
    n_gpu: int = 0
    worker_python: str = ''
    worker_wsl_distro: str = ''
    worker_pythonpath: list[str] = field(default_factory=list)
    budget_h: float = 10.5
    stall_min: int = 15
    max_restarts: int = 5
    n_layout: int = 0
    layout_budget_h: float = 1.0
    layout_prompt: str = LAYOUT_PROMPT
    export: bool = True
    fresh: bool = False
    resume_from: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        if self.engine not in ('hunyuan', 'paddle', 'ppocr', 'tonggu', 'dummy'):
            raise ValueError(f'engine không hợp lệ: {self.engine}')
        if self.dtype not in ('fp32', 'fp16', 'bf16'):
            raise ValueError(f'dtype không hợp lệ: {self.dtype}')
        if self.dpi <= 0 or self.max_side <= 0 or self.max_new_tokens <= 0:
            raise ValueError('dpi, max_side, max_new_tokens phải > 0')
        if self.segmentation not in {'none', 'auto', 'columns'}:
            raise ValueError('segmentation must be none, auto or columns')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('reading_direction must be vertical_rtl or horizontal_ltr')
        if self.region_max_pixels < 0 or self.region_max_side < 0 or (not 1 <= self.region_upscale <= 4):
            raise ValueError('region caps must be >= 0; region_upscale must be in [1, 4]')
        if any((v is not None and v <= 0 for v in (self.model_min_pixels, self.model_max_pixels))):
            raise ValueError('model pixel budgets must be positive or None')
        if self.model_min_pixels is not None and self.model_max_pixels is not None and (self.model_min_pixels > self.model_max_pixels):
            raise ValueError('model_min_pixels must not exceed model_max_pixels')
        if not isinstance(self.segmentation_args, dict):
            raise ValueError('segmentation_args must be a dictionary')
        if self.image_crop_type not in {'page', 'group', 'column', 'native'}:
            raise ValueError('image_crop_type must be page, group, column or native')
        if self.image_variant not in {'original', 'clahe', 'bicubic_x2'}:
            raise ValueError('Unsupported image_variant')
        if not self.image_dataset and (self.image_variant != 'original' or self.image_dataset_key):
            raise ValueError('Image variants/snapshot identity require image_dataset')

    def cache_key(self) -> str:
        exclude = {'input_dir', 'workdir', 'run_name', 'limit', 'page_timeout', 'n_gpu', 'worker_python', 'worker_wsl_distro', 'worker_pythonpath', 'budget_h', 'stall_min', 'max_restarts', 'n_layout', 'layout_budget_h', 'export', 'fresh', 'resume_from'}
        if not self.image_dataset:
            exclude.update({'image_dataset', 'image_dataset_key', 'image_variant', 'image_crop_type', 'image_dataset_fallback'})
        return config_hash(self, exclude)

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)

def load_ocr_config(path: str | Path, *, base: OCRConfig | None=None) -> OCRConfig:
    """Load a JSON OCR config over a copy of the notebook/runtime configuration.

    Fields omitted from the JSON (engine, input_dir, workdir, worker_python,
    pages, etc.) keep the values from base. The original base is not modified.
    """
    path = Path(path)
    data = json.loads(path.read_text(encoding='utf-8-sig'))
    if not isinstance(data, dict):
        raise ValueError(f'{path}: OCR config must be a JSON object')
    known = {f.name for f in dataclasses.fields(OCRConfig)}
    unknown = set(data) - known
    if unknown:
        raise ValueError(f'{path}: unknown OCR config fields: {sorted(unknown)}')
    if 'books' in data and (not isinstance(data['books'], list) or any((not isinstance(book, str) or not book for book in data['books']))):
        raise ValueError(f'{path}: books must be a list of nonempty strings')
    values = (base or OCRConfig()).to_dict()
    values.update(data)
    return OCRConfig(**values)


import os
from pathlib import Path, PureWindowsPath
import re

def to_wsl_path(value: str | Path) -> str:
    value = str(value)
    if value.startswith('/'):
        return value
    path = PureWindowsPath(value)
    if not re.fullmatch('[A-Za-z]:', path.drive) or not path.is_absolute():
        raise ValueError(f'WSL requires an absolute drive path or Linux path: {value}')
    return '/mnt/' + path.drive[0].lower() + '/' + '/'.join(path.parts[1:])

def runtime_path(value: str | Path) -> Path:
    value = str(value)
    return Path(to_wsl_path(value)) if os.name != 'nt' and re.match('^[A-Za-z]:[\\\\/]', value) else Path(value)


import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from fnmatch import fnmatch
from datetime import datetime, timezone
from pathlib import Path
from tqdm.auto import tqdm

def _sanitize(name: str) -> str:
    return re.sub('[^\\w\\-]+', '_', name).strip('_') or 'run'

def _run_dir(workdir: str | Path, kind: str, name: str, cfg) -> Path:
    digest = cfg.cache_key()[:8]
    return Path(workdir) / 'runs' / f'{kind}_{_sanitize(name)}_{digest}'

def parse_pages(spec: str | int, n: int) -> list[int]:
    """'1-10,15' -> [0..9, 14] (đếm từ 1); '' -> mọi trang."""
    spec = str(spec)
    if not spec.strip():
        return list(range(n))
    out: list[int] = []
    for part in spec.split(','):
        a, _, b = part.strip().partition('-')
        out += range(int(a) - 1, min(int(b or a), n))
    return sorted(set((p for p in out if 0 <= p < n)))

def _pdf_pages(path: Path) -> int:
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(str(path))
    try:
        return len(doc)
    finally:
        doc.close()

def _entry_matches(entry: dict, pattern: str) -> bool:
    return fnmatch(entry['name'], pattern) or fnmatch(entry['stem'], pattern) or fnmatch(Path(entry['path']).name, pattern)

def select_entries(entries: list[dict], patterns: list[str]) -> list[dict]:
    """Chọn sách theo tên/glob. [] = tất cả. Pattern không khớp -> báo lỗi."""
    if not patterns:
        return entries
    chosen: list[dict] = []
    seen: set[str] = set()
    missing: list[str] = []
    for pat in patterns:
        hits = [e for e in entries if _entry_matches(e, pat)]
        if not hits:
            missing.append(pat)
        for e in hits:
            if e['path'] not in seen:
                chosen.append(e)
                seen.add(e['path'])
    if missing:
        names = ', '.join((e['name'] for e in entries))
        raise ValueError(f'books không khớp: {missing}. Có sẵn: {names}')
    return chosen

def list_pdfs(input_dir: str | Path) -> list[dict]:
    """Liệt kê PDF trong input_dir: name (đã làm sạch), stem, path, pages."""
    root = Path(input_dir)
    if not root.exists():
        raise FileNotFoundError(f'không thấy input_dir: {root}')
    files = [p for p in sorted(root.rglob('*.pdf')) if '__MACOSX' not in str(p)]
    if not files:
        raise FileNotFoundError(f'không thấy PDF nào trong {root}')
    entries: list[dict] = []
    used: set[str] = set()
    for p in files:
        name = _sanitize(p.stem)
        while name in used:
            name += '_'
        used.add(name)
        entries.append({'name': name, 'stem': p.stem, 'path': str(p)})
    for entry, path in zip(entries, files):
        entry['n_pages'] = _pdf_pages(path)
    return entries


from math import floor, isfinite, sqrt
from pathlib import Path

def image_budget(cfg):
    defaults = {'hunyuan': 4194304, 'paddle': 1048576, 'tonggu': 1048576, 'ppocr': 4194304, 'dummy': 4194304}
    pixels = cfg.region_max_pixels or defaults[cfg.engine]
    if cfg.model_max_pixels and cfg.engine == 'hunyuan':
        pixels = min(pixels, cfg.model_max_pixels)
    return (pixels, cfg.region_max_side)

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert('L')
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get('ink_threshold', 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    left, right = (max(1, int(w * 0.03)), min(w - 1, int(w * 0.97)))
    top, bottom = (max(1, int(h * 0.03)), min(h - 1, int(h * 0.97)))
    if left >= right or top >= bottom:
        return []
    rows = [sum((v > 0 for v in data[y * w + left:y * w + right])) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < 0.7 * (right - left)]
    if not valid_rows:
        return []
    cols = [sum((data[y * w + x] > 0 for y in valid_rows)) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < 0.7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == 'x':
        values = [sum((data[y * w + x] > 0 for y in valid_rows)) / len(valid_rows) for x in range(w)]
        extent, length = (image.width, w)
    else:
        values = [sum((data[y * w + x] > 0 for x in valid_cols)) / len(valid_cols) for y in range(h)]
        extent, length = (image.height, h)
    values = [0.0 if value >= 0.7 else value for value in values]
    threshold = options.get('gap_ink_fraction', 0.006)
    minimum = max(2, round(length * options.get('min_gap_fraction', 0.006)))
    edge = max(minimum, round(length * 0.06))
    gaps, start = ([], None)
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                span = max(minimum * 3, round(length * 0.035))
                before = values[max(edge, start - span):start]
                after = values[i:min(length - edge, i + span)]
                support = max(3, round(length * 0.006))
                if sum((v > threshold * 2 for v in before)) >= support and sum((v > threshold * 2 for v in after)) >= support:
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2 - x1 if axis == 'x' else y2 - y1]
    result = [(x1 + a, y1, x1 + b, y2) for a, b in zip(ends, ends[1:])] if axis == 'x' else [(x1, y1 + a, x2, y1 + b) for a, b in zip(ends, ends[1:])]
    return result[::-1] if axis == 'x' and direction == 'vertical_rtl' else result

def plan_regions(image, cfg, *, page=0):
    """Return a disjoint, complete partition of the rendered page in read order."""
    cfg.__post_init__()
    full = (0, 0, image.width, image.height)
    info = {'method': 'whole_page', 'warnings': [], 'coverage': 1.0, 'overlap': False}
    if cfg.segmentation == 'none':
        return ([full], info)
    options = cfg.segmentation_args
    allowed = {'column_cuts', 'page_column_cuts', 'columns_per_region', 'max_regions', 'ink_threshold', 'gap_ink_fraction', 'min_gap_fraction'}
    if set(options) - allowed:
        raise ValueError(f'Unknown segmentation options: {sorted(set(options) - allowed)}')
    cuts = options.get('page_column_cuts', {}).get(str(page + 1), options.get('column_cuts'))
    native = cfg.segmentation == 'auto' and cfg.engine in {'paddle', 'tonggu'} and (cuts is None)
    if native:
        return ([full], {**info, 'method': 'native_layout', 'warnings': []})
    max_regions = int(options.get('max_regions', 64))
    group = int(options.get('columns_per_region', 3))
    if max_regions < 1 or group < 1:
        raise ValueError('max_regions and columns_per_region must be >= 1')
    boxes = [full]
    if cuts is not None:
        if not isinstance(cuts, list) or any((not isinstance(v, (int, float)) or not isfinite(v) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts)):
            raise ValueError('column_cuts must be unique increasing fractions in (0, 1)')
        pixel_cuts = [round(x * image.width) for x in cuts]
        if len(set([0, *pixel_cuts, image.width])) != len(cuts) + 2:
            raise ValueError('column_cuts collapse at this image resolution')
        boxes = _partition_box(full, pixel_cuts, 'x', cfg.reading_direction)
        info['method'] = 'manual_columns'
    elif cfg.segmentation == 'columns' or (cfg.engine == 'hunyuan' and cfg.reading_direction == 'vertical_rtl'):
        found = projection_gaps(image, 'x', options)
        if cfg.reading_direction == 'vertical_rtl':
            selected = list(reversed(list(reversed(found))[group - 1::group]))
        else:
            selected = found[group - 1::group]
        if cfg.segmentation == 'auto' and len(found) < 2 * group - 1:
            selected = []
            info['warnings'].append('too_few_gutters_for_context_groups; kept whole page')
        boxes = _partition_box(full, selected, 'x', cfg.reading_direction)
        info.update(method='whitespace_columns', detected_gutters=found)
        info['warnings'].append('heuristic_layout; preview cuts on text/annotation pages before OCR')
        if not found:
            info['warnings'].append('no_confident_column_gutter; kept whole page')
    pixels, side = image_budget(cfg)

    def too_big(box):
        w, h = (box[2] - box[0], box[3] - box[1])
        return w * h > pixels or (side > 0 and max(w, h) > side)
    pending, boxes = (boxes, [])
    while pending:
        box = pending.pop(0)
        if too_big(box):
            crop = image.crop(box)
            axes = (['x', 'y'] if crop.height >= 4 * crop.width else ['x']) if cfg.reading_direction == 'vertical_rtl' else ['y', 'x']
            children = None
            for axis in axes:
                candidates = projection_gaps(crop, axis, options)
                size = crop.width if axis == 'x' else crop.height
                candidates = [c for c in candidates if 0.15 * size <= c <= 0.85 * size]
                if candidates:
                    cut = min(candidates, key=lambda x: abs(x - size / 2))
                    children = _partition_box(box, [cut], axis, cfg.reading_direction)
                    break
            if children and len(boxes) + len(pending) + 2 <= max_regions:
                pending[0:0] = children
                continue
            info['warnings'].append('unsplittable_region_exceeds_budget')
        boxes.append(box)
    if len(boxes) > max_regions:
        raise ValueError('Region plan exceeds max_regions; increase columns_per_region')
    info.update(regions=len(boxes), coverage=1.0, overlap=False)
    return (boxes, info)

def prepare_region(image, box, cfg, *, native=False):
    from PIL import Image
    crop = image.crop(box)
    original = crop.size
    pixels, side = image_budget(cfg)
    if native:
        return (crop, {'crop_size': list(original), 'input_size': list(original), 'resized': False, 'scale_xy': [1.0, 1.0], 'budget_scope': 'native_blocks'})
    scale = min(cfg.region_upscale, sqrt(pixels / (crop.width * crop.height)))
    if side:
        scale = min(scale, side / max(crop.size))
    if cfg.resolution_strict and scale < 1:
        raise ValueError(f'Cannot preserve region {box}: {crop.size} exceeds pixel/side budget')
    target = (max(1, floor(crop.width * scale)), max(1, floor(crop.height * scale)))
    if target != original:
        crop = crop.resize(target, Image.Resampling.LANCZOS)
    return (crop, {'crop_size': list(original), 'input_size': list(target), 'resized': target != original, 'downsampled': target[0] < original[0] or target[1] < original[1], 'scale_xy': [target[0] / original[0], target[1] / original[1]], 'max_pixels': pixels, 'max_side': side, 'budget_scope': 'backend_input'})

def _page_geometry(extra, box, input_size, offset, region_index):
    """Map genuine crop detections back to the page; preserve their provenance."""
    from copy import deepcopy
    extra = deepcopy(extra)
    if 'blocks' in extra:
        extra['blocks'] = ordered_page_blocks(extra['blocks'])
    sx, sy = ((box[2] - box[0]) / input_size[0], (box[3] - box[1]) / input_size[1])

    def transform(points):
        if not isinstance(points, (list, tuple)):
            return points
        if len(points) == 4 and all((isinstance(v, (int, float)) for v in points)):
            return [points[0] * sx + box[0], points[1] * sy + box[1], points[2] * sx + box[0], points[3] * sy + box[1]]
        return [[x * sx + box[0], y * sy + box[1]] for x, y in points]
    for entry in extra.get('character_boxes', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['offset'] += offset
        entry['region_index'] = region_index
    for entry in extra.get('text_lines', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['region_index'] = region_index
    for field in ('blocks', 'boxes'):
        for entry in extra.get(field, []):
            entry['box'] = transform(entry.get('box'))
            entry['region_index'] = region_index
    return extra

def run_prepared_page(backend, image, image_path, cfg, *, page=0):
    """Run all regions transactionally; partial region text never becomes a page."""
    import tempfile
    if cfg.segmentation == 'none' and (not image.info.get('prepared_dataset')):
        return backend.run(str(image_path))
    with tempfile.TemporaryDirectory(prefix='vtm-regions-') as directory:
        return _run_prepared_regions(backend, image, cfg, page=page, directory=Path(directory))

def _run_prepared_regions(backend, image, cfg, *, page, directory):
    prepared = image.info.get('prepared_dataset')
    if prepared:
        boxes = [tuple(r['source_box']) for r in prepared['regions']]
        layout = {**prepared['layout'], 'warnings': list(prepared['layout']['warnings']), 'method': 'prepared_' + prepared['crop_type']}
    else:
        boxes, layout = plan_regions(image, cfg, page=page)
    if image.info.get('render', {}).get('downsampled'):
        layout['warnings'].append('page_render_downsampled_before_segmentation; raise max_side')
        if cfg.resolution_strict:
            raise ValueError('Page render was downsampled before segmentation; raise max_side')
    records, texts, merged, offset = ([], [], {}, 0)
    native = prepared['crop_type'] == 'native' and cfg.engine in {'paddle', 'tonggu'} if prepared else layout['method'] == 'native_layout'
    for i, box in enumerate(boxes):
        if prepared:
            entry = prepared['regions'][i]['images'][prepared['variant']]
            variant = dataset_load_image(prepared['root'], entry)
            crop, meta = prepare_region(variant, (0, 0, variant.width, variant.height), cfg, native=native)
            meta.update(variant_scale_xy=meta['scale_xy'], variant_downsampled=meta.get('downsampled', False), resized_from_variant=meta['resized'])
            meta.update(crop_size=[box[2] - box[0], box[3] - box[1]], variant_size=list(variant.size), scale_xy=[crop.width / (box[2] - box[0]), crop.height / (box[3] - box[1])], downsampled=crop.width < box[2] - box[0] or crop.height < box[3] - box[1], resized=crop.size != (box[2] - box[0], box[3] - box[1]), variant=prepared['variant'], variant_sha256=entry['sha256'], operation=entry['operation'], canonical_region_id=prepared['regions'][i]['id'], input_to_source=[(box[2] - box[0]) / crop.width, 0, 0, (box[3] - box[1]) / crop.height, box[0], box[1]])
        else:
            crop, meta = prepare_region(image, box, cfg, native=native)
        path = directory / f'region_{i:03d}.png'
        crop.save(path, format='PNG')
        try:
            text, extra = backend.run(str(path))
        finally:
            path.unlink(missing_ok=True)
        observed = extra.get('model_input', {}).get('processed_sizes_wh', [])
        if len(observed) == 1:
            actual_w, actual_h = observed[0]
            meta['processor_size'] = [actual_w, actual_h]
            meta['processor_scale_xy'] = [actual_w / crop.width, actual_h / crop.height]
            meta['processor_downsampled'] = actual_w < crop.width or actual_h < crop.height
            if meta['processor_downsampled']:
                layout['warnings'].append(f'region_{i}: processor_downsampled (includes patch-grid rounding)')
                if cfg.resolution_strict:
                    raise ValueError(f'Processor downsampled region {i}: {crop.size} -> {observed[0]}')
        if cfg.engine == 'ppocr':
            ordered, order_info = order_ppocr_columns({'text': text, **extra}, mode=('auto' if cfg.segmentation == 'auto' else 'vertical_rtl') if cfg.reading_direction == 'vertical_rtl' else 'detection')
            text = ordered.pop('text')
            extra = {**ordered, 'region_reading_order': order_info}
        records.append({'index': i, 'page_box': list(box), **meta, 'text': text, 'raw': extra})
        mapped = _page_geometry(extra, box, crop.size, offset, i)
        for field in ('character_boxes', 'text_lines', 'blocks', 'boxes'):
            if field in mapped:
                merged.setdefault(field, []).extend(mapped[field])
        offset += len(comparison_text(text))
        texts.append(text)
    for i, block in enumerate(merged.get('blocks', [])):
        block['order'] = i
    issues = sorted({issue for rec in records for issue in rec['raw'].get('quality_issues', [])})
    hit_flags = [rec['raw'].get('hit_max') for rec in records]
    hit_max = True if any((v is True for v in hit_flags)) else None if any((v is None for v in hit_flags)) else False
    if any((rec.get('downsampled') for rec in records)):
        layout['warnings'].append('backend_input_downsampled; inspect regions or enable resolution_strict')
    if any((rec.get('variant_downsampled') for rec in records)):
        layout['warnings'].append('backend_variant_downsampled; inspect selected variant and input budget')
    observation = {'page_size': list(image.size), 'render': image.info.get('render', {}), 'layout': layout, 'regions': records, 'model_min_pixels': cfg.model_min_pixels, 'model_max_pixels': cfg.model_max_pixels, 'processor_observation': 'see region raw.model_input; native internal tensors may be unavailable'}
    if prepared:
        observation['dataset'] = {k: prepared[k] for k in ('snapshot_id', 'source_sha256', 'source', 'variant', 'crop_type', 'acquisition')}
    return ('\n'.join(texts), {**merged, 'hit_max': hit_max, 'quality_issues': issues, 'coordinate_frame': 'input_image', 'image_preparation': observation, 'reading_direction': cfg.reading_direction, 'region_order_applied': True})

def preview_preparation(src, page, cfg, *, output=None):
    """Model-free preview using 1-based PDF page numbers; never starts OCR."""
    from PIL import ImageDraw
    image = render(str(src), page - 1, cfg.dpi, cfg.max_side)
    boxes, layout = plan_regions(image, cfg, page=page - 1)
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    rows = []
    for i, box in enumerate(boxes):
        _, meta = prepare_region(image, box, cfg, native=layout['method'] == 'native_layout')
        draw.rectangle((box[0], box[1], box[2] - 1, box[3] - 1), outline='red', width=max(1, image.width // 600))
        draw.text((box[0] + 3, box[1] + 3), str(i + 1), fill='red')
        rows.append({'region': i + 1, 'page_box': list(box), **meta})
    if output:
        Path(output).parent.mkdir(parents=True, exist_ok=True)
        preview.save(output)
    return (preview, {'source': str(src), 'page': page, 'page_size': list(image.size), 'render': image.info.get('render', {}), 'engine': cfg.engine, 'layout': layout, 'regions': rows})


import math
PDF_IMAGE_VERSION = 'pdfium-visible-v1'

def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t - b) / w, (r - l) / h, 0, l, b]
    if rotation == 180:
        return [-(r - l) / w, 0, 0, (t - b) / h, r, b]
    if rotation == 270:
        return [0, -(t - b) / w, -(r - l) / h, 0, r, t]
    return [(r - l) / w, 0, 0, -(t - b) / h, l, t]

def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w * scale), math.ceil(h * scale)]
    if min(size) < 1 or size[0] * size[1] > max_pixels:
        raise ValueError(f'Page render {size} exceeds max_source_pixels={max_pixels}; review acquisition settings (source is never silently reduced)')
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert('RGB')
    finally:
        bitmap.close()

def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100000000, strategy='auto'):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops
    if strategy not in {'auto', 'render'} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError('Invalid PDF acquisition options')
    width, height = page.get_size()
    bbox, rotation = (list(page.get_bbox()), page.get_rotation())
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = ([], 0.0)
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x, y) for x, y in ((0, 0), (1, 0), (1, 1), (0, 1))]
        bounds = [min((c[0] for c in corners)), min((c[1] for c in corners)), max((c[0] for c in corners)), max((c[1] for c in corners))]
        full = max(0, min(bbox[2], bounds[2]) - max(bbox[0], bounds[0])) * max(0, min(bbox[3], bounds[3]) - max(bbox[1], bounds[1])) >= 0.8 * (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        native = max(pixels[0] / axes[0], pixels[1] / axes[1]) if min(axes) > 0 else 0
        density = max(density, native)
        images.append({'native_size': list(pixels), 'bounds_pdf': bounds, 'matrix_pdf': [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f], 'level': obj.level, 'full_page_scan': full, 'pixels_per_72_canvas_units': 72 * native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append('not_one_top_level_image; preserve_composition')
    if rotation:
        reasons.append('page_rotation')
    if annotations != 0:
        reasons.append('page_annotations_or_unknown')
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if obj.level or abs(m.b) > 1e-06 or abs(m.c) > 1e-06 or (m.a <= 0) or (m.d <= 0) or any((abs(a - v) > 0.001 for a, v in zip(obj.get_bounds(), bbox))) or any((abs(a - v) > 0.001 for a, v in zip([m.e, m.f, m.a, m.d], [l, b, r - l, t - b]))):
            reasons.append('placement_crop_rotation_or_reflection')
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append('image_clip_path')
    image, method, validation = (None, 'pdf_render', None)
    if strategy == 'auto' and (not reasons):
        nw, nh = obj.get_px_size()
        if nw * nh > max_source_pixels:
            raise ValueError('Native image exceeds max_source_pixels')
        scale = math.nextafter(min(nw / width, nh / height), 0)
        if abs(nw / width - nh / height) > 0.0001 * max(nw / width, nh / height):
            reasons.append('anisotropic_native_geometry')
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert('RGB')
            finally:
                if hasattr(bitmap, 'warn_on_close'):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {'method': 'full_pixel_equality_to_visible_render', 'equal': equal, 'render_size': list(visible.size), 'scale': scale}
            if equal:
                image, method = (candidate, 'native_validated')
            else:
                reasons.append('native_pixels_differ_from_visible_page')
    if image is None:
        raster_only = bool(images) and all((o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM} for o in objects)) and (annotations == 0)
        scale = density if raster_only and density else max(dpi / 72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation['scale']
    meta = {'version': PDF_IMAGE_VERSION, 'method': method, 'size': list(image.size), 'page_size_pdf': [width, height], 'page_bbox_pdf': bbox, 'media_box_pdf': list(page.get_mediabox()), 'crop_box_pdf': list(page.get_cropbox()), 'rotation': rotation, 'object_types': [o.type for o in objects], 'annotations': annotations, 'embedded_images': images, 'requested_dpi': dpi, 'render_scale': scale, 'native_validation': validation, 'fallback_reasons': reasons, 'source_downsampled': False, 'source_to_pdf': pdf_pixel_to_page(image.size, bbox, rotation), 'resolution_unit': 'pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable', 'warnings': ['low_native_resolution; interpolation_cannot_recover_detail'] if images and max((max(i['native_size']) for i in images)) <= 750 else []}
    return (image, meta)


from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import time
IMAGE_DATASET_FORMAT = 'vtm-image-dataset'
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = ''

@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = 'auto'
    max_source_pixels: int = 100000000
    analysis_max_side: int = 1200
    reading_direction: str = 'vertical_rtl'
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = 0.006
    min_gap_fraction: float = 0.006
    variants: list[str] = field(default_factory=lambda: ['original'])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.dpi <= 0 or self.max_source_pixels < 1 or (not 32 <= self.analysis_max_side <= 1600) or (self.columns_per_group < 1) or (self.max_columns < 1):
            raise ValueError('Invalid preparation dimensions/group limits')
        if self.acquisition not in {'auto', 'render'}:
            raise ValueError('acquisition must be auto or render')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('Invalid reading_direction')
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or (not 0 < self.min_gap_fraction < 0.5):
            raise ValueError('Invalid projection thresholds')
        if not isinstance(self.variants, list) or 'original' not in self.variants or len(set(self.variants)) != len(self.variants) or set(self.variants) - {'original', 'bicubic_x2', 'clahe'}:
            raise ValueError('variants require original; supported experiments: bicubic_x2, clahe. Denoising/SR need separate evidence and are not implemented')
        if set(self.variant_options) - {'clahe'}:
            raise ValueError('Only clahe accepts variant_options')
        for name, options in self.variant_options.items():
            if set(options) - {'clip_limit', 'tile_grid'}:
                raise ValueError(f'Unknown {name} parameters')
            if not 0 < options.get('clip_limit', 1.5) <= 4 or not 2 <= options.get('tile_grid', 8) <= 32:
                raise ValueError('CLAHE clip_limit must be (0,4], tile_grid [2,32]')
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError('page_column_cuts uses 1-based page keys')
            dataset_validate_cuts(cuts)

def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')

def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()

def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root / name) for name in ('image_dataset.py', 'image_preparation.py', 'pdf_images.py')})

def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {'pillow': PIL.__version__, 'pypdfium2': str(pdfium.PYPDFIUM_INFO), 'pdfium': str(pdfium.PDFIUM_INFO), 'python': sys.version.split()[0]}
    return result

def dataset_path(root, relative):
    relative = str(relative)
    if '\\' in relative or ':' in relative:
        raise ValueError(f'Dataset requires portable relative paths: {relative}')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or '..' in parts.parts or (not parts.parts):
        raise ValueError(f'Invalid dataset path: {relative}')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Dataset path escapes root: {relative}')
    return path

def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_bytes(data)
    temporary.replace(path)
    return {'path': str(relative), 'sha256': hashlib.sha256(data).hexdigest()}

def dataset_read(root, reference):
    data = dataset_path(root, reference['path']).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference['sha256']:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data

def dataset_store_image(root, image):
    image = image.convert('RGB')
    output = io.BytesIO()
    image.save(output, format='PNG', compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': hashlib.sha256(image.tobytes()).hexdigest()}

def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert('RGB')
    if list(image.size) != reference['size'] or hashlib.sha256(image.tobytes()).hexdigest() != reference['pixel_sha256']:
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image

@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root) / 'prepare.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            if os.name == 'nt':
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream, fcntl.LOCK_UN)

def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts))):
        raise ValueError('column_cuts must be unique increasing fractions in (0,1)')

def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == 'horizontal_ltr' else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region['source_box']
        if not all((isinstance(v, int) for v in (x1, y1, x2, y2))) or not 0 <= x1 < x2 <= w or (y1, y2) != (0, h) or (region['reading_order'] != index):
            raise ValueError('Invalid canonical crop bounds/order')
        start, end = (x1, x2) if direction == 'horizontal_ltr' else (x2, x1)
        if start != expected:
            raise ValueError('Canonical partition has a gap, overlap or inconsistent reading order')
        expected = end
    if not regions or expected != (w if direction == 'horizontal_ltr' else 0):
        raise ValueError('Canonical partition omits page content')

def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ('ink_threshold', 'gap_ink_fraction', 'min_gap_fraction')}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page + 1), config.column_cuts)
    if manual is not None:
        cuts = [round(v * w) for v in manual]
        method = 'manual_columns'
    elif config.reading_direction == 'vertical_rtl':
        cuts = sorted(set((round(v * w / aw) for v in projection_gaps(analysis, 'x', options))))
        method = 'whitespace_projection'
    else:
        cuts, method = ([], 'whole_page_horizontal')
    if len(set([0, *cuts, w])) != len(cuts) + 2:
        raise ValueError('Column cuts collapse in source space')
    warnings = ['heuristic_layout; geometric_coverage_is_not_text_coverage']
    if len(cuts) + 1 > config.max_columns:
        if manual is not None:
            raise ValueError('Manual layout exceeds max_columns')
        warnings.append('too_many_gutters; retained_whole_page')
        cuts = []
    if not cuts:
        warnings.append('no_partition; whole_page_retained')
    thresholds = sorted(set([max(1, config.ink_threshold - 8), config.ink_threshold, min(254, config.ink_threshold + 8)]))
    found = [projection_gaps(analysis, 'x', {**options, 'ink_threshold': t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = (int(0.03 * ah), max(1, int(0.97 * ah)))
    projection = [sum((ink[y * aw + x] > 0 for y in range(top, bottom))) / (bottom - top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw - 1, max(0, round(cut * aw / w)))
        lo, hi = (x, x)
        while lo > 0 and projection[lo - 1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw - 1 and projection[hi + 1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2, round(aw * 0.006))
        stability = sum((any((abs(c - x) <= tolerance for c in cs)) for cs in found)) / len(found)
        radius = max(2, round(aw * 0.035))
        left, right = (projection[max(0, x - radius):x], projection[x + 1:min(aw, x + radius + 1)])
        risks = []
        if projection[x] > 0.02:
            risks.append('cut_crosses_analysis_ink; rules_or_strokes_need_review')
        if stability < 2 / 3:
            risks.append('unstable_across_ink_thresholds')
        if not left or not right or min(max(left), max(right)) <= config.gap_ink_fraction * 2:
            risks.append('weak_text_support_on_one_side')
        evidence.append({'source_x': cut, 'analysis_x': x, 'gutter_width_analysis': hi - lo + 1, 'ink_fraction': projection[x], 'threshold_stability': stability, 'warnings': risks})
    full = (0, 0, w, h)
    columns = _partition_box(full, cuts, 'x', config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0, len(columns), group):
        chunk = columns[start:start + group]
        groups.append((min((b[0] for b in chunk)), 0, max((b[2] for b in chunk)), h))
    regions = []
    for kind, boxes in (('page', [full]), ('group', groups), ('column', columns)):
        for index, box in enumerate(boxes):
            regions.append({'id': f'{kind}_{index:03d}', 'crop_type': kind, 'reading_order': index, 'source_box': list(box), 'overlap': False})
        dataset_partition_qc([r for r in regions if r['crop_type'] == kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r['crop_type'] == 'column']:
        box = region['source_box']
        x1, x2 = (round(box[0] * aw / w), round(box[2] * aw / w))
        draw.rectangle((x1, 0, max(x1, x2 - 1), ah - 1), outline='red', width=1)
        draw.text((x1 + 2, 3), str(region['reading_order'] + 1), fill='red')
    for item in evidence:
        if item['warnings']:
            draw.line((item['analysis_x'], 0, item['analysis_x'], ah - 1), fill='orange', width=2)
    draw.text((3, ah - 14), 'RTL <-' if config.reading_direction == 'vertical_rtl' else 'LTR ->', fill='red')
    layout = {'method': method, 'source_size': [w, h], 'analysis_size': [aw, ah], 'analysis_to_source': [w / aw, 0, 0, h / ah, 0, 0], 'reading_direction': config.reading_direction, 'regions': regions, 'cuts': evidence, 'excluded_regions': [], 'uncertain_regions': [r['id'] for r in regions if r['crop_type'] == 'column'], 'geometric_coverage': 1.0, 'text_coverage': 'unverified', 'warnings': warnings, 'thresholds_tested': thresholds, 'review_status': 'needs_visual_review'}
    return (layout, mask.convert('RGB'), preview)

def dataset_variant(image, name, options):
    from PIL import Image
    if name == 'original':
        return (image, {'method': 'identity', 'parameters': {}, 'warnings': []})
    if name == 'bicubic_x2':
        return (image.resize((image.width * 2, image.height * 2), Image.Resampling.BICUBIC), {'method': 'bicubic', 'parameters': {'scale': 2}, 'warnings': ['experimental_interpolation; no_recovered_source_detail']})
    if name == 'clahe':
        try:
            import cv2
            import numpy as np
        except ImportError as exc:
            raise RuntimeError('clahe requires an already available OpenCV/NumPy installation; use original offline') from exc
        params = {'clip_limit': 1.5, 'tile_grid': 8, **options}
        lab = cv2.cvtColor(np.asarray(image.convert('RGB')), cv2.COLOR_RGB2LAB)
        grid = int(params['tile_grid'])
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=float(params['clip_limit']), tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        return (Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)), {'method': 'clahe_lab_l', 'parameters': params, 'opencv': cv2.__version__, 'warnings': ['experimental_contrast; inspect_weak_strokes']})
    raise ValueError(f'Unsupported image variant: {name}')

def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root, record['source']) if check_pixels else None
    acquisition = json.loads(dataset_read(root, record['acquisition']))
    layout = json.loads(dataset_read(root, record['layout']))
    if record['source']['size'] != layout['source_size'] or acquisition['size'] != layout['source_size']:
        raise ValueError('Source, acquisition and layout size mismatch')
    dataset_read(root, layout['analysis_mask'])
    dataset_read(root, layout['preview'])
    for kind in ('page', 'group', 'column'):
        regions = [r for r in layout['regions'] if r['crop_type'] == kind]
        dataset_partition_qc(regions, layout['source_size'], layout['reading_direction'])
    if [r['id'] for r in record['regions']] != [r['id'] for r in layout['regions']]:
        raise ValueError('Image dataset omits canonical regions')
    for region, geometry in zip(record['regions'], layout['regions']):
        if any((region[k] != geometry[k] for k in ('source_box', 'crop_type', 'reading_order'))):
            raise ValueError('Region geometry differs from canonical layout')
        if 'original' not in region['images']:
            raise ValueError('Region original variant is missing')
        x1, y1, x2, y2 = region['source_box']
        for name, entry in region['images'].items():
            expected = [(x2 - x1) * (2 if name == 'bicubic_x2' else 1), (y2 - y1) * (2 if name == 'bicubic_x2' else 1)]
            if entry['size'] != expected or entry['image_to_source'] != [(x2 - x1) / expected[0], 0, 0, (y2 - y1) / expected[1], x1, y1]:
                raise ValueError('Variant geometry/transform mismatch')
            if check_pixels:
                image = dataset_load_image(root, entry)
                if name == 'original' and image.tobytes() != source.crop(region['source_box']).tobytes():
                    raise ValueError('Original crop is not pixel-identical to source')
    return layout

def dataset_open(directory):
    path = runtime_path(directory)
    descriptor = path / 'dataset.json' if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name == 'snapshots' else descriptor.parent
    data = json.loads(descriptor.read_text(encoding='utf-8'))
    if data.get('format') != IMAGE_DATASET_FORMAT or data.get('version') != IMAGE_DATASET_VERSION:
        raise ValueError('Unsupported image dataset format/version')
    body = {k: v for k, v in data.items() if k != 'snapshot_id'}
    if dataset_hash(body) != data['snapshot_id']:
        raise ValueError('Dataset snapshot identity mismatch')
    return (root, data)

def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor['pages'] if p['source_sha256'] == source_sha and p['page'] == page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root, e['record'])) for e in entries]
    if len({r['input_key'] for r in records}) != 1:
        raise ValueError('Ambiguous prepared layouts for the same source/page')
    record = records[0]
    if record['source_sha256'] != source_sha or record['page'] != page:
        raise ValueError('Dataset page identity mismatch')
    return record

def bind_image_dataset(cfg, source_sha, pages):
    """Bind an immutable snapshot before looking up OCR checkpoints."""
    if not cfg.image_dataset:
        return
    root, descriptor = dataset_open(cfg.image_dataset)
    snapshot_path = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot_path.is_file():
        raise ValueError('Dataset immutable snapshot file is missing')
    cfg.image_dataset = str(runtime_path(snapshot_path).resolve())
    cfg.image_dataset_key = descriptor['snapshot_id']
    for page in pages:
        record = dataset_page(root, descriptor, source_sha, page)
        if record is None:
            if cfg.image_dataset_fallback:
                continue
            raise ValueError(f'Prepared dataset missing source {source_sha}, page {page + 1}')
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ; choose a matching preparation profile')
        kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
        for region in record['regions']:
            if region['crop_type'] == kind and cfg.image_variant not in region['images']:
                raise ValueError(f"Missing variant {cfg.image_variant} for {region['id']}")

def dataset_ocr_page(cfg, source_sha, page):
    """Load prepared source with inference selection attached; never render a PDF."""
    root, descriptor = dataset_open(cfg.image_dataset)
    if cfg.image_dataset_key and descriptor['snapshot_id'] != cfg.image_dataset_key:
        raise ValueError('Prepared input snapshot changed since OCR plan')
    record = dataset_page(root, descriptor, source_sha, page)
    if record is None:
        if cfg.image_dataset_fallback:
            return None
        raise ValueError(f'Prepared dataset missing page {page + 1}')
    layout = dataset_validate_page(root, record, check_pixels=False)
    image = dataset_load_image(root, record['source'])
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    regions = [r for r in record['regions'] if r['crop_type'] == kind]
    if not regions:
        raise ValueError(f'Prepared crop type missing: {kind}')
    for region in regions:
        dataset_load_image(root, region['images'][cfg.image_variant])
    image.info['prepared_dataset'] = {'root': str(root), 'snapshot_id': descriptor['snapshot_id'], 'source_sha256': source_sha, 'source': record['source'], 'layout': layout, 'regions': regions, 'variant': cfg.image_variant, 'crop_type': cfg.image_crop_type, 'acquisition': json.loads(dataset_read(root, record['acquisition']))}
    return (image, dataset_path(root, record['source']['path']))

def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages='1-2', limit=2, max_books=1, profile_dir=None):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or (max_books is not None and max_books < 1):
        raise ValueError('limit/max_books must be positive or None')
    entries = select_entries(list_pdfs(input_dir), books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = ([], limit)
    for entry in entries:
        wanted = parse_pages(pages, entry['n_pages'])
        if remaining is not None:
            wanted = wanted[:remaining]
            remaining -= len(wanted)
        if wanted:
            selected.append((entry, wanted))
    if not selected:
        raise ValueError('No pages selected')
    root = Path(output_dir).resolve()
    stats = {'pages': 0, 'source_reused': 0, 'layout_reused': 0, 'variants_reused': 0}
    started, environment, code = (time.monotonic(), dataset_environment(), dataset_code_signature())
    page_entries, manifest_rows, configurations = ([], [], {})
    with dataset_lock(root):
        marker = root / 'dataset.json'
        if marker.exists():
            dataset_open(root)
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir) / (entry['stem'] + '.json')).read_text(encoding='utf-8-sig'))
                for key in ('dpi', 'reading_direction'):
                    if key in profile:
                        setattr(cfg, key, profile[key])
            cfg.__post_init__()
            configurations[entry['name']] = asdict(cfg)
            source_sha = dataset_file_hash(entry['path'])
            doc = pdfium.PdfDocument(entry['path'])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({'source_sha256': source_sha, 'page': page, 'acquisition': cfg.acquisition, 'dpi': cfg.dpi, 'max_source_pixels': cfg.max_source_pixels, 'code': code, 'environment': environment})
                    source_receipt = f'sources/{source_key}.json'
                    try:
                        cached = json.loads(dataset_path(root, source_receipt).read_bytes())
                        if cached['key'] != source_key:
                            raise ValueError('Source cache key mismatch')
                        image = dataset_load_image(root, cached['source'])
                        acquisition = json.loads(dataset_read(root, cached['acquisition']))
                        stats['source_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page, dpi=cfg.dpi, max_source_pixels=cfg.max_source_pixels, strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root, image)
                        acquisition_ref = dataset_write(root, f'sources/{source_key}.acquisition.json', dataset_json_bytes(acquisition))
                        cached = {'key': source_key, 'source': source_asset, 'acquisition': acquisition_ref}
                        dataset_write(root, source_receipt, dataset_json_bytes(cached))
                    layout_options = {k: v for k, v in asdict(cfg).items() if k not in {'dpi', 'acquisition', 'max_source_pixels', 'variants', 'variant_options'}}
                    layout_options['page_column_cuts'] = {str(page + 1): cfg.page_column_cuts.get(str(page + 1), cfg.column_cuts)}
                    layout_key = dataset_hash({'source': cached['source']['sha256'], 'options': layout_options, 'code': code})
                    layout_relative = f'layouts/{layout_key}.json'
                    try:
                        layout_data = dataset_path(root, layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout['key'] != layout_key:
                            raise ValueError('Layout key mismatch')
                        dataset_read(root, layout['analysis_mask'])
                        dataset_read(root, layout['preview'])
                        stats['layout_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        layout, mask, preview = dataset_plan_layout(image, cfg, page=page)
                        layout.update(key=layout_key, analysis_mask=dataset_store_image(root, mask), preview=dataset_store_image(root, preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root, layout_relative, layout_data)
                    layout_ref = {'path': layout_relative, 'sha256': hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout['regions']:
                        crop = image.crop(geometry['source_box'])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name, {})
                            variant_environment = dict(environment)
                            if name == 'clahe':
                                import cv2
                                variant_environment['opencv'] = cv2.__version__
                            key = dataset_hash({'source': cached['source']['sha256'], 'box': geometry['source_box'], 'variant': name, 'options': options, 'code': code, 'environment': variant_environment})
                            receipt = f'variants/{key}.json'
                            try:
                                variant = json.loads(dataset_path(root, receipt).read_bytes())
                                if variant['key'] != key:
                                    raise ValueError('Variant cache key mismatch')
                                dataset_load_image(root, variant)
                                stats['variants_reused'] += 1
                            except (OSError, ValueError, KeyError):
                                prepared, operation = dataset_variant(crop, name, options)
                                x1, y1, x2, y2 = geometry['source_box']
                                variant = {**dataset_store_image(root, prepared), 'key': key, 'operation': operation, 'image_to_source': [(x2 - x1) / prepared.width, 0, 0, (y2 - y1) / prepared.height, x1, y1]}
                                dataset_write(root, receipt, dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page + 1, 'page_index': page, 'acquisition_method': acquisition['method'], 'source_size': cached['source']['size'], 'source_image': cached['source'], 'analysis_size': layout['analysis_size'], 'analysis_to_source': layout['analysis_to_source'], 'source_to_pdf': acquisition['source_to_pdf'], 'region_id': geometry['id'], 'crop_type': geometry['crop_type'], 'source_box': geometry['source_box'], 'reading_order': geometry['reading_order'], 'reading_direction': cfg.reading_direction, 'variant': name, **variant, 'qc_warnings': acquisition['warnings'] + layout['warnings'] + [w for c in layout['cuts'] for w in c['warnings']], 'layout': layout_ref, 'acquisition': cached['acquisition'], 'code_sha256': code, 'processing_version': IMAGE_DATASET_VERSION})
                        regions.append({**geometry, 'images': variants})
                    record = {'input_key': dataset_hash({'source': source_key, 'layout': layout_key, 'variants': cfg.variants, 'options': cfg.variant_options}), 'source_sha256': source_sha, 'page': page, 'source': cached['source'], 'acquisition': cached['acquisition'], 'layout': layout_ref, 'regions': regions}
                    dataset_validate_page(root, record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root, f'pages/{record_key}.json', record_bytes)
                    page_entries.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page, 'record': record_ref})
                    stats['pages'] += 1
            finally:
                doc.close()
        manifest_data = b''.join((dataset_json_bytes(row) for row in manifest_rows))
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root, f'manifests/{manifest_sha}.jsonl', manifest_data)
        descriptor = {'format': IMAGE_DATASET_FORMAT, 'version': IMAGE_DATASET_VERSION, 'pages': page_entries, 'expected_pages': [[e['book'], e['source_sha256'], e['page']] for e in page_entries], 'manifest': manifest_ref, 'configurations': configurations, 'environment': environment, 'code_sha256': code, 'geometry': 'top_left_source_pixels; half_open_boxes', 'ocr_text_coverage': 'unverified'}
        descriptor['snapshot_id'] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root, snapshot, data)
        dataset_write(root, 'dataset.json', data)
    stats.update(snapshot_id=descriptor['snapshot_id'], dataset=str(root / snapshot), seconds=round(time.monotonic() - started, 3), manifest_rows=len(manifest_rows))
    return stats

def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    dataset_read(root, descriptor['manifest'])
    actual = [[e['book'], e['source_sha256'], e['page']] for e in descriptor['pages']]
    if actual != descriptor['expected_pages'] or len(set(map(tuple, actual))) != len(actual):
        raise ValueError('Dataset expected-page coverage mismatch')
    warnings = []
    for entry in descriptor['pages']:
        record = dataset_page(root, descriptor, entry['source_sha256'], entry['page'])
        layout = dataset_validate_page(root, record)
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}: {w}" for w in layout['warnings']))
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}/x{c['source_x']}: {w}" for c in layout['cuts'] for w in c['warnings']))
    return {'pages': len(actual), 'snapshot_id': descriptor['snapshot_id'], 'warnings': warnings, 'geometric_coverage': 1.0, 'ocr_text_coverage': 'unverified'}

def package_image_dataset(directory, archive):
    """ZIP_STORED PNGs; include only this validated snapshot's referenced files."""
    import zipfile
    root, descriptor = dataset_open(directory)
    validate_image_dataset(directory)
    archive = Path(archive).resolve()
    if archive.exists():
        raise FileExistsError(f'Archive already exists: {archive}')
    references = {descriptor['manifest']['path']}
    for entry in descriptor['pages']:
        references.add(entry['record']['path'])
        record = json.loads(dataset_read(root, entry['record']))
        references.update((record[k]['path'] for k in ('source', 'acquisition', 'layout')))
        layout = json.loads(dataset_read(root, record['layout']))
        references.update((layout[k]['path'] for k in ('analysis_mask', 'preview')))
        references.update((v['path'] for r in record['regions'] for v in r['images'].values()))
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
        for name in ['dataset.json', f"snapshots/{descriptor['snapshot_id']}.json", *sorted(references)]:
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.external_attr = 33188 << 16
            data = dataset_json_bytes(descriptor) if name == 'dataset.json' or name.startswith('snapshots/') else dataset_path(root, name).read_bytes()
            output.writestr(info, data)
    return {'archive': str(archive), 'files': len(references) + 2, 'bytes': archive.stat().st_size, 'snapshot_id': descriptor['snapshot_id']}

def add_image_dataset_commands(parser):
    actions = parser.add_subparsers(dest='image_action', required=True)
    prepare = actions.add_parser('prepare', help='prepare once; repeat to resume (default: one PDF/two pages)')
    prepare.add_argument('--input-dir', default='data')
    prepare.add_argument('--output-dir', required=True)
    prepare.add_argument('--config', help='JSON ImageDatasetConfig; no OCR/model settings')
    prepare.add_argument('--books', nargs='+', default=[])
    prepare.add_argument('--pages', default='1-2', help='1-based selection; empty string selects all pages')
    prepare.add_argument('--limit', type=int, default=2, help='total page limit; 0 means unlimited')
    prepare.add_argument('--max-books', type=int, default=1, help='0 means all selected PDFs')
    prepare.add_argument('--profile-dir', help='reuse only dpi/reading_direction from OCR book profiles')
    validate = actions.add_parser('validate', help='verify hashes, pixels, geometry, pages and references')
    validate.add_argument('dataset')
    package = actions.add_parser('package', help='package one validated snapshot for offline Kaggle input')
    package.add_argument('dataset')
    package.add_argument('archive')

def run_image_dataset_command(args):
    if args.image_action == 'prepare':
        config = ImageDatasetConfig(**json.loads(Path(args.config).read_text(encoding='utf-8-sig'))) if args.config else ImageDatasetConfig()
        result = prepare_image_dataset(args.input_dir, args.output_dir, config=config, books=args.books, pages=args.pages, limit=None if args.limit == 0 else args.limit, max_books=None if args.max_books == 0 else args.max_books, profile_dir=args.profile_dir)
    elif args.image_action == 'validate':
        result = validate_image_dataset(args.dataset)
    else:
        result = package_image_dataset(args.dataset, args.archive)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


IMAGE_DATASET_CODE_SIGNATURE = '77a207cd108f3a591b0c28a7510a3bd50968204d71e5bfb730962067af10ca10'

import argparse
import json
import os
import re
import signal
import tempfile
import threading
import time
from contextlib import contextmanager
from pathlib import Path
PADDLE_IGNORE = {'header', 'header_image', 'footer', 'footer_image', 'number', 'footnote', 'aside_text', 'image', 'chart', 'seal'}

def clean_degenerate(s: str, max_unit: int=40, min_chars: int=60, min_cycles: int=4) -> str:
    """Cắt đuôi lặp vô hạn (VLM kẹt vào 1 chu kỳ ngắn rồi lặp tới hết token)."""
    st = s.rstrip()
    for u in range(1, max_unit + 1):
        unit = st[-u:]
        if len(st) < min_chars or not unit.strip():
            continue
        n, i = (0, len(st))
        while i >= u and st[i - u:i] == unit:
            n, i = (n + 1, i - u)
        if n * u >= min_chars and n >= min_cycles:
            return st[:i].rstrip()
    return s

def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub('</t[dh]>', ' ', md)
    md = re.sub('</tr>|<br\\s*/?>', '\n', md)
    md = re.sub('<[^>]+>', '', md)
    md = re.sub('!\\[[^\\]]*\\]\\([^)]*\\)', '', md)
    out = []
    for ln in md.splitlines():
        ln = re.sub('^\\s{0,3}#{1,6}\\s*', '', ln)
        ln = re.sub('^\\s*[-*+]\\s+(?=\\S)', '', ln)
        ln = re.sub('(?:[ \\t,，]*\\(\\d{1,4},\\d{1,4}\\),\\(\\d{1,4},\\d{1,4}\\))+\\s*$', '', ln)
        ln = ln.replace('**', '').replace('__', '').strip()
        if ln.startswith('|') and ln.endswith('|'):
            ln = ' '.join((c.strip() for c in ln.strip('|').split('|') if c.strip()))
        if ln and (not re.fullmatch('[-|:\\s]+', ln)):
            out.append(ln)
    return out

def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if 'blocks' not in rec:
        return md_to_lines(rec['text'])
    return [ln for b in ordered_page_blocks(rec['blocks']) for ln in md_to_lines(b['text'])]

def ordered_page_blocks(blocks):
    """Use the same block sequence for exported text and segmented geometry."""
    ordered = sorted((b for b in blocks if b['order'] is not None), key=lambda b: b['order'])
    center = lambda b: ((b['box'][0] + b['box'][2]) / 2, (b['box'][1] + b['box'][3]) / 2)
    for b in (b for b in blocks if b['order'] is None):
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)
    return ordered
_DOCS = {}

def render(src: str, page: int, dpi: int=200, max_side: int=1600):
    """Trang PDF -> ảnh PIL (cạnh dài tối đa max_side)."""
    import pypdfium2 as pdfium
    if src not in _DOCS:
        _DOCS[src] = pdfium.PdfDocument(src)
    pdf_page = _DOCS[src][page]
    bitmap = None
    try:
        bitmap = pdf_page.render(scale=dpi / 72)
        img = bitmap.to_pil().convert('RGB')
    finally:
        if bitmap is not None:
            bitmap.close()
        pdf_page.close()
    original_size = img.size
    if max(img.size) > max_side:
        s = max_side / max(img.size)
        from PIL import Image
        img = img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.Resampling.LANCZOS)
    img.info['render'] = {'dpi': dpi, 'uncapped_size': list(original_size), 'size': list(img.size), 'downsampled': img.size != original_size, 'max_side': max_side}
    return img

def result_file(out: Path, book: str, name: str, shard: int) -> Path:
    return out / book / f'{name}_shard{shard}.jsonl'

def load_results(out: Path, book: str, name: str) -> dict[int, dict]:
    """{trang: bản ghi} của 1 sách, gộp mọi shard (dòng hỏng thì bỏ)."""
    recs = {}
    for f in sorted((out / book).glob(f'{name}_shard*.jsonl')):
        with open(f, encoding='utf-8') as stream:
            for line in stream:
                try:
                    r = json.loads(line)
                    recs[r['page']] = r
                except (json.JSONDecodeError, KeyError):
                    pass
    return recs


from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import uuid
from tqdm.auto import tqdm

def now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)

def ocr_source_sha256(path):
    path = Path(path)
    if not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def corpus_model_key(config):
    """Selection changes do not invalidate already completed OCR pages."""
    config = OCRConfig(**config.to_dict()) if isinstance(config, OCRConfig) else OCRConfig(**config)
    config.books, config.pages = ([], '')
    return config.cache_key()

@contextmanager
def gpu_lock(directory: Path):
    """Prevent separate notebook kernels from using this campaign's GPU at once."""
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / 'notebook_gpu.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        try:
            if os.name == 'nt':
                import msvcrt
                msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError:
            raise RuntimeError('Một kernel khác đang chạy OCR cho tập dữ liệu này. Chờ lượt đó kết thúc trước khi chạy lại.') from None
        try:
            yield
        finally:
            stream.seek(0)
            if os.name == 'nt':
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream, fcntl.LOCK_UN)

def prepare_multi_corpus(models: dict[str, OCRConfig], profile_dir: str | Path | None=None, *, max_books: int | None=None, cache_renders=True) -> Path:
    """Prepare identical selections for all configured engines; preserve model identities."""
    if len(models) < 2 or any((key != cfg.engine for key, cfg in models.items())):
        raise ValueError('Cần >= 2 model với key trùng engine, mỗi engine một cấu hình.')
    base_configs = list(models.values())
    first = base_configs[0]
    if len({str(Path(cfg.input_dir).resolve()) for cfg in base_configs}) != 1:
        raise ValueError('Các model phải dùng cùng thư mục PDF.')
    if len({str(Path(cfg.workdir).resolve()) for cfg in base_configs}) != 1:
        raise ValueError('Các model phải dùng cùng workdir.')
    if len({cfg.limit for cfg in base_configs}) != 1:
        raise ValueError('Các model phải dùng cùng limit.')
    if first.limit is not None and first.limit < 1:
        raise ValueError('limit phải >= 1 hoặc None.')
    if max_books is not None and max_books < 1:
        raise ValueError('max_books phải >= 1 hoặc None.')
    for base in base_configs:
        if base.fresh or base.resume_from:
            raise ValueError('Đổi workdir để chạy thử nghiệm mới; không dùng fresh/resume_from khi so nhiều model.')
        if base.n_gpu not in (0, 1):
            raise ValueError('Pipeline này giữ một model trên một GPU; n_gpu phải là 0 hoặc 1.')
    entries = list_pdfs(first.input_dir)
    selected = {}
    for base in base_configs:
        chosen = select_entries(entries, base.books)
        if max_books is not None:
            chosen = chosen[:max_books]
        selected[base.engine] = {e['name']: (e, parse_pages(base.pages, e['n_pages'])) for e in chosen}
        remaining = base.limit
        trimmed = {}
        for book, (entry, pages) in selected[base.engine].items():
            if remaining is not None:
                pages = pages[:remaining]
                remaining -= len(pages)
            if pages:
                trimmed[book] = (entry, pages)
        selected[base.engine] = trimmed
    pages_h = {b: p for b, (_, p) in selected[first.engine].items()}
    if not pages_h or any(({b: p for b, (_, p) in chosen.items()} != pages_h for chosen in selected.values())):
        raise ValueError('Các model phải OCR đúng cùng sách và cùng tập trang không rỗng.')
    workdir = Path(first.workdir).resolve()
    pairs, startup = ([], {})
    with gpu_lock(workdir):
        prior_path = workdir / 'manifest.json'
        prior = json.loads(prior_path.read_text(encoding='utf-8')) if prior_path.is_file() else {}
        prior_pairs = {pair['book']: pair for pair in prior.get('pairs', [])}
        for book, (entry, pages) in selected[first.engine].items():
            pair = {'book': book}
            source_sha = ocr_source_sha256(entry['path'])
            for base in base_configs:
                cfg = load_ocr_config(Path(profile_dir) / f"{entry['stem']}.json", base=base) if profile_dir is not None else OCRConfig(**base.to_dict())
                cfg.engine, cfg.books, cfg.pages = (base.engine, [Path(entry['path']).name], base.pages)
                for identity in ('model', 'model_revision', 'run_name', 'paddle_version', 'paddle_args', 'tonggu_repo', 'tonggu_config', 'worker_python', 'worker_wsl_distro', 'worker_pythonpath'):
                    setattr(cfg, identity, getattr(base, identity))
                cfg.input_dir, cfg.workdir = (str(Path(base.input_dir).resolve()), str(workdir))
                cfg.limit, cfg.n_layout, cfg.fresh, cfg.resume_from = (base.limit, 0, False, [])
                bind_image_dataset(cfg, source_sha, pages)
                signature = (cfg.model, cfg.model_revision, cfg.dtype, cfg.device, cfg.paddle_version, json.dumps(cfg.paddle_args, sort_keys=True), cfg.tonggu_repo, cfg.tonggu_config)
                if cfg.engine in startup and signature != startup[cfg.engine]:
                    raise ValueError('Các sách phải dùng cùng model/dtype/device để chỉ tải model một lần.')
                startup[cfg.engine] = signature
                name = cfg.run_name or cfg.engine
                directory = _run_dir(workdir, 'ocr', name, cfg)
                if source_sha:
                    directory = directory.with_name(directory.name + '_' + source_sha[:12])
                prior_dir = Path(prior_pairs.get(book, {}).get(cfg.engine, directory))
                prior_manifest = prior_dir / 'run_manifest.json'
                if prior_manifest.is_file() and prior_dir.resolve().is_relative_to(workdir):
                    cached = json.loads(prior_manifest.read_text(encoding='utf-8'))
                    source_info = cached.get('books', {}).get(book, {})
                    if corpus_model_key(cached['config']) == corpus_model_key(cfg) and source_info.get('src') == str(Path(entry['path']).resolve()) and (source_info.get('sha256', source_sha) == source_sha):
                        directory = prior_dir
                directory.mkdir(parents=True, exist_ok=True)
                tasks = [{'book': book, 'src': str(Path(entry['path']).resolve()), 'page': p} for p in pages]
                (directory / 'tasks.jsonl').write_text(''.join((json.dumps(t, ensure_ascii=False) + '\n' for t in tasks)), encoding='utf-8')
                manifest_path = directory / 'run_manifest.json'
                old = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
                existing = load_results(directory / 'ocr_raw', book, name)
                count = len(set(existing) & set(pages))
                write_json(manifest_path, {**old, 'kind': 'ocr', 'config': cfg.to_dict(), 'created': old.get('created', now()), 'updated': now(), 'status': 'done' if count == len(pages) else 'pending', 'books': {book: {'src': tasks[0]['src'], 'pages': pages, 'sha256': source_sha, 'legacy_source_unverified': bool(existing) and (old.get('books', {}).get(book, {}).get('legacy_source_unverified', False) or not old.get('books', {}).get(book, {}).get('sha256'))}}, 'tasks_total': len(pages), 'tasks_selected': len(pages), 'counts': {'pages': len(pages), 'done': count, 'missing': len(pages) - count}})
                pair[cfg.engine] = str(directory)
            pairs.append(pair)
        manifest = workdir / 'manifest.json'
        write_json(manifest, {'workflow': 'notebook_sequential_models', 'created': now(), 'models': list(models), 'total_books': len(pairs), 'total_pages': sum(map(len, pages_h.values())), 'pairs': pairs, 'cache_renders': bool(cache_renders), 'render_cache': str(workdir / 'render_cache')})
    return manifest

def engine_progress(manifest_path: str | Path, engine: str, *, attempt_id=None) -> dict:
    campaign = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    if attempt_id:
        snapshot = Path(manifest_path).parent / f'{engine}_progress.json'
        try:
            state = json.loads(snapshot.read_text(encoding='utf-8'))
            if state.get('attempt') == attempt_id and state.get('engine') == engine and (state.get('pages') == campaign['total_pages']) and (type(state.get('done')) is int) and (0 <= state['done'] <= state['pages']) and (state.get('missing') == state['pages'] - state['done']):
                return {key: state[key] for key in ('pages', 'done', 'missing')}
        except (OSError, ValueError, KeyError):
            pass
        return None
    done = 0
    for pair in campaign['pairs']:
        directory = Path(pair[engine])
        tasks = [json.loads(line) for line in (directory / 'tasks.jsonl').read_text(encoding='utf-8').splitlines() if line]
        pages = {t['page'] for t in tasks}
        cfg = json.loads((directory / 'run_manifest.json').read_text(encoding='utf-8'))['config']
        done += len(pages & load_results(directory / 'ocr_raw', pair['book'], cfg.get('run_name') or engine).keys())
    return {'pages': campaign['total_pages'], 'done': done, 'missing': campaign['total_pages'] - done}

def _terminate_worker(process: subprocess.Popen, *, wsl_distro='', pid_file=None, token='') -> None:
    if wsl_distro and pid_file and Path(pid_file).exists():
        try:
            marker = json.loads(Path(pid_file).read_text(encoding='utf-8'))
            if marker.get('token') == token and isinstance(marker.get('pgid'), int) and (marker['pgid'] > 1):
                subprocess.run(['wsl.exe', '--distribution', wsl_distro, '--exec', 'kill', '-KILL', '--', f"-{marker['pgid']}"], capture_output=True, timeout=15, check=True)
        except (OSError, ValueError, subprocess.SubprocessError) as exc:
            print(f'Không xác nhận được việc dừng worker WSL: {exc}')
    if process.poll() is not None:
        return
    if os.name == 'nt':
        subprocess.run(['taskkill', '/PID', str(process.pid), '/T', '/F'], capture_output=True)
    else:
        os.killpg(process.pid, signal.SIGTERM)
    process.wait(timeout=15)

def run_corpus_engine(manifest_path: str | Path, engine: str) -> dict:
    """Blocking notebook call. The child process owns and releases the GPU model."""
    if engine not in {'hunyuan', 'paddle', 'ppocr', 'tonggu'}:
        raise ValueError('engine phải là hunyuan, paddle, ppocr hoặc tonggu')
    manifest_path = Path(manifest_path).resolve()
    campaign = json.loads(manifest_path.read_text(encoding='utf-8'))
    first = json.loads((Path(campaign['pairs'][0][engine]) / 'run_manifest.json').read_text(encoding='utf-8'))['config']
    budget = float(first.get('budget_h', 24)) * 3600
    max_restarts = int(first.get('max_restarts', 5))
    progress = engine_progress(manifest_path, engine)
    if progress['missing'] == 0:
        print(f"{engine}: đã có đủ {progress['done']:,} trang cùng cấu hình; dùng lại kết quả.")
        return progress
    with gpu_lock(manifest_path.parent):
        log = manifest_path.parent / f'{engine}_notebook.log'
        state_path = manifest_path.parent / 'status.json'
        started = time.monotonic()
        environment = os.environ.copy()
        environment.update(PYTHONUTF8='1', PYTHONIOENCODING='utf-8')
        environment['VTM_OCR_ATTEMPT'] = uuid.uuid4().hex
        environment['PYTHONPATH'] = os.pathsep.join([str(NOTEBOOK_WORKER_FILE.parent), *first.get('worker_pythonpath', []), environment.get('PYTHONPATH', '')])
        restarts = 0
        with tqdm(total=progress['pages'], initial=progress['done'], desc=f'OCR {engine}', unit='trang') as bar:
            while True:
                worker_token = uuid.uuid4().hex
                command, pid_file = corpus_worker_command(first, manifest_path, engine, worker_token)
                command.extend(['--attempt', environment['VTM_OCR_ATTEMPT']])
                offset = log.stat().st_size if log.exists() else 0
                with log.open('a', encoding='utf-8') as output:
                    process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT, env=environment, start_new_session=os.name != 'nt')
                try:
                    while process.poll() is None:
                        time.sleep(1)
                        with log.open(encoding='utf-8', errors='replace') as stream:
                            stream.seek(offset)
                            while True:
                                position = stream.tell()
                                line = stream.readline()
                                if not line or not line.endswith('\n'):
                                    stream.seek(position)
                                    break
                                if line.startswith(('[model]', '[corpus]')):
                                    bar.write(line.rstrip())
                            offset = stream.tell()
                        progress = engine_progress(manifest_path, engine, attempt_id=environment['VTM_OCR_ATTEMPT']) or progress
                        bar.n = progress['done']
                        bar.refresh()
                        write_json(state_path, {'status': 'running', 'active_engine': engine, 'workflow': 'notebook_sequential_models', 'pid': os.getpid(), 'worker_pid': process.pid, 'updated': now(), **progress})
                        if time.monotonic() - started > budget:
                            raise TimeoutError(f'{engine}: vượt ngân sách {budget / 3600:g} giờ; kết quả đã xong được giữ lại.')
                    code = process.wait()
                    with log.open(encoding='utf-8', errors='replace') as stream:
                        stream.seek(offset)
                        for line in stream:
                            if line.startswith(('[model]', '[corpus]')):
                                bar.write(line.rstrip())
                except BaseException:
                    _terminate_worker(process, wsl_distro=first.get('worker_wsl_distro', ''), pid_file=pid_file, token=worker_token)
                    write_json(state_path, {'status': 'interrupted', 'active_engine': None, 'updated': now()})
                    raise
                if code == 124 and restarts < max_restarts:
                    restarts += 1
                    bar.write(f'{engine}: khởi động lại sau timeout ({restarts}/{max_restarts}).')
                    continue
                if code != 0:
                    tail = '\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-16:])
                    write_json(state_path, {'status': 'error', 'active_engine': None, 'updated': now()})
                    raise RuntimeError(f'{engine} worker lỗi (exit={code}). Log: {log}\n{tail}')
                progress = engine_progress(manifest_path, engine)
                bar.n = progress['done']
                bar.refresh()
                break
        write_json(state_path, {'status': 'stage_complete' if not progress['missing'] else 'partial', 'active_engine': None, 'completed_engine': engine, 'updated': now(), **progress})
    print(f"{engine}: {progress['done']:,}/{progress['pages']:,} trang có output; worker đã thoát và giải phóng model.")
    return progress


def corpus_worker_command(first, manifest_path, engine, token):
    """Launch only the standalone worker displayed in this notebook."""
    distro = first.get("worker_wsl_distro", "")
    if distro:
        if os.name != "nt" or not first.get("worker_python", "").startswith("/"):
            raise ValueError("WSL cần Windows và worker_python là đường dẫn Linux tuyệt đối")
        pid_file = manifest_path.parent / f"{engine}_wsl_pid.json"
        return ["wsl.exe", "--distribution", distro, "--exec", first["worker_python"], "-u",
                to_wsl_path(NOTEBOOK_WORKER_FILE), "--manifest", to_wsl_path(manifest_path),
                "--engine", engine, "--token", token, "--pid-file", to_wsl_path(pid_file)], pid_file
    return [first.get("worker_python") or sys.executable, "-u", str(NOTEBOOK_WORKER_FILE),
            "--manifest", str(manifest_path), "--engine", engine], None

import unicodedata

class AlignmentLimitError(ValueError):
    """The exact alignment exceeds the configured CPU/memory budget."""

def comparison_text(text: str, mode: str='visible') -> str:
    if mode not in {'han', 'visible'}:
        raise ValueError('mode must be han or visible')
    if mode == 'visible':
        return ''.join((c for c in text if not c.isspace()))
    return ''.join((c for c in text if ord(c) >= 13312 and unicodedata.name(c, '').startswith(('CJK UNIFIED IDEOGRAPH-', 'CJK COMPATIBILITY IDEOGRAPH-'))))

def align_reference_lines(reference, candidate, lines, *, max_cells=8000000):
    """Constrain star alignment with unique text anchors inside reference lines.

    This is a text heuristic, not geometric correspondence. Keep pairwise edit
    distances separate: these forced anchors need not give a global minimum.
    Ambiguous anchors, reordered lines, or weak evidence use the exact fallback.
    """
    filtered = [comparison_text(line) for line in lines]
    if ''.join(filtered) != reference:
        return (None, {'status': 'fallback', 'reason': 'line_text_mismatch'})
    anchors, offset = ([], 0)
    for line in filtered:
        match = None
        for width in range(min(8, len(line)), 3, -1):
            for start in range(len(line) - width + 1):
                token = line[start:start + width]
                if not all((c.isalnum() for c in token)):
                    continue
                left, right = (reference.find(token), candidate.find(token))
                if left == offset + start and reference.find(token, left + 1) < 0 and (right >= 0) and (candidate.find(token, right + 1) < 0):
                    match = (left, right, width)
                    break
            if match:
                break
        if match:
            anchors.append(match)
        offset += len(line)
    chains = []
    for i, anchor in enumerate(anchors):
        previous = [chains[j] for j in range(i) if anchors[j][1] + anchors[j][2] <= anchor[1]]
        chains.append(max(previous, key=len, default=[]) + [anchor])
    selected = max(chains, key=len, default=[])
    if len(selected) < 2 or len(selected) < 0.75 * len(anchors):
        return (None, {'status': 'fallback', 'reason': 'insufficient_monotone_line_anchors', 'anchors': len(selected)})
    aligned, left, right = ([], 0, 0)
    for x, y, width in selected:
        aligned.extend(align_characters(reference[left:x], candidate[right:y], max_cells=max_cells))
        aligned.extend(((c, c) for c in reference[x:x + width]))
        left, right = (x + width, y + width)
    aligned.extend(align_characters(reference[left:], candidate[right:], max_cells=max_cells))
    return (aligned, {'status': 'anchored', 'anchors': len(selected), 'segments': len(selected) + 1})

def align_characters(left: str, right: str, *, max_cells: int=8000000) -> list[tuple[str, str]]:
    """Unit-cost Levenshtein alignment; an empty string denotes an alignment gap.

    Trim equal ends, keep only two cost rows and one traceback byte per cell.
    Equally optimal paths prefer diagonal, deletion, then insertion. Repeated
    characters can admit multiple optimal alignments; positions are indicative.
    """
    if isinstance(max_cells, bool) or not isinstance(max_cells, int) or max_cells < 1:
        raise ValueError('max_cells must be a positive integer')
    prefix = 0
    while prefix < min(len(left), len(right)) and left[prefix] == right[prefix]:
        prefix += 1
    end_l, end_r = (len(left), len(right))
    while end_l > prefix and end_r > prefix and (left[end_l - 1] == right[end_r - 1]):
        end_l -= 1
        end_r -= 1
    a, b = (left[prefix:end_l], right[prefix:end_r])
    n, m = (len(a), len(b))
    if n * m > max_cells:
        raise AlignmentLimitError(f'{n * m:,} alignment cells exceed max_cells={max_cells:,}')
    if not a or not b:
        middle = [(c, '') for c in a] + [('', c) for c in b]
    else:
        trace = [bytearray(m + 1) for _ in range(n + 1)]
        previous = list(range(m + 1))
        for i, ca in enumerate(a, 1):
            current = [i] + [0] * m
            for j, cb in enumerate(b, 1):
                diagonal = previous[j - 1] + (ca != cb)
                deletion, insertion = (previous[j] + 1, current[j - 1] + 1)
                best = min(diagonal, deletion, insertion)
                current[j] = best
                trace[i][j] = 0 if best == diagonal else 1 if best == deletion else 2
            previous = current
        middle = []
        i, j = (n, m)
        while i or j:
            direction = trace[i][j] if i and j else 1 if i else 2
            if direction == 0:
                middle.append((a[i - 1], b[j - 1]))
                i, j = (i - 1, j - 1)
            elif direction == 1:
                middle.append((a[i - 1], ''))
                i -= 1
            else:
                middle.append(('', b[j - 1]))
                j -= 1
        middle.reverse()
    return [(c, c) for c in left[:prefix]] + middle + [(c, c) for c in left[end_l:]]


from bisect import bisect_right
from math import isfinite
from statistics import median

def line_rectangle(polygon):
    if not isinstance(polygon, (list, tuple)):
        return None
    if len(polygon) == 4 and all((isinstance(v, (int, float)) for v in polygon)):
        x1, y1, x2, y2 = polygon
    else:
        if len(polygon) < 4 or any((not isinstance(p, (list, tuple)) or len(p) != 2 for p in polygon)):
            return None
        if any((not isinstance(v, (int, float)) or not isfinite(v) for p in polygon for v in p)):
            return None
        x1, y1 = (min((p[0] for p in polygon)), min((p[1] for p in polygon)))
        x2, y2 = (max((p[0] for p in polygon)), max((p[1] for p in polygon)))
    if not all((isfinite(v) for v in (x1, y1, x2, y2))) or x1 >= x2 or y1 >= y2:
        return None
    return (x1, y1, x2, y2)

def order_ppocr_columns(raw, *, mode='auto'):
    """Return comparison geometry; the original checkpoint remains untouched.

    Auto requires at least two tall detections and 70% of detected characters
    in tall detections. Unknown/missing geometry keeps detection order. Within
    a shared column, fragments are read top to bottom. Every character offset
    is moved with its original box, even when the same glyph occurs repeatedly.
    """
    if mode not in {'auto', 'vertical_rtl', 'detection'}:
        raise ValueError('reading_order must be auto, vertical_rtl or detection')
    info = {'mode': 'detection', 'changed': False}
    lines = raw.get('text_lines', [])
    if mode == 'detection' or not lines:
        return (raw, info)
    texts = [comparison_text(line.get('text', '')) for line in lines]
    if ''.join(texts) != comparison_text(raw.get('text', '')):
        return (raw, {**info, 'reason': 'line_text_mismatch'})
    boxes = [line_rectangle(line.get('polygon')) for line in lines]
    if any((box is None for box in boxes)):
        return (raw, {**info, 'reason': 'incomplete_line_geometry'})
    vertical = [i for i, (x1, y1, x2, y2) in enumerate(boxes) if y2 - y1 >= 1.5 * (x2 - x1)]
    total = sum(map(len, texts))
    if mode == 'auto' and (len(vertical) < 2 or not total or sum((len(texts[i]) for i in vertical)) < 0.7 * total):
        return (raw, info)
    tolerance = 0.45 * median((box[2] - box[0] for box in boxes))
    groups = []
    for index in sorted(range(len(lines)), key=lambda i: -(boxes[i][0] + boxes[i][2]) / 2):
        center = (boxes[index][0] + boxes[index][2]) / 2
        if groups and abs(groups[-1][0] - center) <= tolerance:
            groups[-1][1].append(index)
        else:
            groups.append((center, [index]))
    order = [index for _, group in groups for index in sorted(group, key=lambda i: boxes[i][1])]
    starts, offset = ([], 0)
    for text in texts:
        starts.append(offset)
        offset += len(text)
    new_starts, offset = ({}, 0)
    for index in order:
        new_starts[index] = offset
        offset += len(texts[index])
    characters = []
    for entry in raw.get('character_boxes', []):
        old = entry.get('offset')
        if not isinstance(old, int) or old < 0 or old >= total:
            continue
        index = bisect_right(starts, old) - 1
        local = old - starts[index]
        if texts[index][local:local + 1] == entry.get('text'):
            characters.append({**entry, 'offset': new_starts[index] + local})
    geometry = {**raw, 'text': '\n'.join((lines[i]['text'] for i in order)), 'text_lines': [lines[i] for i in order], 'character_boxes': sorted(characters, key=lambda entry: entry['offset'])}
    return (geometry, {'mode': 'vertical_rtl', 'changed': order != list(range(len(lines))), 'line_order': order})


from collections import Counter
import re
from statistics import median

def ocr_quality_issues(text, raw):
    issues = list(raw.get('quality_issues', []))
    if raw.get('hit_max'):
        issues.append('token_limit')
    visible = comparison_text(text)
    if re.search('(.{1,40}?)\\1{11,}', visible):
        match = max((m.end() - m.start() for m in re.finditer('(.{1,40}?)\\1{11,}', visible)), default=0)
        if match >= 80:
            issues.append('repetitive_output')
    if len(visible) >= 80:
        lines = [comparison_text(line) for line in text.splitlines()]
        signatures = [re.sub('[0-9０-９]+', '0', line) for line in lines if len(line) >= 12]
        if signatures and max(Counter(signatures).values()) >= 6:
            issues.append('repetitive_output')
        symbols = sum((not c.isalnum() for c in visible))
        if symbols >= 80 and symbols / len(visible) > 0.5:
            issues.append('symbol_dominated_output')
    return sorted(set(issues))

def page_quality_issues(records):
    issues = {name: list(record.get('quality_issues', [])) for name, record in records.items() if record is not None}
    lengths = {name: record['visible'] for name, record in records.items() if record is not None and (not issues[name]) and (record['visible'] > 0)}
    for name, length in lengths.items():
        peers = [value for other, value in lengths.items() if other != name]
        if len(peers) < 2 or max(peers) > 1.6 * min(peers):
            continue
        typical = median(peers)
        if typical >= 80 and length < 0.35 * typical:
            issues[name].append('suspected_incomplete_output')
        elif length > max(3 * typical, typical + 200):
            issues[name].append('length_outlier')
    return {name: sorted(set(reasons)) for name, reasons in issues.items() if reasons}


from collections import Counter
from itertools import combinations
from math import log2

def _pair_map(reference, candidate, max_cells, *, alignment=None):
    insertions = [''] * (len(reference) + 1)
    characters = [''] * len(reference)
    index = 0
    for a, b in alignment if alignment is not None else align_characters(reference, candidate, max_cells=max_cells):
        if a:
            characters[index] = b
            index += 1
        else:
            insertions[index] += b
    return (insertions, characters)

def _star_columns(texts, reference, max_cells):
    """Align each sequence to a fixed anchor; merge its insertion slots."""
    maps = {name: _pair_map(texts[reference], text, max_cells) for name, text in texts.items()}
    columns = []
    for slot in range(len(texts[reference]) + 1):
        width = max((len(value[0][slot]) for value in maps.values()))
        for index in range(width):
            columns.append({name: value[0][slot][index:index + 1] for name, value in maps.items()})
        if slot < len(texts[reference]):
            columns.append({name: value[1][slot] for name, value in maps.items()})
    return columns

def multi_character_consensus(candidates: dict[str, str | None], *, reference='ppocr', mode='visible', max_cells=8000000, excluded_models=None, reference_lines=None) -> dict:
    """Keep missing engines distinct from a real empty output and alignment gaps.

    This is star alignment, not a globally optimal multiple-sequence alignment.
    Insertions are aligned locally to their longest candidate before merging.
    A strict majority requires > half of ALL configured models; a missing model
    never votes. Votes are evidence, not calibrated probabilities or truth.
    """
    if len(candidates) < 2:
        raise ValueError('At least two configured models are required')
    texts = {name: comparison_text(text, mode) if text is not None else None for name, text in candidates.items()}
    excluded = dict(excluded_models or {})
    if set(excluded) - candidates.keys():
        raise ValueError('Excluded models must be configured candidates')
    available = {name: text for name, text in texts.items() if text is not None and name not in excluded}
    result = {'models': list(candidates), 'texts': texts, 'reference': reference, 'alignment': 'star', 'status': 'missing_output', 'columns': [], 'conflicts': [], 'candidate_scores': {}, 'pairwise_distances': {}, 'missing_models': [name for name, text in texts.items() if text is None], 'excluded_models': excluded, 'line_alignment': {}, 'mean_entropy_bits': None, 'disagreement_pct': None}
    align_characters('', '', max_cells=max_cells)
    if not available or (excluded and len(available) < 2):
        if excluded:
            result['status'] = 'quality_review'
        return result
    if reference not in available or not available[reference]:
        reference = max(available, key=lambda name: len(available[name]))
    result['reference'] = reference
    if not any(available.values()):
        result['status'] = 'both_empty' if len(available) == 2 else 'all_empty'
        return result
    distances = {name: [] for name in available}
    reference_alignments = {}
    try:
        for a, b in combinations(available, 2):
            left, right = (b, a) if b == reference else (a, b)
            aligned = align_characters(available[left], available[right], max_cells=max_cells)
            if left == reference:
                reference_alignments[right] = aligned
            denominator = max(len(available[a]), len(available[b]))
            distance = sum((x != y for x, y in aligned)) / denominator if denominator else None
            result['pairwise_distances'][f'{a}|{b}'] = distance
            if distance is not None:
                distances[a].append(distance)
                distances[b].append(distance)
        if reference_lines:
            for name in available:
                if name == reference:
                    continue
                anchored, evidence = align_reference_lines(available[reference], available[name], reference_lines, max_cells=max_cells)
                result['line_alignment'][name] = evidence
                if anchored is not None:
                    reference_alignments[name] = anchored
                    result['alignment'] = 'star_reference_line_anchors'
        maps = {reference: ([''] * (len(available[reference]) + 1), list(available[reference]))}
        maps.update({name: _pair_map(available[reference], text, max_cells, alignment=reference_alignments[name]) for name, text in available.items() if name != reference})
        aligned_columns = []
        for slot in range(len(available[reference]) + 1):
            inserted = {name: value[0][slot] for name, value in maps.items()}
            if any(inserted.values()):
                longest = max(inserted, key=lambda name: len(inserted[name]))
                aligned_columns.extend(_star_columns(inserted, longest, max_cells))
            if slot < len(available[reference]):
                aligned_columns.append({name: value[1][slot] for name, value in maps.items()})
    except AlignmentLimitError as exc:
        result.update(status='alignment_limit', alignment_error=str(exc), pairwise_distances={})
        return result
    result['candidate_scores'] = {name: sum(values) / len(values) if values else None for name, values in distances.items()}
    offsets = dict.fromkeys(available, 0)
    for index, values in enumerate(aligned_columns):
        votes = Counter(values.values())
        best, support = votes.most_common(1)[0]
        majority = best if support > len(candidates) / 2 else None
        entropy = -sum((n / len(available) * log2(n / len(available)) for n in votes.values()))
        column = {'index': index, 'predictions': {name: values.get(name) for name in candidates}, 'offsets': {name: offsets.get(name) if values.get(name) else None for name in candidates}, 'votes': dict(votes), 'entropy_bits': entropy, 'majority': majority, 'support': support, 'configured_models': len(candidates), 'available_models': len(available), 'disagreement': len(votes) > 1, 'localization': None}
        result['columns'].append(column)
        if column['disagreement']:
            result['conflicts'].append(column)
        for name, value in values.items():
            offsets[name] += len(value)
    result.update(status='partial' if result['missing_models'] or excluded else 'compared', mean_entropy_bits=sum((c['entropy_bits'] for c in result['columns'])) / len(aligned_columns), disagreement_pct=100 * len(result['conflicts']) / len(aligned_columns))
    return result


import hashlib
import json
import unicodedata
from pathlib import Path

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8-sig'))

def read_jsonl(path: Path, *, allow_incomplete_tail: bool=False) -> list[dict]:
    rows = []
    lines = path.read_text(encoding='utf-8-sig').splitlines(keepends=True)
    for number, line in enumerate(lines, 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            if allow_incomplete_tail and number == len(lines) and (not line.endswith('\n')):
                break
            raise ValueError(f'Invalid JSON: {path}:{number}: {exc.msg}') from exc
    return rows

def count_text(text: str) -> tuple[int, int]:
    """Return (Han ideographs, non-whitespace characters), without normalizing glyphs."""
    visible = sum((not c.isspace() for c in text))
    han = sum((ord(c) >= 13312 and unicodedata.name(c, '').startswith(('CJK UNIFIED IDEOGRAPH-', 'CJK COMPATIBILITY IDEOGRAPH-')) for c in text))
    return (han, visible)

def prepare_ocr_record(record, engine, *, reading_order='auto'):
    """Normalize a comparison copy, keeping the exact checkpoint in raw."""
    geometry, order = order_ppocr_columns(record, mode=reading_order) if engine == 'ppocr' and (not record.get('region_order_applied')) else (record, {'mode': 'model', 'changed': False})
    text = geometry['text'] if engine in {'ppocr', 'tonggu'} else '\n'.join(page_lines(record))
    han, visible = count_text(text)
    return {'han': han, 'visible': visible, 'text': text, 'raw': record, 'geometry': geometry, 'reading_order': order, 'quality_issues': ocr_quality_issues(text, record), 'hit_max': record.get('hit_max'), 'sec': record.get('sec')}

def load_run(directory: Path, engine: str, warnings: list[str], *, reading_order='auto') -> dict:
    manifest = read_json(directory / 'run_manifest.json')
    cfg = manifest.get('config', {})
    if manifest.get('kind') != 'ocr' or cfg.get('engine') != engine:
        raise ValueError(f'Expected an OCR {engine} run: {directory}')
    name = cfg.get('run_name') or engine
    records = {}
    for path in sorted((directory / 'ocr_raw').glob('*/*.jsonl')):
        if not path.name.startswith(f'{name}_shard'):
            continue
        for record in read_jsonl(path, allow_incomplete_tail=manifest.get('status') == 'running'):
            key = (path.parent.name, int(record['page']))
            if key[1] < 0:
                raise ValueError(f'Negative page in {path}')
            if key in records:
                warnings.append(f'{engine}: duplicate {key}; last record in sorted file order retained.')
            records[key] = prepare_ocr_record(record, engine, reading_order=reading_order)
    tasks = directory / 'tasks.jsonl'
    if tasks.exists():
        selected = {(str(t['book']), int(t['page'])) for t in read_jsonl(tasks)}
        extras = records.keys() - selected
        if extras:
            warnings.append(f'{engine}: {len(extras)} cached pages outside tasks.jsonl excluded.')
    else:
        selected = set(records)
        warnings.append(f'{engine}: tasks.jsonl absent; cannot detect pages with no output in either model.')
    if not records:
        warnings.append(f'{engine}: no OCR page records found.')
    if manifest.get('resumed_files') or cfg.get('resume_from'):
        warnings.append(f'{engine}: resumed outputs; manifest config may not describe every copied record.')
    return {'manifest': manifest, 'selected': selected, 'records': records}

def check_sources(left: dict, right: dict, warnings: list[str], *, digest_cache=None) -> list[dict]:
    """Hash available PDFs so equal book names cannot silently pair different files."""
    books_h = left['manifest'].get('books', {})
    books_p = right['manifest'].get('books', {})
    cache = digest_cache if digest_cache is not None else {}

    def digest(source: str | None) -> str | None:
        if not source:
            return None
        path = Path(source)
        if not path.is_file():
            return None
        key = str(path.resolve())
        if key not in cache:
            hasher = hashlib.sha256()
            with path.open('rb') as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                    hasher.update(chunk)
            cache[key] = hasher.hexdigest()
        return cache[key]
    checks = []
    for book in sorted({b for b, _ in left['selected']} & {b for b, _ in right['selected']}):
        h = digest(books_h.get(book, {}).get('src'))
        p = digest(books_p.get(book, {}).get('src'))
        if h and p and (h != p):
            raise ValueError(f'Different source PDFs for book {book!r}; do not compare these runs.')
        verified = bool(h and p)
        for info, actual in ((books_h.get(book, {}), h), (books_p.get(book, {}), p)):
            if actual and info.get('sha256') and (info['sha256'] != actual):
                raise ValueError(f'Source PDF changed since OCR for book {book!r}')
            if info.get('legacy_source_unverified'):
                warnings.append(f'{book}: legacy cached pages have no original source hash; check source provenance.')
        checks.append({'book': book, 'source_verified': verified, 'sha256': h if verified else None})
        if not verified:
            warnings.append(f'{book}: original PDFs unavailable; source identity unverified (matching book/page only).')
    return checks


from math import ceil, floor, isfinite

def normalized_character_box(entry: dict, width: int, height: int):
    """Validate a model-estimated character rectangle/polygon, in input pixels."""
    if entry.get('granularity') != 'character' or len(entry.get('text', '')) != 1 or width <= 0 or (height <= 0):
        return None
    return normalized_polygon(entry.get('polygon'), width, height)

def normalized_polygon(points, width: int, height: int):
    """Normalize an actual OCR polygon without inferring character boundaries."""
    if width <= 0 or height <= 0:
        return None
    if not isinstance(points, (list, tuple)):
        return None
    if len(points) == 4 and all((isinstance(x, (int, float)) for x in points)):
        x1, y1, x2, y2 = points
        points = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
    if len(points) < 4 or any((not isinstance(p, (list, tuple)) or len(p) != 2 for p in points)):
        return None
    if any((not isinstance(v, (int, float)) or not isfinite(v) for point in points for v in point)):
        return None
    xs, ys = ([p[0] for p in points], [p[1] for p in points])
    if min(xs) < 0 or min(ys) < 0 or max(xs) > width or (max(ys) > height) or (min(xs) >= max(xs)) or (min(ys) >= max(ys)):
        return None
    return {'bbox': [min(xs) / width, min(ys) / height, max(xs) / width, max(ys) / height], 'polygon': [[x / width, y / height] for x, y in points]}

def locator_character_boxes(consensus: dict, records: dict, locator='ppocr'):
    record = records.get(locator)
    lookup = {}
    if locator in consensus.get('excluded_models', {}):
        return lookup
    if record and record.get('geometry', record['raw']).get('coordinate_frame') == 'input_image':
        raw = record.get('geometry', record['raw'])
        for entry in raw.get('character_boxes', []):
            offset = entry.get('offset')
            if not isinstance(offset, int) or offset < 0:
                continue
            text = consensus['texts'].get(locator) or ''
            if text[offset:offset + 1] != entry.get('text'):
                continue
            box = normalized_character_box(entry, raw.get('w', 0), raw.get('h', 0))
            if box:
                if offset in lookup:
                    lookup[offset] = None
                else:
                    lookup[offset] = {**box, 'model': locator, 'method': entry.get('method'), 'estimated': True, 'offset': offset}
    return lookup

def attach_character_regions(consensus: dict, records: dict, *, locator='ppocr') -> None:
    """Use one locator model's coordinates; never fabricate boxes for its gaps.

    The votes from other models are linked by string alignment, not independent
    detections of that glyph. A box is still a model estimate, not a human label.
    """
    lookup = locator_character_boxes(consensus, records, locator)

    def stable(column):
        values = [v for v in column['predictions'].values() if v is not None]
        return len(values) >= 2 and len(set(values)) == 1 and all(values)

    def unique_anchor(value):
        if not value:
            return False
        for model, text in consensus['texts'].items():
            if text is None or model in consensus.get('excluded_models', {}):
                continue
            first = text.find(value)
            if first < 0 or text.find(value, first + 1) >= 0:
                return False
        return True
    columns = consensus['columns']
    for index, column in enumerate(columns):
        offset = column['offsets'].get(locator)
        location = lookup.get(offset)
        if location and column['disagreement']:
            before, after = ('', '')
            for neighbor in reversed(columns[max(0, index - 8):index]):
                if not stable(neighbor):
                    break
                before = neighbor['predictions'][locator] + before
            for neighbor in columns[index + 1:index + 9]:
                if not stable(neighbor):
                    break
                after += neighbor['predictions'][locator]
            supported = (index == 0 or unique_anchor(before)) and (index == len(columns) - 1 or unique_anchor(after))
            if not supported:
                location = None
        column['localization'] = location
        column['localization_status'] = 'model_estimated_character' if location else 'unresolved'

def span_has_shared_anchors(consensus, start, stop, locator):
    """Validate a disputed run as a whole, including adjacent disagreements."""
    columns = consensus['columns']

    def stable(column):
        values = [v for v in column['predictions'].values() if v is not None]
        return len(values) >= 2 and len(set(values)) == 1 and all(values)

    def unique(value):
        if not value:
            return False
        for model, text in consensus['texts'].items():
            if text is None or model in consensus.get('excluded_models', {}):
                continue
            position = text.find(value)
            if position < 0 or text.find(value, position + 1) >= 0:
                return False
        return True
    before, after = ('', '')
    for column in reversed(columns[max(0, start - 8):start]):
        if not stable(column):
            break
        before = (column['predictions'].get(locator) or '') + before
    for column in columns[stop:stop + 8]:
        if not stable(column):
            break
        after += column['predictions'].get(locator) or ''
    return (start == 0 or unique(before)) and (stop == len(columns) or unique(after))

def union_region(locations, *, locator, method, granularity):
    x1 = min((location['bbox'][0] for location in locations))
    y1 = min((location['bbox'][1] for location in locations))
    x2 = max((location['bbox'][2] for location in locations))
    y2 = max((location['bbox'][3] for location in locations))
    return {'bbox': [x1, y1, x2, y2], 'polygon': [[x1, y1], [x2, y1], [x2, y2], [x1, y2]], 'model': locator, 'method': method, 'granularity': granularity, 'estimated': True}

def attach_conflict_regions(consensus: dict, records: dict, *, locator='ppocr') -> None:
    """Group disputed columns and attach measured character or line-context boxes.

    A line crop is context only, including when the locator omitted a character.
    It never upgrades the individual column's unresolved character coordinates.
    """
    columns, groups = (consensus['columns'], [])
    for column in columns:
        if not column['disagreement']:
            continue
        if groups and groups[-1][-1]['index'] + 1 == column['index']:
            groups[-1].append(column)
        else:
            groups.append([column])
    lookup = locator_character_boxes(consensus, records, locator)
    record = records.get(locator) or {}
    raw = record.get('geometry', record.get('raw', {})) if locator not in consensus.get('excluded_models', {}) else {}
    lines, offset = ([], 0)
    text_lines = raw.get('text_lines', [])
    if raw.get('coordinate_frame') == 'input_image' and ''.join((comparison_text(line.get('text', '')) for line in text_lines)) == consensus['texts'].get(locator):
        for line in text_lines:
            length = len(comparison_text(line.get('text', '')))
            box = normalized_polygon(line.get('polygon'), raw.get('w', 0), raw.get('h', 0))
            if length and box:
                lines.append({**box, 'start': offset, 'stop': offset + length})
            offset += length
    regions = []
    for number, group in enumerate(groups, 1):
        start, stop = (group[0]['index'], group[-1]['index'] + 1)
        offsets = [column['offsets'].get(locator) for column in group]
        present = [value for value in offsets if value is not None]
        region = {'number': number, 'start': start, 'stop': stop, 'columns': [c['index'] for c in group], 'predictions': {name: None if text is None or name in consensus.get('excluded_models', {}) else ''.join((c['predictions'][name] or '' for c in group)) for name, text in consensus['texts'].items()}, 'localization': None, 'localization_status': 'unresolved'}
        if span_has_shared_anchors(consensus, start, stop, locator):
            if len(present) == len(offsets) and all((lookup.get(value) for value in present)):
                locations = [lookup[value] for value in present]
                location = dict(locations[0], granularity='character') if len(locations) == 1 else union_region(locations, locator=locator, method='ppocr_character_box_union', granularity='character_group')
                location['source_offsets'] = present
                region.update(localization=location, localization_status='model_estimated_character_group')
            else:
                if present:
                    selected = [line for line in lines if line['start'] <= max(present) and line['stop'] > min(present)]
                    if any((not any((line['start'] <= value < line['stop'] for line in selected)) for value in present)):
                        selected = []
                else:
                    anchors = [columns[i]['offsets'].get(locator) for i in (start - 1, stop) if 0 <= i < len(columns)]
                    anchors = [value for value in anchors if value is not None]
                    selected = [line for line in lines if anchors and all((line['start'] <= value < line['stop'] for value in anchors))]
                if selected:
                    region.update(localization=union_region(selected, locator=locator, method='ppocr_line_polygon', granularity='line_context'), localization_status='model_estimated_line_context')
        regions.append(region)
    consensus['conflict_regions'] = regions

def character_crops(image, location: dict, *, context_chars=3):
    """Crop original raster pixels, plus a context view highlighting the polygon."""
    from PIL import ImageDraw
    width, height = image.size
    x1, y1, x2, y2 = location['bbox']
    rectangle = (max(0, floor(x1 * width)), max(0, floor(y1 * height)), min(width, ceil(x2 * width)), min(height, ceil(y2 * height)))
    if rectangle[2] <= rectangle[0] or rectangle[3] <= rectangle[1]:
        raise ValueError('Invalid character crop')
    glyph = image.crop(rectangle)
    unit = min(glyph.size) if location.get('granularity') in {'character_group', 'line_context'} else max(glyph.size)
    margin = unit * context_chars
    context_rect = (max(0, rectangle[0] - margin), max(0, rectangle[1] - margin), min(width, rectangle[2] + margin), min(height, rectangle[3] + margin))
    context = image.crop(context_rect)
    polygon = [(x * width - context_rect[0], y * height - context_rect[1]) for x, y in location['polygon']]
    draw = ImageDraw.Draw(context)
    draw.line(polygon + [polygon[0]], fill='red', width=max(1, round(max(image.size) / 1200)))
    return (glyph, context, rectangle)


from collections import Counter, defaultdict
from itertools import combinations
PAGE_CONSENSUS_DEFAULTS = {'anchor_chars': 4, 'anchors_per_region': 10, 'min_region_chars': 12, 'max_unit_chars': 80, 'max_text_chars': 20000, 'max_regions': 256, 'max_anchor_occurrences': 8, 'max_global_anchors': 64}
PAGE_FINDING_LABELS = {'suspected_missing_region': 'Nghi thiếu vùng/cột', 'suspected_duplicate_region': 'Nghi lặp vùng/cột', 'suspected_reading_order': 'Nghi sai thứ tự đọc', 'suspected_extra_region': 'Văn bản dư so với đa số', 'region_text_mismatch': 'Có khung vùng nhưng chữ không khớp', 'unmatched_region_text': 'Chữ không khớp; chưa phân biệt thay thế hay dư/thiếu'}

def _page_location(box, raw, method, granularity):
    location = normalized_polygon(box, raw.get('w', 0), raw.get('h', 0))
    return {**location, 'method': method, 'granularity': granularity, 'estimated': True} if location else None

def _page_units(record, engine, options):
    """Preserve stream offsets and explicit granularity; no guessed glyph boxes."""
    text = comparison_text(record['text'])
    raw = record.get('geometry', record.get('raw', {}))
    lines = raw.get('text_lines', [])
    entries, method, granularity = ([], 'text_lines', 'text_span')
    if lines and ''.join((comparison_text(e.get('text', '')) for e in lines)) == text:
        for entry in lines:
            location = _page_location(entry.get('polygon'), raw, 'ocr_line', 'line_context') if raw.get('coordinate_frame') == 'input_image' else None
            entries.append((comparison_text(entry.get('text', '')), location))
        method, granularity = ('ocr_lines', 'line_context')
    elif raw.get('blocks'):
        blocks = ordered_page_blocks(raw['blocks'])
        values = [comparison_text('\n'.join(md_to_lines(e.get('text', '')))) for e in blocks]
        if ''.join(values) == text:
            entries = [(value, _page_location(entry.get('box'), raw, 'ocr_block', 'block_context') if raw.get('coordinate_frame') == 'input_image' else None) for value, entry in zip(values, blocks)]
            method, granularity = ('ocr_blocks', 'block_context')
    if not entries:
        entries = [(comparison_text(line), None) for line in record['text'].splitlines()]
        method, granularity = ('text_lines', 'text_span')
    if ''.join((value for value, _ in entries)) != text:
        entries = [(text, None)]
    units, offset = ([], 0)
    for value, location in entries:
        for start in range(0, len(value), options['max_unit_chars']):
            stop = min(len(value), start + options['max_unit_chars'])
            units.append({'start': offset + start, 'stop': offset + stop, 'location': location, 'method': method, 'granularity': granularity})
        offset += len(value)
    return units

def _page_anchor_index(text, width, maximum):
    positions, counts = (defaultdict(list), Counter())
    for offset in range(len(text) - width + 1):
        token = text[offset:offset + width]
        if not token.isalnum():
            continue
        counts[token] += 1
        if len(positions[token]) < maximum:
            positions[token].append(offset)
    return {'positions': positions, 'counts': counts}

def _page_anchors(text, start, stop, indexes, model, voters, options):
    width = options['anchor_chars']
    own = indexes[model]
    candidates = []
    for offset in range(start, stop - width + 1):
        token = text[offset:offset + width]
        if own['counts'].get(token) != 1:
            continue
        support = sum((bool(indexes[name]['counts'].get(token)) for name in voters))
        candidates.append((offset, token, support))
    bins = defaultdict(list)
    for entry in candidates:
        bins[min(options['anchors_per_region'] - 1, (entry[0] - start) * options['anchors_per_region'] // max(1, stop - start))].append(entry)
    return [{'offset': offset, 'text': token} for offset, token, _ in (max(values, key=lambda e: (e[2], -e[0])) for _, values in sorted(bins.items()))]

def _page_span_matches(anchors, start, stop, index, options):
    """Cluster independently located anchors into copies of a region."""
    width = options['anchor_chars']
    if any((index['counts'].get(a['text'], 0) > options['max_anchor_occurrences'] for a in anchors)):
        return {'status': 'ambiguous', 'reason': 'anchor_occurrence_limit', 'matches': []}
    hits = sorted(((position - (anchor['offset'] - start), number, position) for number, anchor in enumerate(anchors) for position in index['positions'].get(anchor['text'], [])))
    if not hits:
        return {'status': 'absent', 'copies': 0, 'matches': [], 'anchor_hits': 0}
    tolerance = max(3, (stop - start) * 0.15)
    groups = []
    for hit in hits:
        if not groups or hit[0] - groups[-1][0][0] > tolerance:
            groups.append([hit])
        else:
            groups[-1].append(hit)
    matches = []
    for group in groups:
        numbers = sorted({entry[1] for entry in group})
        if len(numbers) < 2:
            continue
        source_span = anchors[numbers[-1]]['offset'] + width - anchors[numbers[0]]['offset']
        if source_span < min(12, (stop - start) * 0.65) or source_span < 0.65 * (stop - start):
            continue
        left, right = (min((e[2] for e in group)), max((e[2] for e in group)) + width)
        if matches and left < matches[-1]['span'][1]:
            continue
        matches.append({'span': [left, right], 'anchors': len(numbers), 'source_anchor_coverage': round(source_span / (stop - start), 4)})
    return {'status': 'present', 'copies': len(matches), 'matches': matches, 'anchor_hits': len(hits)} if matches else {'status': 'ambiguous', 'reason': 'insufficient_coherent_anchors', 'matches': [], 'anchor_hits': len(hits)}

def _page_unclaimed(start, stop, claimed):
    beginning = None
    for index in range(start, stop + 1):
        free = index < stop and (not claimed[index])
        if free and beginning is None:
            beginning = index
        elif not free and beginning is not None:
            yield (beginning, index)
            beginning = None

def _page_matching_location(units, span):
    selected = [u['location'] for u in units if u.get('location') and max(u['start'], span[0]) < min(u['stop'], span[1])]
    if not selected:
        return None
    box = [min((e['bbox'][0] for e in selected)), min((e['bbox'][1] for e in selected)), max((e['bbox'][2] for e in selected)), max((e['bbox'][3] for e in selected))]
    return {'bbox': box, 'method': 'ocr_region_context', 'granularity': 'region_context', 'estimated': True}

def _page_spatial_counterpart(location, units):
    """Only compare similar-sized detected contexts, never a whole-page block."""
    if not location:
        return None
    a = location['bbox']
    area = (a[2] - a[0]) * (a[3] - a[1])
    for unit in units:
        other = unit.get('location')
        if not other:
            continue
        b = other['bbox']
        other_area = (b[2] - b[0]) * (b[3] - b[1])
        intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
        if 0.5 <= other_area / area <= 2 and intersection / (area + other_area - intersection) >= 0.5:
            return other
    return None

def _page_order_sign(first, second):
    if len(first.get('matches', [])) != 1 or len(second.get('matches', [])) != 1:
        return None
    a, b = (first['matches'][0]['span'], second['matches'][0]['span'])
    return 1 if a[1] <= b[0] else -1 if b[1] <= a[0] else None

def _page_flanking_gap(text, start, stop, own, target, target_length, width):
    """Text insertion/deletion evidence requires unique flanks or page edges."""
    left, right = (0 if start == 0 else None, target_length if stop == len(text) else None)
    flanks = []
    for side, candidates in (('left', range(start - width, max(-1, start - 80 - width), -1)), ('right', range(stop, min(len(text) - width + 1, stop + 80)))):
        if side == 'left' and start == 0 or (side == 'right' and stop == len(text)):
            continue
        for offset in candidates:
            if offset < 0 or offset + width > len(text):
                continue
            token = text[offset:offset + width]
            if own['counts'].get(token) == target['counts'].get(token) == 1:
                position = target['positions'][token][0]
                if side == 'left':
                    left = position + width
                else:
                    right = position
                flanks.append({'side': side, 'text': token, 'reference_offset': offset, 'target_offset': position})
                break
    return {'status': 'anchored' if left is not None and right is not None and (left <= right) else 'unresolved', 'gap_span': [left, right], 'gap_chars': right - left if left is not None and right is not None and (left <= right) else None, 'anchors': flanks}

def _page_global_order_events(texts, indexes, voters, quorum, raw_indexes, options):
    """Detect long-range inversions even when every backend emits one paragraph."""
    reference = max(voters, key=lambda name: len(texts[name]))
    width = options['anchor_chars']
    candidates = []
    previous = -width
    for token, positions in indexes[reference]['positions'].items():
        position = positions[0]
        if indexes[reference]['counts'][token] != 1 or position < previous + width:
            continue
        if sum((indexes[name]['counts'].get(token) == 1 for name in voters)) >= quorum:
            candidates.append((position, token))
            previous = position
    if len(candidates) > options['max_global_anchors']:
        count = options['max_global_anchors']
        candidates = [candidates[round(i * (len(candidates) - 1) / max(1, count - 1))] for i in range(count)]
    events = defaultdict(list)
    for (x, first), (y, second) in combinations(candidates, 2):
        if y - x < options['min_region_chars']:
            continue
        signs, positions = ({}, {})
        for name, index in indexes.items():
            a, b = (index['positions'].get(first, []), index['positions'].get(second, []))
            if index['counts'].get(first) == index['counts'].get(second) == 1:
                signs[name] = 1 if a[0] < b[0] else -1
                positions[name] = [a[0], b[0]]
        counts = Counter((signs[name] for name in voters if name in signs))
        winner = next((sign for sign, n in counts.items() if n >= quorum), None)
        if winner is None:
            continue
        supporters = [name for name in voters if signs.get(name) == winner]
        for name, sign in signs.items():
            if sign != winner and abs(positions[name][0] - positions[name][1]) >= options['min_region_chars']:
                events[name, 'comparison'].append({'regions': [], 'supporters': supporters, 'anchors': [first, second], 'positions': positions})
        for name, index in raw_indexes.items():
            if index['counts'].get(first) == index['counts'].get(second) == 1:
                a, b = (index['positions'][first][0], index['positions'][second][0])
                if (1 if a < b else -1) != winner and abs(a - b) >= options['min_region_chars']:
                    independent = [peer for peer in supporters if peer != name]
                    if len(independent) >= quorum:
                        events[name, 'raw_before_normalization'].append({'regions': [], 'supporters': independent, 'anchors': [first, second], 'positions': {**positions, name: [a, b]}})
    return {key: values for key, values in events.items() if len(values) >= 2 and len({token for event in values for token in event['anchors']}) >= 4}

def page_level_consensus(records, *, excluded_models=None, source_verified=False, options=None):
    settings = {**PAGE_CONSENSUS_DEFAULTS, **(options or {})}
    if set(settings) != set(PAGE_CONSENSUS_DEFAULTS):
        raise ValueError('Unknown page consensus options')
    if any((isinstance(v, bool) or not isinstance(v, int) or v < 1 for v in settings.values())):
        raise ValueError('Page consensus limits must be positive integers')
    if settings['anchors_per_region'] < 2 or settings['min_region_chars'] < 2 * settings['anchor_chars']:
        raise ValueError('Page consensus requires at least two nonoverlapping anchors per region')
    if settings['max_unit_chars'] < settings['min_region_chars']:
        raise ValueError('max_unit_chars must be >= min_region_chars')
    if len(records) < 2:
        raise ValueError('At least two configured models are required')
    excluded = excluded_models or {}
    texts = {name: comparison_text(rec['text']) if rec is not None else None for name, rec in records.items()}
    voters = [name for name, text in texts.items() if text and name not in excluded]
    quorum = len(records) // 2 + 1
    result = {'schema': 'vtm-page-consensus-v1', 'status': 'insufficient_evidence', 'complete': True, 'quorum': quorum, 'configured_models': len(records), 'voters': voters, 'source_verified': bool(source_verified), 'regions': [], 'findings': [], 'settings': settings, 'models': {name: {'status': 'missing_output' if text is None else 'empty_output' if not text else 'unassessed', 'flags': [], 'comparison_chars': len(text) if text is not None else None, 'reading_order_normalization': (records[name] or {}).get('reading_order', {}), 'excluded_from_region_votes': name in excluded} for name, text in texts.items()}, 'limitations': ['Text-anchor consensus is not OCR accuracy or ground truth.', 'Text-only spans do not establish a source-image column or a hallucination.', 'All models may omit the same content; shared omissions are not detectable here.']}
    if any((text is not None and len(text) > settings['max_text_chars'] for text in texts.values())):
        result.update(status='analysis_limit', complete=False, limit_reason='max_text_chars')
        return result
    if len(voters) < quorum:
        return result
    units = {name: _page_units(rec, name, settings) if rec is not None else [] for name, rec in records.items()}
    indexes = {name: _page_anchor_index(text or '', settings['anchor_chars'], settings['max_anchor_occurrences']) for name, text in texts.items()}
    claimed = {name: bytearray(len(text or '')) for name, text in texts.items()}
    proposals = sorted(voters, key=lambda name: (-sum((bool(u['location']) for u in units[name])), -sum((u['stop'] - u['start'] >= settings['min_region_chars'] for u in units[name])), name))
    for reference in proposals:
        for unit in units[reference]:
            for start, stop in list(_page_unclaimed(unit['start'], unit['stop'], claimed[reference])):
                if stop - start < settings['min_region_chars']:
                    continue
                anchors = _page_anchors(texts[reference], start, stop, indexes, reference, voters, settings)
                if len(anchors) < 2:
                    continue
                matches = {name: _page_span_matches(anchors, start, stop, index, settings) if texts[name] is not None else {'status': 'missing_output', 'matches': []} for name, index in indexes.items()}
                if matches[reference]['status'] != 'present':
                    continue
                if len(result['regions']) >= settings['max_regions']:
                    result.update(status='analysis_limit', complete=False, limit_reason='max_regions')
                    break
                location = unit['location']
                region = {'id': len(result['regions']) + 1, 'reference_model': reference, 'reference_span': [start, stop], 'text': texts[reference][start:stop], 'granularity': unit['granularity'], 'location': location, 'localization_status': 'model_estimated_context' if location else 'unresolved', 'anchors': anchors, 'models': matches}
                counts = Counter((matches[name].get('copies') for name in voters if 'copies' in matches[name]))
                winners = [(count, n) for count, n in counts.items() if n >= quorum]
                region['majority_copies'] = winners[0][0] if winners else None
                region['supporters'] = [name for name in voters if matches[name].get('copies') == winners[0][0]] if winners else []
                for name, match in matches.items():
                    for found in match['matches']:
                        found['location'] = _page_matching_location(units[name], found['span'])
                        left, right = found['span']
                        claimed[name][left:right] = b'\x01' * (right - left)
                claimed[reference][start:stop] = b'\x01' * (stop - start)
                result['regions'].append(region)
            if not result['complete']:
                break
        if not result['complete']:
            break

    def finding(model, kind, region_ids, supporters, **evidence):
        result['findings'].append({'model': model, 'kind': kind, 'label': PAGE_FINDING_LABELS[kind], 'suspected': True, 'region_ids': region_ids, 'supporters': supporters, 'evidence_level': 'text_anchors', 'source_verified': bool(source_verified), **evidence})
    for region in result['regions']:
        expected = region['majority_copies']
        if expected is None:
            continue
        for name, match in region['models'].items():
            copies = match.get('copies')
            if copies is None or copies == expected:
                continue
            gap = _page_flanking_gap(texts[region['reference_model']], *region['reference_span'], indexes[region['reference_model']], indexes[name], len(texts[name] or ''), settings['anchor_chars'])
            if expected == 0:
                peer_gaps = {peer: _page_flanking_gap(texts[region['reference_model']], *region['reference_span'], indexes[region['reference_model']], indexes[peer], len(texts[peer] or ''), settings['anchor_chars']) for peer in region['supporters']}
                flanking_support = [peer for peer, evidence in peer_gaps.items() if evidence['gap_chars'] is not None and evidence['gap_chars'] <= max(2, len(region['text']) * 0.2)]
            else:
                peer_gaps, flanking_support = ({}, [])
            if expected == 0 and copies > 0:
                kind = 'suspected_extra_region' if len(flanking_support) >= quorum else 'unmatched_region_text'
            elif expected > 0 and copies == 0:
                counterpart = _page_spatial_counterpart(region['location'], units[name]) if source_verified else None
                deletion = gap['gap_chars'] is not None and gap['gap_chars'] <= max(2, len(region['text']) * 0.2)
                kind = 'region_text_mismatch' if counterpart else 'suspected_missing_region' if deletion or not texts[name] else 'unmatched_region_text'
                match['spatial_counterpart'] = counterpart
            elif copies > expected:
                kind = 'suspected_duplicate_region'
            else:
                kind = 'suspected_missing_region'
            finding(name, kind, [region['id']], region['supporters'], expected_copies=expected, observed_copies=copies, stream='comparison', flanking_gap=gap, peer_flanking_gaps=peer_gaps)
    order_events = defaultdict(list)
    supported = [r for r in result['regions'] if r['majority_copies'] == 1]
    raw_indexes = {name: _page_anchor_index(comparison_text(rec['raw'].get('text', '')), settings['anchor_chars'], settings['max_anchor_occurrences']) for name, rec in records.items() if rec is not None and rec.get('reading_order', {}).get('changed')}
    raw_matches = {name: {r['id']: _page_span_matches(r['anchors'], *r['reference_span'], index, settings) for r in supported} for name, index in raw_indexes.items()}
    for first, second in combinations(supported, 2):
        signs = {name: _page_order_sign(first['models'][name], second['models'][name]) for name in voters}
        frequencies = Counter((sign for sign in signs.values() if sign is not None))
        winner = next((sign for sign, n in frequencies.items() if n >= quorum), None)
        if winner is None:
            continue
        supporters = [name for name, sign in signs.items() if sign == winner]
        for name in records:
            sign = _page_order_sign(first['models'][name], second['models'][name])
            if sign is not None and sign != winner:
                order_events[name, 'comparison'].append({'regions': [first['id'], second['id']], 'supporters': supporters})
            if name in raw_matches:
                raw_sign = _page_order_sign(raw_matches[name][first['id']], raw_matches[name][second['id']])
                if raw_sign is not None and raw_sign != winner:
                    independent = [peer for peer in supporters if peer != name]
                    if len(independent) >= quorum:
                        order_events[name, 'raw_before_normalization'].append({'regions': [first['id'], second['id']], 'supporters': independent})
    for key, events in _page_global_order_events(texts, indexes, voters, quorum, raw_indexes, settings).items():
        order_events[key].extend(events)
    for (name, stream), events in sorted(order_events.items()):
        finding(name, 'suspected_reading_order', sorted({r for e in events for r in e['regions']}), sorted({m for e in events for m in e['supporters']}), stream=stream, inverted_pairs=events[:32], inverted_pair_count=len(events), corrected_before_character_consensus=stream == 'raw_before_normalization')
    for item in result['findings']:
        summary = result['models'][item['model']]
        summary['flags'] = sorted(set([*summary['flags'], item['kind']]))
    for name, summary in result['models'].items():
        if texts[name]:
            summary['status'] = 'review' if summary['flags'] else 'assessed' if any((r['majority_copies'] is not None and r['models'][name].get('copies') is not None for r in result['regions'])) else 'unassessed'
        summary['anchored_char_fraction'] = round(sum((bool(v) for v in claimed[name])) / len(texts[name]), 4) if texts[name] else None
    result['unresolved_regions'] = [r['id'] for r in result['regions'] if r['majority_copies'] is None or any((m['status'] == 'ambiguous' for m in r['models'].values()))]
    different = len({text for text in texts.values() if text is not None}) > 1
    if result['complete']:
        result['status'] = 'structural_review' if result['findings'] else 'unresolved_structure' if different and (result['unresolved_regions'] or any(((model.get('anchored_char_fraction') or 0) < 0.8 for model in result['models'].values()))) else 'no_structural_disagreement_detected' if any((r['majority_copies'] == 1 for r in result['regions'])) else 'insufficient_evidence'
    return result

def page_consensus_exclusions(diagnostic):
    """Only uncorrected, majority-backed structural findings abstain from votes."""
    if not diagnostic['complete']:
        return {}
    excluded = defaultdict(set)
    for finding in diagnostic['findings']:
        if finding.get('corrected_before_character_consensus'):
            continue
        if finding['kind'] in {'unmatched_region_text', 'region_text_mismatch'}:
            continue
        excluded[finding['model']].add('page_' + finding['kind'])
    return {name: sorted(issues) for name, issues in excluded.items()}

def page_consensus_summary(diagnostic):
    return '; '.join((f'{name}: ' + ', '.join((PAGE_FINDING_LABELS[kind] for kind in model['flags'])) for name, model in diagnostic.get('models', {}).items() if model['flags']))

def page_structure_overview(image, diagnostic):
    """Draw only measured reference contexts; never create boxes from text spans."""
    from PIL import ImageDraw
    numbers = {number for finding in diagnostic['findings'] for number in finding['region_ids']}
    regions = [r for r in diagnostic['regions'] if r['id'] in numbers and r['location']]
    if not regions:
        return None
    overview = image.copy()
    draw = ImageDraw.Draw(overview)
    for region in regions:
        x1, y1, x2, y2 = region['location']['bbox']
        box = [round(x1 * image.width), round(y1 * image.height), round(x2 * image.width), round(y2 * image.height)]
        draw.rectangle(box, outline='red', width=max(1, image.width // 700))
        draw.text((box[0] + 2, box[1] + 2), str(region['id']), fill='red')
    return overview


from datetime import datetime, timezone
from pathlib import Path

def compare_multi_corpus(manifest_path, *, max_alignment_cells=8000000, review_threshold=0.05, reading_order='auto', page_consensus=True, page_consensus_options=None) -> dict:
    if not 0 <= review_threshold <= 1:
        raise ValueError('review_threshold must be between 0 and 1')
    if reading_order not in {'auto', 'vertical_rtl', 'detection'}:
        raise ValueError('reading_order must be auto, vertical_rtl or detection')
    campaign = read_json(Path(manifest_path))
    models = campaign['models']
    if len(models) < 2 or len(set(models)) != len(models):
        raise ValueError('Campaign must name at least two distinct models')
    rows, warnings, configs, seen, digest_cache = ([], [], {}, set(), {})
    for pair in campaign['pairs']:
        runs = {name: load_run(Path(pair[name]), name, warnings, reading_order=reading_order) for name in models}
        first = runs[models[0]]
        source_checks = [check_sources(first, runs[name], warnings, digest_cache=digest_cache) for name in models[1:]]
        verified = {check['book']: all((any((c['book'] == check['book'] and c['source_verified'] for c in group)) for group in source_checks)) for group in source_checks for check in group}
        keys = set().union(*(run['selected'] for run in runs.values()))
        for book, page in sorted(keys):
            if (book, page) in seen:
                raise ValueError(f'Duplicate book/page: {book}, {page + 1}')
            seen.add((book, page))
            records = {name: run['records'].get((book, page)) if (book, page) in run['selected'] else None for name, run in runs.items()}
            candidates = {name: r['text'] if r is not None else None for name, r in records.items()}
            quality = page_quality_issues(records)
            page_diagnostic = page_level_consensus(records, excluded_models=quality, source_verified=bool(verified.get(book)), options=page_consensus_options) if page_consensus else None
            if page_diagnostic:
                for name, issues in page_consensus_exclusions(page_diagnostic).items():
                    quality[name] = sorted(set(quality.get(name, []) + issues))
            locator = records.get('ppocr')
            reference_lines = [line['text'] for line in locator['geometry'].get('text_lines', [])] if locator and locator['reading_order']['mode'] == 'vertical_rtl' and ('ppocr' not in quality) else None
            consensus = multi_character_consensus(candidates, max_cells=max_alignment_cells, excluded_models=quality, reference_lines=reference_lines)
            attach_character_regions(consensus, records)
            attach_conflict_regions(consensus, records)
            source = first['manifest'].get('books', {}).get(book, {}).get('src')
            reasons = []
            if page_diagnostic and page_diagnostic['findings']:
                reasons.append('page_structure_disagreement')
            if page_diagnostic and (not page_diagnostic['complete']):
                reasons.append('page_structure_analysis_limit')
            if page_diagnostic and page_diagnostic['status'] == 'unresolved_structure':
                reasons.append('page_structure_unresolved')
            if consensus['missing_models']:
                reasons.append('missing_models')
            if quality:
                reasons.append('quality_excluded_models')
                reasons.extend(sorted({issue for issues in quality.values() for issue in issues}))
            if not verified.get(book):
                reasons.append('source_unverified')
            if any(((r or {}).get('hit_max') for r in records.values())):
                if 'token_limit' not in reasons:
                    reasons.append('token_limit')
            if consensus['status'] in {'alignment_limit', 'all_empty', 'both_empty', 'missing_output', 'quality_review'}:
                reasons.append(consensus['status'])
            if consensus['conflicts']:
                reasons.append('character_conflicts')
            if any((c['localization'] is None for c in consensus['conflicts'])):
                reasons.append('unresolved_character_location')
            pair_scores = [d for d in consensus['pairwise_distances'].values() if d is not None]
            max_distance = max(pair_scores) if pair_scores else None
            if max_distance is not None and max_distance > review_threshold + 1e-12:
                reasons.append('above_review_threshold')
            severe = any((r != 'character_conflicts' for r in reasons))
            rows.append({'book': book, 'page': page + 1, 'source': source, 'source_verified': bool(verified.get(book)), 'consensus': consensus, 'page_consensus': page_diagnostic, 'route': 'review_page' if severe else 'review_characters' if reasons else 'agreement_only', 'reasons': reasons, 'quality_issues': quality, 'reading_order': {name: r['reading_order'] for name, r in records.items() if r is not None}, 'counts': {name: {'han': r['han'], 'visible': r['visible']} if r is not None else None for name, r in records.items()}, 'hit_max': {name: r['hit_max'] if r is not None else None for name, r in records.items()}, 'max_pairwise_distance': max_distance})
        configs[pair['book']] = {name: run['manifest'] for name, run in runs.items()}
    return {'models': models, 'rows': rows, 'warnings': sorted(set(warnings)), 'captured_at': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'corpus_manifest': str(Path(manifest_path).resolve()), 'manifests': configs, 'settings': {'mode': 'visible', 'locator': 'ppocr', 'review_threshold': review_threshold, 'max_alignment_cells': max_alignment_cells, 'reading_order': reading_order, 'page_consensus': bool(page_consensus), 'page_consensus_options': page_consensus_options or {}, 'quality_policy': 'flag_and_abstain', 'vote_rule': 'strict_majority_all_models'}}

def multi_tables(result):
    import pandas as pd
    pages, conflicts = ([], [])
    for row in result['rows']:
        c = row['consensus']
        pages.append({'Sách': row['book'], 'Trang': row['page'], 'Xử lý': row['route'], **{name: row['counts'][name]['han'] if row['counts'][name] else None for name in result['models']}, 'Bất đồng': len(c['conflicts']), 'Đã định vị': sum((x['localization'] is not None for x in c['conflicts'])), 'Entropy (bit)': c['mean_entropy_bits'], 'Lý do': ', '.join(row['reasons'])})
        diagnostic = row.get('page_consensus') or {}
        pages[-1]['Cấu trúc trang'] = diagnostic.get('status', 'not_assessed')
        pages[-1]['Dấu hiệu cấp trang'] = page_consensus_summary(diagnostic)
        pages[-1]['Không bỏ phiếu'] = '; '.join((f"{name}: {', '.join(issues)}" for name, issues in c.get('excluded_models', {}).items()))
        for conflict in c['conflicts']:
            conflicts.append({'Sách': row['book'], 'Trang': row['page'], 'Cột': conflict['index'], **{name: 'không bỏ phiếu (output cần kiểm tra)' if name in c.get('excluded_models', {}) else 'chưa có output' if value is None else value or '∅' for name, value in conflict['predictions'].items()}, 'Đa số': conflict['majority'] or '∅' if conflict['majority'] is not None else 'hòa/chưa đủ phiếu', 'Phiếu': f"{conflict['support']}/{conflict['configured_models']}", 'Định vị': conflict['localization_status']})
    return (pd.DataFrame(pages), pd.DataFrame(conflicts))

def show_character_conflicts(result, book, page, *, max_characters=20, dpi=250, renderer=None, view='regions'):
    """Show disputed regions or characters, rendering the selected page only once."""
    from IPython.display import display
    if max_characters < 1:
        raise ValueError('max_characters must be positive')
    if view not in {'regions', 'characters'}:
        raise ValueError('view must be regions or characters')
    row = next((r for r in result['rows'] if r['book'] == book and r['page'] == page), None)
    if row is None:
        raise ValueError('Book/page not in selected campaign')
    diagnostic = row.get('page_consensus')
    if diagnostic:
        print('Đồng thuận cấp trang:', diagnostic['status'])
        for finding in diagnostic['findings']:
            print(f"{finding['model']}: {finding['label']}; vùng {finding['region_ids']}; đối chứng {finding['supporters']}; stream={finding['stream']}.")
        if diagnostic['findings']:
            print('Đây là dấu hiệu từ các mốc văn bản; vùng ảnh chỉ được xác nhận khi có tọa độ model.')
    _, table = multi_tables({**result, 'rows': [row]})
    display(table.head(max_characters))
    if not row['consensus']['conflicts']:
        print('Chưa ghi nhận cột bất đồng; kiểm tra trạng thái model thiếu, không bỏ phiếu hoặc chưa căn chỉnh được.')
        return []
    entries = row['consensus'].get('conflict_regions', row['consensus']['conflicts']) if view == 'regions' else row['consensus']['conflicts']
    image = None
    shown = []
    for conflict in entries[:max_characters]:
        excluded = row['consensus'].get('excluded_models', {})
        label = ' | '.join((f"{name}: {('không bỏ phiếu (output cần kiểm tra)' if name in excluded else 'chưa có output' if value is None else value or '∅')}" for name, value in conflict['predictions'].items()))
        title = f"Vùng {conflict['number']} (cột {conflict['start'] + 1}–{conflict['stop']})" if 'number' in conflict else f"Cột {conflict['index'] + 1}"
        print(f'{title} — {label}')
        location = conflict['localization']
        if location is None:
            print('Chưa xác định được vùng ảnh từ tọa độ và căn chỉnh hiện có.')
            continue
        if not row['source_verified'] or not row['source'] or (not Path(row['source']).is_file()):
            print('Chưa xác minh/truy cập được PDF gốc; không hiển thị ảnh có thể nhầm nguồn.')
            continue
        if image is None:
            image = (renderer or render)(row['source'], page - 1, dpi, 100000)
        glyph, context, rectangle = character_crops(image, location)
        if location.get('granularity') == 'line_context':
            print('Ảnh ngữ cảnh dòng theo khung PP-OCR; chưa xác định khung chữ bất đồng.')
        else:
            print('Ảnh cắt ký tự/cụm ký tự theo khung PP-OCR ước lượng.')
        print(f'Pixel trang gốc: {rectangle}. Dự đoán model khác được ghép bằng căn chỉnh chuỗi.')
        display(glyph)
        print('Ngữ cảnh xung quanh; khung đỏ đánh dấu vùng vừa cắt:')
        display(context)
        shown.append({'title': title, 'rectangle': rectangle, 'granularity': location.get('granularity', 'character')})
    print(f'Đã hiển thị {len(shown)}/{min(len(entries), max_characters)} vùng ảnh; giới hạn hiển thị {max_characters}, tổng {len(entries)} mục bất đồng.')
    return shown


import hashlib
from copy import deepcopy
from html import escape as report_escape
import json
from pathlib import Path
import re
import time
import zipfile
BOOK_OUTPUT_FORMAT = 'vtm-ocr-book-v1'
BOOK_CSS = "\n:root{color-scheme:light;--ink:#172b36;--muted:#526471;--line:#d5dfe3;--accent:#075e69}\n*{box-sizing:border-box}body{margin:0;background:#f3f6f5;color:var(--ink);font:16px/1.65 system-ui,sans-serif}\nmain{max-width:1150px;margin:auto;padding:24px}header,section,.card{background:white;border:1px solid var(--line);border-radius:12px;padding:20px;margin-bottom:18px}\nh1,h2,h3{line-height:1.3}a{color:var(--accent)}nav{display:flex;gap:20px;flex-wrap:wrap;margin-bottom:16px}\n.muted{color:var(--muted)}.notice{border-left:4px solid #b66c00;padding:12px;background:#fff6df}\ntable{width:100%;border-collapse:collapse}th,td{padding:9px;text-align:left;border-bottom:1px solid var(--line);vertical-align:top}\n.scroll{overflow:auto}pre{white-space:pre-wrap;overflow-wrap:anywhere;font:18px/1.9 'Noto Serif CJK SC','Microsoft YaHei',serif}\n.tabs{display:flex;gap:8px;flex-wrap:wrap}button,input{font:inherit;padding:9px 14px;border:1px solid var(--line);border-radius:6px;background:white}\nbutton{cursor:pointer}button[aria-selected=true]{background:var(--accent);color:white}.panel[hidden]{display:none}\n.predictions{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:10px}.prediction{background:#f2f6f7;padding:10px;overflow-wrap:anywhere}\n.images{display:flex;gap:20px;align-items:flex-start;flex-wrap:wrap}.images figure{margin:0;max-width:100%}img{max-width:100%;height:auto;border:1px solid var(--line)}figcaption{color:var(--muted);font-size:14px}\n.badge{display:inline-block;padding:2px 8px;border-radius:4px;background:#e4f4ed}.bad{background:#fff0d9}.status{font-weight:600}\n@media(max-width:600px){main{padding:10px}header,section,.card{padding:14px}th,td{padding:6px}}\n"
BOOK_JS = "\ndocument.querySelectorAll('[data-model]').forEach(button=>button.addEventListener('click',()=>{\n document.querySelectorAll('[data-model]').forEach(b=>b.setAttribute('aria-selected',String(b===button)));\n document.querySelectorAll('.panel').forEach(p=>p.hidden=p.id!==button.dataset.model);\n}));\nconst filter=document.querySelector('#page-filter');\nif(filter) filter.addEventListener('input',()=>document.querySelectorAll('tbody tr[data-page]').forEach(row=>{\n row.hidden=!row.textContent.toLowerCase().includes(filter.value.toLowerCase());\n}));\n"

def book_output_slug(book):
    slug = re.sub('[^\\w.-]+', '_', str(book), flags=re.UNICODE).strip('._')[:100] or 'book'
    if slug.upper().split('.')[0] in {'CON', 'PRN', 'AUX', 'NUL', *[f'COM{i}' for i in range(1, 10)], *[f'LPT{i}' for i in range(1, 10)]}:
        slug = '_' + slug
    return slug

def book_file(root, relative):
    root = Path(root).resolve()
    path = root / relative
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Output path escapes book folder: {relative}')
    return path

def book_write_text(root, relative, text, generated):
    path = book_file(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(text, encoding='utf-8', newline='\n')
    temporary.replace(path)
    generated.add(relative)

def book_html(title, body, *, page=False):
    return f'<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>{report_escape(title)}</title><link rel="stylesheet" href="' + ('../' if page else '') + f'style.css"><body><main>{body}</main><script src="' + ('../' if page else '') + 'reader.js"></script></body></html>'

def book_prediction(value):
    return 'chưa có output' if value is None else value or '∅'

def book_page_consensus(diagnostic):
    """Portable text/coordinate evidence; missing geometry stays unresolved."""
    if not diagnostic:
        return ''
    html = '<section><h2>Đồng thuận cấp trang</h2><p>' + report_escape(diagnostic['status']) + '</p>'
    html += '<p class="muted">Dấu hiệu đối chiếu giữa model; chưa chứng minh cột thật bị thiếu, lặp hoặc văn bản là bịa. Đa số có thể cùng sai.</p>'
    if diagnostic.get('overview'):
        html += '<figure><img loading="lazy" src="../' + report_escape(diagnostic['overview'], quote=True) + '" alt="Ngữ cảnh các vùng có dấu hiệu bất đồng cấp trang"><figcaption>Khung đỏ là vùng ngữ cảnh do model ước lượng; số khung trùng số vùng bên dưới. Vùng chỉ có mốc chữ không được vẽ khung.</figcaption></figure>'
    findings = diagnostic['findings']
    if findings:
        html += '<div class="scroll"><table><thead><tr><th>Model</th><th>Dấu hiệu</th><th>Vùng</th><th>Đối chứng</th><th>Luồng văn bản</th></tr></thead><tbody>'
        for item in findings:
            stream = 'Output gốc, đã chuẩn hóa trước so ký tự' if item.get('corrected_before_character_consensus') else 'Văn bản đưa vào đối chiếu'
            html += '<tr>' + ''.join(('<td>' + report_escape(value) + '</td>' for value in (item['model'], item['label'], ', '.join(map(str, item['region_ids'])), ', '.join(item['supporters']), stream))) + '</tr>'
        html += '</tbody></table></div>'
    else:
        html += '<p>Chưa phát hiện bất đồng cấu trúc với bằng chứng hiện có; không có nghĩa OCR đã đầy đủ hoặc đúng.</p>'
    selected = {number for item in findings for number in item['region_ids']}
    for region in diagnostic['regions']:
        if region['id'] not in selected:
            continue
        html += f"""<details class="card"><summary>Vùng {region['id']} · mốc từ {report_escape(region['reference_model'])}</summary>"""
        html += '<pre>' + report_escape(region['text']) + '</pre>'
        if region['location']:
            html += '<p class="muted">Khung ngữ cảnh do model ước lượng (tọa độ chuẩn hóa 0–1): ' + report_escape(str(region['location']['bbox'])) + '. Khung không xác nhận cùng vị trí ảnh giữa các model.</p>'
        else:
            html += '<p class="muted">Chưa định vị trên ảnh; đây là vùng theo mốc văn bản.</p>'
        html += '<ul>'
        for name, match in region['models'].items():
            spans = [m['span'] for m in match['matches']]
            html += '<li>' + report_escape(f"{name}: {match['status']}; số lần khớp={match.get('copies', 'chưa rõ')}; offsets={spans}") + '</li>'
        html += '</ul><p class="muted">Offsets đếm từ 0 trong chuỗi bỏ khoảng trắng; không phải tọa độ ảnh.</p></details>'
    return html + '</section>'

def book_save_crop(root, relative, image, generated):
    path = book_file(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.png.tmp')
    image.save(temporary, format='PNG', compress_level=1)
    temporary.replace(path)
    generated.add(relative)

def zip_ocr_book(directory, archive_path):
    """ZIP lives beside the book; PNGs are stored, text is lightly compressed."""
    directory, archive_path = (Path(directory).resolve(), Path(archive_path).resolve())
    if archive_path.is_relative_to(directory):
        raise ValueError('ZIP must be outside the book directory')
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive_path.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True, strict_timestamps=False) as archive:
        for path in sorted(directory.rglob('*')):
            if path.is_symlink() or not path.resolve().is_relative_to(directory):
                raise ValueError(f'Cannot archive external/symlink path: {path}')
            if path.is_file():
                compression = zipfile.ZIP_STORED if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'} else zipfile.ZIP_DEFLATED
                archive.write(path, arcname=f'{directory.name}/{path.relative_to(directory).as_posix()}', compress_type=compression, compresslevel=1)
    with zipfile.ZipFile(temporary) as archive:
        bad = archive.testzip()
        if bad:
            raise ValueError(f'ZIP CRC failed: {bad}')
    temporary.replace(archive_path)
    return archive_path

def export_ocr_book(result, book, directory, *, include_crops=True, create_zip=True, dpi=250, renderer=None):
    """Export selected pages; missing outputs remain explicit and never become votes."""
    started = time.monotonic()
    directory = Path(directory).resolve()
    directory.mkdir(parents=True, exist_ok=True)
    previous_path = book_file(directory, 'manifest.json')
    previous = json.loads(previous_path.read_text(encoding='utf-8')) if previous_path.is_file() else {}
    if previous and previous.get('format') != BOOK_OUTPUT_FORMAT:
        raise ValueError(f'Folder contains another output format: {directory}')
    rows = sorted((row for row in result['rows'] if row['book'] == book), key=lambda row: row['page'])
    if not rows:
        raise ValueError(f'Book has no selected pages: {book}')
    campaign_path = Path(result['corpus_manifest'])
    campaign = json.loads(campaign_path.read_text(encoding='utf-8'))
    pair = next((pair for pair in campaign['pairs'] if pair['book'] == book))
    models = result['models']
    warnings, runs, errors, generated = ([], {}, [], set())
    stage_path = campaign_path.parent / 'stage_errors.json'
    if stage_path.is_file():
        for model, error in json.loads(stage_path.read_text(encoding='utf-8')).items():
            errors.append({'model': model, 'book': book, 'page': None, 'stage_error': True, **error})
            warnings.append(f"{model}: {error['error']}: {error['message']}")
    for model in models:
        runs[model] = load_run(Path(pair[model]), model, warnings, reading_order=result.get('settings', {}).get('reading_order', 'auto'))
        cfg = runs[model]['manifest']['config']
        error_path = Path(pair[model]) / 'ocr_raw' / book / f"{cfg.get('run_name') or model}_errors.jsonl"
        if error_path.is_file():
            for error in read_jsonl(error_path, allow_incomplete_tail=True):
                errors.append({'model': model, 'book': book, **error, 'page': error['page'] + 1})
        lines = []
        for row in rows:
            record = runs[model]['records'].get((book, row['page'] - 1))
            quality = row.get('quality_issues', {}).get(model, [])
            geometry = record.get('geometry', record['raw']) if record else {}
            lines.append(json.dumps({'page': row['page'], 'status': 'quality_review' if quality else 'ok' if record else 'missing', 'quality_issues': quality, 'reading_order': record.get('reading_order') if record else None, 'text': record['text'] if record else None, 'raw': record['raw'] if record else None, 'comparison_geometry': {key: geometry[key] for key in ('character_boxes', 'text_lines', 'coordinate_frame', 'w', 'h') if key in geometry}}, ensure_ascii=False))
        book_write_text(directory, f'raw/{model}.jsonl', '\n'.join(lines) + '\n', generated)
        log = campaign_path.parent / f'{model}_notebook.log'
        if log.is_file():
            book_write_text(directory, f'logs/{model}.log', log.read_text(encoding='utf-8', errors='replace'), generated)
    book_write_text(directory, 'errors.jsonl', ''.join((json.dumps(error, ensure_ascii=False) + '\n' for error in errors)), generated)
    book_write_text(directory, 'style.css', BOOK_CSS, generated)
    book_write_text(directory, 'reader.js', BOOK_JS, generated)
    comparisons, index_rows = ([], [])
    render_count = crop_count = structural_overviews = 0
    for position, row in enumerate(rows):
        page, consensus = (row['page'], row['consensus'])
        records = {model: runs[model]['records'].get((book, page - 1)) for model in models}
        nav = ['<a href="../index.html">Mục lục</a>']
        if position:
            nav.append(f"""<a href="p{rows[position - 1]['page']:05d}.html">← Trang trước</a>""")
        if position + 1 < len(rows):
            nav.append(f"""<a href="p{rows[position + 1]['page']:05d}.html">Trang sau →</a>""")
        body = f"<nav>{''.join(nav)}</nav><header><h1>Trang {page} · {report_escape(book)}</h1>"
        body += '<p class="muted">Đối chiếu các dự đoán OCR; đa số không phải đáp án đã kiểm chứng.</p></header>'
        if row['reasons']:
            body += '<p class="notice">Trạng thái: ' + report_escape(', '.join(row['reasons'])) + '</p>'
        tabs, panels = ([], [])
        for index, model in enumerate(models):
            safe_model = report_escape(model, quote=True)
            record = records[model]
            text = record['text'] if record else 'Chưa có output cho trang này. Xem errors.jsonl hoặc log model.'
            tabs.append(f'<button type="button" role="tab" data-model="model-{safe_model}" aria-selected="{str(index == 0).lower()}">{safe_model}</button>')
            quality = consensus.get('excluded_models', {}).get(model, [])
            notice = '<p class="notice">Không bỏ phiếu; cần kiểm tra output: ' + report_escape(', '.join(quality)) + '</p>' if quality else ''
            panels.append(f'''<article class="panel" id="model-{safe_model}"{(' hidden' if index else '')}><h3>{safe_model}</h3>{notice}<pre>{report_escape(text)}</pre></article>''')
        body += '<section><h2>Văn bản OCR</h2><div class="tabs" role="tablist">' + ''.join(tabs) + '</div>' + ''.join(panels) + '</section>'
        diagnostic = deepcopy(row.get('page_consensus'))
        image = None
        if diagnostic and include_crops and row['source_verified'] and row['source'] and Path(row['source']).is_file():
            numbers = {n for item in diagnostic['findings'] for n in item['region_ids']}
            if any((r['id'] in numbers and r['location'] for r in diagnostic['regions'])):
                image = (renderer or render)(row['source'], page - 1, dpi, 100000)
                render_count += 1
                overview = page_structure_overview(image, diagnostic)
                filename = f'crops/p{page:05d}/page_structure.png'
                book_save_crop(directory, filename, overview, generated)
                overview.close()
                diagnostic['overview'] = filename
                structural_overviews += 1
        body += book_page_consensus(diagnostic)
        body += '<section><h2>Vùng bất đồng</h2>'
        regions = []
        for region in consensus.get('conflict_regions', []):
            region = dict(region)
            number = region['number']
            body += f'<article class="card"><h3>Vùng {number}</h3><div class="predictions">'
            for model, value in region['predictions'].items():
                prediction = 'không bỏ phiếu (output cần kiểm tra)' if model in consensus.get('excluded_models', {}) else book_prediction(value)
                body += f'<div class="prediction"><strong>{report_escape(model)}</strong><br>{report_escape(prediction)}</div>'
            body += '</div>'
            location = region['localization']
            if location and include_crops and row['source_verified'] and row['source'] and Path(row['source']).is_file():
                if image is None:
                    image = (renderer or render)(row['source'], page - 1, dpi, 100000)
                    render_count += 1
                crop, context, rectangle = character_crops(image, location)
                crop_name = f'crops/p{page:05d}/region_{number:04d}.png'
                context_name = f'crops/p{page:05d}/region_{number:04d}_context.png'
                book_save_crop(directory, crop_name, crop, generated)
                book_save_crop(directory, context_name, context, generated)
                region.update(crop=crop_name, context_crop=context_name, source_pixel_rectangle=list(rectangle))
                caption = 'Ngữ cảnh dòng; chưa định vị được chữ bất đồng.' if location.get('granularity') == 'line_context' else 'Khung chữ/cụm chữ do PP-OCR ước lượng; các model khác được ghép bằng căn chỉnh chuỗi.'
                body += '<p class="muted">' + caption + '</p><div class="images">'
                for filename, picture, label in ((crop_name, crop, 'Ảnh cắt'), (context_name, context, 'Ngữ cảnh · khung đỏ')):
                    body += f'<figure><img loading="lazy" src="../{filename}" width="{picture.width}" height="{picture.height}" alt="{label}"><figcaption>{label}</figcaption></figure>'
                body += '</div>'
                crop_count += 1
                crop.close()
                context.close()
            else:
                label = 'Xuất ảnh đã tắt.' if not include_crops else 'Chưa xác minh/truy cập được PDF nguồn.' if not row['source_verified'] or not row['source'] or (not Path(row['source']).is_file()) else 'Chưa đủ tọa độ hoặc căn chỉnh để cắt ảnh vùng này.'
                body += f'<p class="muted">{label}</p>'
            body += '</article>'
            regions.append(region)
        if not consensus['conflicts']:
            body += '<p>Không ghi nhận cột bất đồng trong các output tham gia so sánh. Kiểm tra trạng thái model thiếu, không bỏ phiếu hoặc chưa căn chỉnh được phía trên.</p>'
        if image is not None:
            image.close()
        body += '</section>'
        book_write_text(directory, f'pages/p{page:05d}.html', book_html(f'Trang {page} · {book}', body, page=True), generated)
        comparisons.append(json.dumps({'page': page, 'status': consensus['status'], 'source_verified': row['source_verified'], 'reasons': row['reasons'], 'missing_models': consensus['missing_models'], 'reference': consensus['reference'], 'excluded_models': consensus.get('excluded_models', {}), 'reading_order': row.get('reading_order', {}), 'page_consensus': diagnostic, 'alignment': consensus.get('alignment'), 'line_alignment': consensus.get('line_alignment', {}), 'pairwise_distances': consensus['pairwise_distances'], 'conflicts': consensus['conflicts'], 'regions': regions}, ensure_ascii=False))
        cells = ''.join((f"""<td><span class="badge{(' bad' if not records[model] or model in consensus.get('excluded_models', {}) else '')}">{('Không bỏ phiếu' if model in consensus.get('excluded_models', {}) else 'OK' if records[model] else 'Thiếu')}</span></td>""" for model in models))
        structural = page_consensus_summary(row.get('page_consensus') or {})
        index_rows.append(f'''<tr data-page="{page}"><td><a href="pages/p{page:05d}.html">Trang {page}</a></td>{cells}<td>{len(consensus['conflicts'])}</td><td>{report_escape(structural)}</td></tr>''')
    book_write_text(directory, 'comparison.jsonl', '\n'.join(comparisons) + '\n', generated)
    counts = {model: sum(((book, row['page'] - 1) in runs[model]['records'] for row in rows)) for model in models}
    complete = all((count == len(rows) for count in counts.values()))
    quality_pages = sum((bool(row['consensus'].get('excluded_models')) for row in rows))
    body = f"""<header><h1>{report_escape(book)}</h1><p class="status">{('Đủ output các trang đã chọn' if complete else 'Kết quả còn thiếu output')} · {len(rows)} trang</p>"""
    body += '<p>Chọn trang để đọc bốn dự đoán và xem ảnh bất đồng. Số trang theo PDF, bắt đầu từ 1.</p>'
    body += '<p class="muted">Số cột bất đồng là kết quả căn chỉnh chuỗi, không phải số chữ OCR sai đã được xác nhận.</p>'
    if quality_pages:
        body += f'<p class="notice">{quality_pages} trang có output cần kiểm tra và không tham gia bỏ phiếu; xem từng trang để biết model và lý do.</p>'
    body += '<p class="muted">Giữ nguyên thư mục sau khi giải nén. Mở được offline, không cần server.</p></header>'
    all_warnings = sorted(set(result.get('warnings', []) + warnings))
    if all_warnings:
        body += '<section class="notice"><h2>Lưu ý dữ liệu</h2><ul>' + ''.join(('<li>' + report_escape(value) + '</li>' for value in all_warnings)) + '</ul></section>'
    body += '<section><h2>Các trang</h2><label>Lọc trang / trạng thái <input id="page-filter" type="search" placeholder="Ví dụ: Trang 12 hoặc Thiếu"></label>'
    body += '<div class="scroll"><table><thead><tr><th>Trang</th>' + ''.join((f'<th>{report_escape(model)}</th>' for model in models)) + '<th>Bất đồng</th><th>Dấu hiệu cấp trang</th></tr></thead><tbody>'
    body += ''.join(index_rows) + '</tbody></table></div></section>'
    body += '<section><h2>Dữ liệu và nhật ký</h2><ul>' + ''.join((f'<li><a href="raw/{model}.jsonl">{model}.jsonl</a> · {counts[model]}/{len(rows)} trang</li>' for model in models))
    body += '<li><a href="comparison.jsonl">Bất đồng và tọa độ</a></li><li><a href="errors.jsonl">Lịch sử lỗi</a></li><li><a href="manifest.json">Thông tin nguồn và cấu hình</a></li></ul></section>'
    book_write_text(directory, 'index.html', book_html(book, body), generated)
    generated.add('manifest.json')
    manifest = {'format': BOOK_OUTPUT_FORMAT, 'book': book, 'created_at': now(), 'source': rows[0]['source'], 'source_verified': all((row['source_verified'] for row in rows)), 'page_numbers': [row['page'] for row in rows], 'status': 'complete' if complete else 'partial', 'page_number_base': 1, 'raw_page_index_base': 0, 'alignment_column_index_base': 0, 'models': {model: {'done': counts[model], 'selected': len(rows), 'config': runs[model]['manifest']['config'], 'source': runs[model]['manifest'].get('books', {}).get(book)} for model in models}, 'warnings': all_warnings, 'crop_regions': crop_count, 'pages_rendered_for_crops': render_count, 'page_structure_overviews': structural_overviews, 'quality_review_pages': quality_pages, 'comparison_settings': result.get('settings', {}), 'crop_dpi': dpi, 'generated_files': sorted(generated), 'export_seconds': round(time.monotonic() - started, 3)}
    for relative in set(previous.get('generated_files', [])) - generated:
        stale = book_file(directory, relative)
        if stale.is_file():
            stale.unlink()
    write_json(previous_path, manifest)
    archive = zip_ocr_book(directory, directory.parent / f'{directory.name}_results.zip') if create_zip else None
    return {'book': book, 'directory': str(directory), 'index_html': str(directory / 'index.html'), 'zip_path': str(archive) if archive else None, 'pages': len(rows), 'status': manifest['status'], 'crop_regions': crop_count, 'pages_rendered_for_crops': render_count, 'export_seconds': round(time.monotonic() - started, 3)}

def export_corpus_results(result, output_dir, *, include_crops=True, create_zip=True, dpi=250, renderer=None):
    books = list(dict.fromkeys((row['book'] for row in result['rows'])))
    names = [book_output_slug(book) for book in books]
    if len({name.casefold() for name in names}) != len(names):
        names = [f'{name}_{hashlib.sha256(book.encode()).hexdigest()[:8]}' for name, book in zip(names, books)]
    return [export_ocr_book(result, book, Path(output_dir) / name, include_crops=include_crops, create_zip=create_zip, dpi=dpi, renderer=renderer) for book, name in zip(books, names)]


import hashlib
import json
from pathlib import Path

def import_ocr_logs(log_dir, work_dir, *, input_dir=None, books=None, max_books=1, pages='', limit=None):
    """Build a separate checkpoint copy; never rewrite the supplied logs/PDFs."""
    source_root = Path(log_dir).resolve()
    if not source_root.is_dir():
        raise FileNotFoundError(f'Không có thư mục log: {source_root}')
    paths = [source_root / 'manifest.json'] if (source_root / 'raw').is_dir() else sorted(source_root.glob('*/manifest.json'))
    selected = []
    wanted = {Path(str(book)).stem for book in books or []}
    for path in paths:
        manifest = read_json(path)
        if manifest.get('format') != 'vtm-ocr-book-v1':
            continue
        book = str(manifest['book'])
        if book in {'.', '..'} or '/' in book or '\\' in book:
            raise ValueError('Invalid book name in exported manifest')
        if not wanted or book in wanted or path.parent.name in wanted:
            selected.append((path, manifest))
    if max_books is not None:
        if max_books < 1:
            raise ValueError('max_books must be positive or None')
        selected = selected[:max_books]
    if not selected:
        raise FileNotFoundError(f'Không có log sách đã chọn trong {source_root}')
    if limit is not None and limit < 1:
        raise ValueError('limit must be positive or None')
    root = Path(work_dir).resolve() / 'imported_logs'
    if root == source_root or root.is_relative_to(source_root) or source_root.is_relative_to(root):
        raise ValueError('Imported checkpoint must be separate from source logs')
    pairs, models, total, seen = ([], None, 0, set())
    for path, manifest in selected:
        book = str(manifest['book'])
        if book in seen:
            raise ValueError(f'Duplicate exported book: {book}')
        seen.add(book)
        names = list(manifest['models'])
        if models is not None and set(names) != set(models):
            raise ValueError('Exported books must contain the same models')
        models = names
        exported_pages = manifest['page_numbers']
        allowed = set(parse_pages(pages, max(exported_pages)))
        chosen = [number - 1 for number in exported_pages if number - 1 in allowed]
        if limit is not None:
            chosen = chosen[:max(0, limit - total)]
        if not chosen:
            continue
        pair = {'book': book}
        book_key = hashlib.sha256(book.encode()).hexdigest()[:16]
        for name in names:
            if name not in {'ppocr', 'tonggu', 'paddle', 'hunyuan'}:
                raise ValueError(f'Unknown OCR model: {name}')
            info = manifest['models'][name]
            original = dict(info.get('source') or {})
            candidate = Path(input_dir) / f'{book}.pdf' if input_dir else None
            if candidate and candidate.is_file():
                actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
                if original.get('sha256') and original['sha256'] != actual:
                    raise ValueError(f'Different source PDF for exported book {book}')
                if original.get('sha256'):
                    original['src'] = str(candidate.resolve())
            original['pages'] = chosen
            cfg = {**info.get('config', {}), 'engine': name, 'run_name': name}
            directory = root / book_key / name
            raw_dir = directory / 'ocr_raw' / book
            raw_dir.mkdir(parents=True, exist_ok=True)
            records, record_pages = ([], set())
            for exported in read_jsonl(path.parent / 'raw' / f'{name}.jsonl'):
                page = int(exported['page']) - 1
                if page not in chosen or exported.get('raw') is None:
                    continue
                record = exported['raw']
                if record.get('page') != page or page in record_pages:
                    raise ValueError(f'Invalid/duplicate exported page: {book}/{name}/{page + 1}')
                records.append(record)
                record_pages.add(page)
            (raw_dir / f'{name}_shard0.jsonl').write_text(''.join((json.dumps(record, ensure_ascii=False) + '\n' for record in records)), encoding='utf-8')
            (directory / 'tasks.jsonl').write_text(''.join((json.dumps({'book': book, 'page': page, 'src': original.get('src')}) + '\n' for page in chosen)), encoding='utf-8')
            write_json(directory / 'run_manifest.json', {'kind': 'ocr', 'status': 'complete', 'config': cfg, 'books': {book: original}, 'imported_from': str(path.parent)})
            pair[name] = str(directory)
        pairs.append(pair)
        total += len(chosen)
    if not pairs:
        raise ValueError('Không có trang log nào khớp PAGES/LIMIT')
    output = root / 'manifest.json'
    write_json(output, {'models': models, 'pairs': pairs, 'total_books': len(pairs), 'total_pages': total, 'imported_from': str(source_root)})
    return output


In [ ]:
import argparse
import ast
import base64
import csv
from email.parser import Parser
import gzip
import hashlib
import importlib.metadata as metadata
import importlib.util
import io
import json
import os
from pathlib import Path
import platform
import pprint
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import tempfile
import zipfile
KERNEL_REPORT_CORRECTIONS = {'1e47c86c9fe706d2e297ea8c0f574716c1d1aadb7366e8de99b195c720c0c746': {'numpy': {'reported': '1.26.4', 'active': '2.1.3'}}}
OPENCV_PACKAGES = ('opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless')
PRIVATE_SITE_INITIALIZATION = "import sys, site, types\nsys.modules['sitecustomize'] = types.ModuleType('sitecustomize')\nsite.main()\n"
PRIVATE_PYTHON_BOOTSTRAP = PRIVATE_SITE_INITIALIZATION + "\nimport os, runpy\nsys.executable = sys.argv.pop(1)\nargs = sys.argv[1:]\nwhile args and args[0] in ('-u', '-I', '-S', '-E', '-s'):\n    args.pop(0)\nif not args:\n    raise SystemExit('Private Python expects -c, -m, or a script path')\nmode = args.pop(0)\nif mode in ('--version', '-V'):\n    print('Python ' + sys.version.split()[0])\nelif mode == '-c':\n    code = args.pop(0)\n    sys.argv = ['-c', *args]\n    main = types.ModuleType('__main__')\n    main.__dict__.update(__package__=None, __spec__=None)\n    sys.modules['__main__'] = main\n    exec(compile(code, '<private-python>', 'exec'), main.__dict__)\nelif mode == '-m':\n    module = args.pop(0)\n    sys.argv = [module, *args]\n    runpy.run_module(module, run_name='__main__', alter_sys=True)\nelse:\n    sys.argv = [mode, *args]\n    sys.path.insert(0, os.path.dirname(os.path.abspath(mode)))\n    runpy.run_path(mode, run_name='__main__')\n"

def isolated_python_command(python, args):
    return [python, '-u', '-I', '-S', '-c', PRIVATE_PYTHON_BOOTSTRAP, python, *args]

def host_pip_command(args):
    return isolated_python_command(sys.executable, ['-m', 'pip', *args])

def command_description(args):
    return ' '.join(('<private Python bootstrap>' if str(arg) == PRIVATE_PYTHON_BOOTSTRAP else str(arg) for arg in args))

def sha256(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def checked_path(root, relative):
    path = (root / relative).resolve()
    if not path.is_relative_to(root.resolve()):
        raise ValueError(f'Path outside bundle: {relative}')
    return path

def source_tree_digest(files):
    data = json.dumps(sorted(files, key=lambda r: r['path']), sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()

class SourceTreeError(ValueError):

    def __init__(self, message, report):
        self.report = report
        missing = report.get('unresolved_nonempty', [])
        changed = report.get('changed', [])
        details = [f"{report['source']}: expected={report['expected_files']}, actual={report['actual_files']}"]
        if missing:
            details.append(f'Missing nonempty files ({len(missing)}): ' + ', '.join(missing[:12]))
        if changed:
            details.append(f'Integrity failure ({len(changed)}): ' + ', '.join((r['path'] for r in changed[:12])))
        super().__init__(message + '\n' + '\n'.join(details))

def resolve_source_tree(bundle, entry):
    """Resolve only content verified by the manifest; never write to Kaggle Input."""
    directory = checked_path(bundle, entry['path'])
    files = entry['files']
    expected = {r['path'] for r in files}
    if len(files) != len(expected) or source_tree_digest(files) != entry['sha256']:
        raise ValueError(f"Invalid source tree manifest: {entry['path']}")
    actual = {p.relative_to(directory).as_posix(): p for p in directory.rglob('*') if p.is_file()}
    folded = {}
    for name in actual:
        folded.setdefault(name.casefold(), []).append(name)
    report = {'source': entry['path'], 'directory': str(directory), 'expected_files': len(expected), 'actual_files': len(actual), 'expected_bytes': sum((r['bytes'] for r in files)), 'declared_bytes': entry['bytes'], 'missing_uploaded': sorted(expected - actual.keys()), 'ignored_extras': sorted(actual.keys() - expected), 'recreated_empty': [], 'resolved_names': [], 'reused_payloads': [], 'omitted_bytecode': [], 'unresolved_nonempty': [], 'changed': []}
    if report['expected_bytes'] != entry['bytes']:
        raise SourceTreeError('Invalid source tree byte count', report)
    resolved, verified, cache, pending = ({}, {}, {}, [])

    def matches(name, record):
        path = checked_path(directory, name)
        if path.stat().st_size != record['bytes']:
            return False
        if path not in cache:
            cache[path] = sha256(path)
        return cache[path] == record['sha256']
    for record in files:
        name = record['path']
        checked_path(directory, name)
        if name in actual:
            if not matches(name, record):
                report['changed'].append({'path': name, 'expected_bytes': record['bytes'], 'actual_bytes': actual[name].stat().st_size})
                continue
            chosen = name
        else:
            candidates = list(folded.get(name.casefold(), []))
            if record.get('restore_path') and (not record.get('restore_as_gzip')):
                candidates.extend(folded.get(record['restore_path'].casefold(), []))
            chosen = next((candidate for candidate in candidates if matches(candidate, record)), None)
            if chosen is None:
                if record['bytes'] == 0 and record['sha256'] == hashlib.sha256(b'').hexdigest():
                    resolved[name] = None
                    report['recreated_empty'].append(name)
                else:
                    pending.append(record)
                continue
            report['resolved_names'].append({'expected': name, 'actual': chosen})
        resolved[name] = checked_path(directory, chosen)
        verified[record['bytes'], record['sha256']] = resolved[name]
    unavailable = []
    for record in pending:
        donor = verified.get((record['bytes'], record['sha256']))
        if donor is None:
            unavailable.append(record)
        else:
            resolved[record['path']] = donor
            report['reused_payloads'].append({'expected': record['path'], 'actual': donor.relative_to(directory).as_posix()})
    logical_records = {r.get('restore_path', r['path']): r for r in files}
    for record in unavailable:
        source_record = None
        if entry.get('package') == 'python-runtime' and Path(record['path']).suffix == '.pyc' and (not record.get('restore_as_gzip')):
            try:
                source_name = Path(importlib.util.source_from_cache(record.get('restore_path', record['path']))).as_posix()
            except ValueError:
                pass
            else:
                source_record = logical_records.get(source_name)
        if source_record is not None and source_record['path'] in resolved and (not source_record.get('restore_as_gzip')):
            report['omitted_bytecode'].append({'path': record['path'], 'source': source_record['path']})
        else:
            report['unresolved_nonempty'].append(record['path'])
    if report['unresolved_nonempty'] or report['changed']:
        raise SourceTreeError('Incomplete or changed source tree', report)
    return (resolved, report)

def verify_source_tree(bundle, entry):
    _, report = resolve_source_tree(bundle, entry)
    for key, label in (('recreated_empty', 'file rỗng sẽ tạo lại trong /tmp'), ('resolved_names', 'tên file được đối chiếu bằng SHA-256'), ('reused_payloads', 'payload trùng checksum được dùng lại'), ('omitted_bytecode', 'cache .pyc thiếu; dùng file .py đã xác minh'), ('ignored_extras', 'file thừa không được chép vào runtime')):
        if report[key]:
            print(f"{entry['path']}: {len(report[key])} {label}.", flush=True)
    return report

def copy_source_tree(bundle, entry, target):
    """Copy plain uploaded files into writable storage and restore Linux modes."""
    resolved, report = resolve_source_tree(bundle, entry)
    omitted = {r['path'] for r in report['omitted_bytecode']}
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    for record in entry['files']:
        if record['path'] in omitted:
            continue
        source = resolved[record['path']]
        destination = checked_path(target, record.get('restore_path', record['path']))
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source is None:
            if record.get('restore_as_gzip'):
                with gzip.open(destination, 'wb'):
                    pass
            else:
                destination.write_bytes(b'')
        elif record.get('restore_as_gzip'):
            with source.open('rb') as stream, gzip.open(destination, 'wb') as output:
                shutil.copyfileobj(stream, output)
        else:
            shutil.copyfile(source, destination)
        destination.chmod(record['mode'])
    return target

def prepare_python_runtime(bundle, entry, directory):
    if entry.get('layout') == 'expanded_directory':
        copy_source_tree(bundle, entry, directory / 'python')
    else:
        extract_tar(checked_path(bundle, entry['path']), directory)

def prepare_build_source(bundle, entry, runtime):
    if entry.get('layout') == 'expanded_directory':
        return copy_source_tree(bundle, entry, runtime / 'build_sources' / Path(entry['path']).name)
    return checked_path(bundle, entry['path'])

def prepare_tonggu_checkout(bundle, entry, runtime):
    source_root = runtime / 'tonggu_source'
    if entry.get('layout') == 'expanded_directory':
        return copy_source_tree(bundle, entry, source_root / 'TongGuOCR')
    source_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(checked_path(bundle, entry['path'])) as archive:
        for info in archive.infolist():
            checked_path(source_root, info.filename)
        archive.extractall(source_root)
    return next(source_root.glob('TongGuOCR-*/tongguocr/pipeline.py')).parents[1]

def verify(bundle, *, report_path=None):
    records = {}
    reports = []
    groups = ['paddle', 'tonggu', 'system_assets']
    if (bundle / 'app_manifest.json').is_file():
        groups.append('app')
    for name in groups:
        path = bundle / f'{name}_manifest.json'
        if not path.is_file():
            raise FileNotFoundError(f'Incomplete bundle: {path}')
        manifest = json.loads(path.read_text(encoding='utf-8'))
        if manifest.get('failed'):
            raise ValueError(f'Bundle has failed downloads: {name}')
        if name == 'tonggu' and any((r['package'].lower().replace('_', '-') == 'nvidia-nccl-cu13' for r in manifest.get('reused', []))):
            raise ValueError('Dataset packages còn dùng lại NCCL từ kernel. Cập nhật Dataset packages bằng bộ đã bổ sung wheel NCCL riêng cho TongGu, rồi gắn phiên bản mới vào notebook. Chỉ thay ô setup chưa đủ cho lỗi checksum NCCL.')
        for entry in manifest['records']:
            if entry.get('layout') == 'expanded_directory':
                try:
                    reports.append(verify_source_tree(bundle, entry))
                except SourceTreeError as exc:
                    if report_path is not None:
                        report_path = Path(report_path)
                        report_path.parent.mkdir(parents=True, exist_ok=True)
                        report_path.write_text(json.dumps({'ok': False, 'source_trees': reports + [exc.report]}, ensure_ascii=False, indent=2), encoding='utf-8')
                    raise
                records[entry['path']] = entry
                continue
            path = checked_path(bundle, entry['path'])
            if path.stat().st_size != entry['bytes'] or sha256(path) != entry['sha256']:
                raise ValueError(f'Integrity failure: {path.name}')
            records[entry['path']] = entry
    if report_path is not None:
        report_path = Path(report_path)
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(json.dumps({'ok': True, 'source_trees': reports}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Verified {len(records)} artifacts, {sum((r['bytes'] for r in records.values())) / 1024 ** 3:.2f} GiB", flush=True)
    return list(records.values())

def extract_tar(path, target):
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(path) as archive:
        archive.extractall(target, filter='data')

def target_site(python, runtime):
    code = PRIVATE_SITE_INITIALIZATION + "\nimport json,sysconfig; print(json.dumps([sysconfig.get_paths()['purelib'],sys.version.split()[0]]))"
    site, version = json.loads(subprocess.check_output([str(python), '-u', '-I', '-S', '-c', code], env=native_environment(), text=True).strip())
    site = Path(site).resolve()
    if not site.is_relative_to(runtime.resolve()):
        raise ValueError(f'Target interpreter is outside the private runtime: {site}')
    site.mkdir(parents=True, exist_ok=True)
    return (site, version)

def clear_kernel_links(site):
    ledger = site / '.vtm_kernel_links.json'
    if not ledger.is_file():
        return
    for relative in json.loads(ledger.read_text(encoding='utf-8')):
        path = site / relative
        if not path.parent.resolve().is_relative_to(site.resolve()):
            raise ValueError(f'Kernel link escapes target site: {relative}')
        if path.is_symlink():
            path.unlink()
    ledger.unlink()

def clear_reused_packages(site, records):
    """Remove previous private copies, including a failed install; never follow kernel links."""
    from packaging.utils import canonicalize_name
    names = {canonicalize_name(r['package']) for r in records}
    owned, protected = (set(), set())
    for dist in metadata.distributions(path=[str(site)]):
        name = canonicalize_name(dist.metadata.get('Name', ''))
        paths = owned if name in names else protected
        for row in csv.reader(io.StringIO(dist.read_text('RECORD') or '')):
            if row:
                paths.add(row[0])
    ledger = site / '.vtm_kernel_files.json'
    if ledger.is_file():
        owned.update(json.loads(ledger.read_text(encoding='utf-8')))
    targets = []
    for relative in owned - protected:
        path = Path(relative)
        if path.is_absolute() or '..' in path.parts:
            continue
        target = site / path
        if not target.parent.resolve().is_relative_to(site.resolve()):
            raise ValueError(f'Reused package cleanup escapes private site: {relative}')
        if target.is_symlink() or target.is_file():
            targets.append(target)
    for target in targets:
        target.unlink()
    if ledger.is_file():
        ledger.unlink()
    clear_kernel_links(site)

def private_kernel_cleanup_records(manifest):
    """Clean kernel copies; let the dedicated cv2 repair own its shared namespace."""
    from packaging.utils import canonicalize_name
    records = [*manifest.get('reused', []), *manifest.get('kernel_overlap_repairs', [])]
    shipped = {canonicalize_name(r['package']) for r in manifest.get('records', [])}
    if shipped.intersection(OPENCV_PACKAGES):
        records = [r for r in records if canonicalize_name(r['package']) not in OPENCV_PACKAGES]
    return list({canonicalize_name(r['package']): r for r in records}.values())

def opencv_metadata_directories(site):
    """Find even damaged OpenCV metadata without relying on importer caches."""
    from packaging.utils import canonicalize_name
    return [path for path in Path(site).glob('*.dist-info') if canonicalize_name(path.name[:-len('.dist-info')].rsplit('-', 1)[0]) in OPENCV_PACKAGES]

def clear_private_opencv_files(site, runtime):
    """Remove only OpenCV-owned namespaces in a verified private site."""
    site, runtime = (Path(site).resolve(), Path(runtime).resolve())
    if site == runtime or not site.is_relative_to(runtime):
        raise ValueError(f'OpenCV cleanup site is outside private runtime: {site}')
    targets = [site / 'cv2', site / 'cv2.py', *site.glob('cv2.*.so'), *site.glob('cv2*.pyd'), *opencv_metadata_directories(site)]
    targets.extend((site / (name.replace('-', '_') + '.libs') for name in OPENCV_PACKAGES))
    for target in targets:
        if target.parent.resolve() != site:
            raise ValueError(f'OpenCV cleanup target escapes private site: {target}')
        if target.is_symlink():
            target.unlink()
        elif target.exists():
            resolved = target.resolve()
            if resolved == site or not resolved.is_relative_to(site):
                raise ValueError(f'OpenCV cleanup target escapes private site: {target}')
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()

def audit_private_opencv(wheel, site):
    """Compare the installed cv2 payload and vendor libraries against wheel RECORD."""
    site = Path(site).resolve()
    problems, checked = ([], 0)
    with zipfile.ZipFile(wheel) as archive:
        record_name = next((n for n in archive.namelist() if n.endswith('.dist-info/RECORD') and n.count('/') == 1))
        expected_info = record_name.rsplit('/', 1)[0]
        expected_files = set(archive.namelist())
        installed_record = site / record_name
        if installed_record.is_symlink() or not installed_record.is_file():
            problems.append({'path': record_name, 'reason': 'missing_record'})
        for row in csv.reader(io.StringIO(archive.read(record_name).decode('utf-8'))):
            if not row or row[0] == record_name:
                continue
            name, checksum, size = row
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts or (not checksum):
                raise ValueError(f'Unsafe or unverifiable OpenCV wheel entry: {name}')
            target = site / relative
            if not target.parent.resolve().is_relative_to(site):
                problems.append({'path': name, 'reason': 'parent_symlink'})
                continue
            if target.is_symlink() or not target.is_file():
                problems.append({'path': name, 'reason': 'linked_or_missing'})
                continue
            algorithm, expected_hash = checksum.split('=', 1)
            if size and target.stat().st_size != int(size):
                problems.append({'path': name, 'reason': 'size'})
                continue
            with target.open('rb') as stream:
                actual = base64.urlsafe_b64encode(hashlib.file_digest(stream, algorithm).digest()).rstrip(b'=').decode()
            if actual != expected_hash:
                problems.append({'path': name, 'reason': 'checksum'})
            checked += 1
        for metadata_dir in opencv_metadata_directories(site):
            if metadata_dir.name != expected_info and (metadata_dir.is_file() or any(metadata_dir.iterdir())):
                problems.append({'path': metadata_dir.name, 'reason': 'other_provider'})
        cv2_root = site / 'cv2'
        if cv2_root.is_dir() and cv2_root.resolve().is_relative_to(site):
            for path in cv2_root.rglob('*'):
                name = path.relative_to(site).as_posix()
                if path.is_file() and path.suffix in {'.py', '.so', '.pyd'} and (name not in expected_files) and ('__pycache__' not in path.parts):
                    problems.append({'path': name, 'reason': 'untracked_loader_or_binary'})
        for path in [site / 'cv2.py', *site.glob('cv2.*.so'), *site.glob('cv2*.pyd')]:
            if path.is_file() and path.name not in expected_files:
                problems.append({'path': path.name, 'reason': 'legacy_root_module'})
    return {'wheel': str(wheel), 'site': str(site), 'checked_files': checked, 'problems': problems}
PRIVATE_OPENCV_PROBE = "import importlib.metadata as md, json, sys\nfrom pathlib import Path\nsite, expected, version, output = sys.argv[1:5]\nreport = {'package': expected, 'version': version}\ntry:\n    import cv2, numpy as np\n    report.update(origin=getattr(cv2,'__file__',None), cv2_version=getattr(cv2,'__version__',None))\n    needed = ['resize','cvtColor','warpAffine','getRotationMatrix2D','findContours','threshold',\n              'minAreaRect','boxPoints','copyMakeBorder','imread','imwrite','imencode','imdecode']\n    missing = [name for name in needed if not callable(getattr(cv2,name,None))]\n    report['missing_api'] = missing\n    assert not missing, 'Incomplete cv2 API: ' + ', '.join(missing)\n    assert report['origin'] and Path(report['origin']).resolve().is_relative_to(Path(site).resolve()), report\n    variants = {'opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless'}\n    providers = {d.metadata.get('Name','').lower().replace('_','-'): d.version\n                 for d in md.distributions(path=[site]) if d.metadata.get('Name','').lower().replace('_','-') in variants}\n    report['providers'] = providers\n    assert providers == {expected:version}, providers\n    image = np.zeros((8,8,3),dtype=np.uint8); image[2:6,2:6] = 255\n    assert cv2.resize(image,(16,16)).shape == (16,16,3)\n    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(gray,0,255,cv2.THRESH_BINARY | cv2.THRESH_OTSU)\n    contours, _ = cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n    assert contours and cv2.boxPoints(cv2.minAreaRect(contours[0])).shape == (4,2)\n    matrix = cv2.getRotationMatrix2D((4.,4.),0.,1.)\n    assert cv2.warpAffine(image,matrix,(8,8)).shape == image.shape\n    assert cv2.copyMakeBorder(image,1,1,1,1,cv2.BORDER_CONSTANT).shape == (10,10,3)\n    ok, encoded = cv2.imencode('.png',image)\n    assert ok and cv2.imdecode(encoded,cv2.IMREAD_COLOR).shape == image.shape\n    report['ok'] = True\n    print(json.dumps(report))\nexcept Exception as exc:\n    report.update(ok=False,error=repr(exc),sys_path=sys.path)\n    raise\nfinally:\n    Path(output).write_text(json.dumps(report,indent=2),encoding='utf-8')\n"

def ensure_private_opencv(bundle, manifest, site, python, environment, runtime):
    """Repair partial OpenCV installs even when pip considers their metadata satisfied."""
    from packaging.utils import canonicalize_name
    records = [r for r in manifest.get('records', []) if canonicalize_name(r['package']) in OPENCV_PACKAGES]
    if not records:
        return None
    if len(records) != 1:
        raise ValueError('A private runtime must ship exactly one OpenCV provider')
    record = records[0]
    wheel = checked_path(Path(bundle), record['path'])
    if sha256(wheel) != record['sha256']:
        raise ValueError(f'OpenCV repair wheel checksum differs: {wheel}')
    site, runtime = (Path(site).resolve(), Path(runtime).resolve())
    if site == runtime or not site.is_relative_to(runtime):
        raise ValueError(f'OpenCV site is outside private runtime: {site}')
    audit = audit_private_opencv(wheel, site)
    repaired = bool(audit['problems'])
    if repaired:
        print(f"Repair incomplete OpenCV payload: {len(audit['problems'])} mismatches; first: {audit['problems'][:5]}", flush=True)
        clear_private_opencv_files(site, runtime)
        run(host_pip_command(['--python', python, 'install', '--no-index', '--no-deps', '--force-reinstall', wheel]), environment)
        audit = audit_private_opencv(wheel, site)
    audit['repaired'] = repaired
    audit_path = site / '.vtm_opencv_payload.json'
    audit_path.write_text(json.dumps(audit, indent=2), encoding='utf-8')
    if audit['problems']:
        raise RuntimeError(f'OpenCV payload remains incomplete after reinstall: {audit_path}')
    probe = site / '.vtm_opencv_probe.json'
    run([python, '-c', PRIVATE_OPENCV_PROBE, site, canonicalize_name(record['package']), record['version'], probe], environment)
    print(f"OpenCV payload verified: {audit['checked_files']} files; probe: {probe}", flush=True)
    return str(probe)

def kernel_distribution_compatible(dist, name, target_version):
    from packaging.specifiers import SpecifierSet
    from packaging.tags import compatible_tags, cpython_tags, parse_tag, platform_tags
    requires_python = dist.metadata.get('Requires-Python')
    if requires_python and (not SpecifierSet(requires_python).contains(target_version)):
        raise ValueError(f'Kernel package requires another Python: {name} {requires_python}, target {target_version}')
    minor = int(target_version.split('.')[1])
    platforms = list(platform_tags())
    tags = set(cpython_tags((3, minor), abis=[f'cp3{minor}'], platforms=platforms))
    tags.update(compatible_tags((3, minor), interpreter=f'cp3{minor}', platforms=platforms))
    wheel = dist.read_text('WHEEL') or ''
    wheel_tags = set().union(*(parse_tag(line[5:].strip()) for line in wheel.splitlines() if line.startswith('Tag: ')))
    if (wheel_tags or minor != sys.version_info.minor) and (not wheel_tags & tags):
        raise ValueError(f'Kernel package ABI cannot be reused in Python {target_version}: {name}')

def plan_kernel_reuse(bundle, manifest, target_version, report_path):
    """Check live metadata and its dependency closure against the fixed offline artifacts."""
    from packaging.markers import default_environment
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    from packaging.version import Version
    environment = {**default_environment(), 'python_version': '.'.join(target_version.split('.')[:2]), 'python_full_version': target_version, 'os_name': 'posix', 'sys_platform': 'linux', 'platform_system': 'Linux', 'platform_machine': 'x86_64', 'implementation_name': 'cpython'}
    selected, live, original = ({}, {}, {canonicalize_name(r['package']): r for r in manifest.get('reused', [])})
    extras = {canonicalize_name(n): set(values) for n, values in manifest.get('extras', {}).items()}
    report = {'target_python': target_version, 'version_changes': [], 'added_dependencies': [], 'excluded_packages': [], 'errors': []}
    protected = {'torch', 'torchvision', 'transformers', 'numpy', 'paddlepaddle-gpu', 'paddleocr', 'paddlex'}
    corrections = KERNEL_REPORT_CORRECTIONS.get(manifest.get('base_report_sha256'), {})

    def select_kernel(name):
        if name in OPENCV_PACKAGES:
            existing = set(selected).intersection(OPENCV_PACKAGES) - {name}
            if existing:
                raise ValueError(f'Multiple OpenCV providers for cv2: {name} conflicts with {sorted(existing)}')
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError as exc:
            raise ValueError(f'Offline dependency unavailable in kernel: {name}') from exc
        if canonicalize_name(dist.metadata.get('Name', '')) != name:
            raise ValueError(f'Kernel distribution identity differs: {name}')
        kernel_distribution_compatible(dist, name, target_version)
        prior = original.get(name)
        baseline = prior['version'] if prior else None
        correction = corrections.get(name, {})
        if baseline == correction.get('reported'):
            baseline = correction.get('active', baseline)
        if prior and (name in protected or name.startswith('nvidia-')) and (Version(dist.version) != Version(baseline)):
            raise ValueError(f'Core kernel package changed: {name} expected {baseline}, got {dist.version}')
        if prior and Version(dist.version) != Version(prior['version']):
            report['version_changes'].append({'package': name, 'reported': prior['version'], 'actual': dist.version})
        if prior is None:
            report['added_dependencies'].append({'package': name, 'version': dist.version})
        live[name] = {'package': name, 'version': dist.version, 'reported_version': prior['version'] if prior else None}
        selected[name] = {'version': dist.version, 'requires': dist.metadata.get_all('Requires-Dist', [])}
    try:
        for record in manifest['records']:
            name = canonicalize_name(record['package'])
            requires = record.get('requires', [])
            if record['kind'] == 'wheel':
                with zipfile.ZipFile(checked_path(bundle, record['path'])) as archive:
                    candidates = [n for n in archive.namelist() if n.endswith('.dist-info/METADATA') and n.count('/') == 1]
                    if len(candidates) != 1:
                        raise ValueError(f"Invalid wheel metadata: {record['path']}")
                    data = Parser().parsestr(archive.read(candidates[0]).decode('utf-8'))
                if canonicalize_name(data['Name']) != name or Version(data['Version']) != Version(record['version']):
                    raise ValueError(f"Offline wheel identity differs: {record['path']}")
                requires = data.get_all('Requires-Dist', [])
            selected[name] = {'version': record['version'], 'requires': requires}
        shipped_opencv = set(selected).intersection(OPENCV_PACKAGES)
        if len(shipped_opencv) > 1:
            raise ValueError(f'Multiple shipped OpenCV providers for cv2: {sorted(shipped_opencv)}')
        for name in original:
            if name in selected:
                raise ValueError(f'Both shipped and reused package: {name}')
            if shipped_opencv and name in OPENCV_PACKAGES:
                report['excluded_packages'].append({'package': name, 'reason': f'cv2 is supplied by the shipped {next(iter(shipped_opencv))} wheel'})
                continue
            select_kernel(name)
        requirements = [('root', Requirement(text)) for text in manifest.get('roots', [])]
        changed = True
        while changed:
            changed = False
            for name, record in list(selected.items()):
                for text in record['requires']:
                    req = Requirement(text)
                    if req.marker and (not any((req.marker.evaluate({**environment, 'extra': extra}) for extra in {'', *extras.get(name, set())}))):
                        continue
                    if (name, req) not in requirements:
                        requirements.append((name, req))
                        changed = True
            for parent, req in requirements:
                if parent == 'root' and req.marker and (not req.marker.evaluate({**environment, 'extra': ''})):
                    continue
                if req.url:
                    raise ValueError(f'Offline URL dependency is unsupported: {parent} -> {req}')
                name = canonicalize_name(req.name)
                if name not in selected:
                    try:
                        select_kernel(name)
                    except ValueError as exc:
                        raise ValueError(f'{parent} -> {req}: {exc}') from exc
                    changed = True
                if not req.specifier.contains(selected[name]['version'], prereleases=True):
                    raise ValueError(f"Unsatisfied offline dependency: {parent} -> {req}; have {selected[name]['version']}")
                active = extras.setdefault(name, set())
                if not req.extras <= active:
                    active.update(req.extras)
                    changed = True
        report.update(ok=True, reused=list(live.values()), dependency_edges_checked=len(requirements), opencv_provider=next(iter(set(selected).intersection(OPENCV_PACKAGES)), None))
    except Exception as exc:
        report.update(ok=False)
        report['errors'].append(str(exc))
        raise
    finally:
        report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    for change in report['version_changes']:
        print(f"Compatible kernel update: {change['package']} {change['reported']} -> {change['actual']}", flush=True)
    if report['opencv_provider']:
        print(f"OpenCV provider: {report['opencv_provider']}; excluded {len(report['excluded_packages'])} kernel variants", flush=True)
    print(f'Kernel dependency plan: {len(live)} packages, {len(requirements)} dependency constraints checked', flush=True)
    return list(live.values())

def reuse_kernel_packages(records, site, target_version, *, bundle=None, manifest=None):
    """Copy Python files and link ABI-compatible native files, never add cp313 site to cp310."""
    from packaging.version import Version
    if manifest is not None:
        records = plan_kernel_reuse(bundle, manifest, target_version, site / '.vtm_kernel_plan.json')
    links, copied, reused = ([], [], [])
    try:
        for expected in records:
            dist = metadata.distribution(expected['package'])
            if Version(dist.version) != Version(expected['version']):
                raise ValueError(f"Kernel package changed: {expected['package']} expected {expected['version']}, got {dist.version}")
            kernel_distribution_compatible(dist, expected['package'], target_version)
            source_root = Path(dist.locate_file('')).resolve()
            record_text = dist.read_text('RECORD')
            if not record_text:
                raise ValueError(f"Kernel package has no verifiable RECORD: {expected['package']}")
            count = 0
            for row in csv.reader(io.StringIO(record_text)):
                if not row:
                    continue
                item, checksum, _size = row
                relative = Path(item)
                if relative.is_absolute() or '..' in relative.parts or '__pycache__' in relative.parts or (relative.suffix == '.pyc'):
                    continue
                source = Path(dist.locate_file(item)).resolve()
                if not source.is_relative_to(source_root) or not source.is_file():
                    raise ValueError(f'Kernel package file unavailable: {item}')
                if checksum:
                    algorithm, expected_hash = checksum.split('=', 1)
                    hasher = hashlib.new(algorithm)
                    with source.open('rb') as stream:
                        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                            hasher.update(block)
                    actual = base64.urlsafe_b64encode(hasher.digest()).rstrip(b'=').decode()
                    if actual != expected_hash:
                        raise ValueError(f"Kernel RECORD checksum differs: {expected['package']}/{item}")
                elif '.so' in source.name:
                    raise ValueError(f'Unverifiable native kernel file: {item}')
                target = site / relative
                if not target.parent.resolve().is_relative_to(site.resolve()):
                    raise ValueError(f'Reused package file escapes private site: {relative}')
                target.parent.mkdir(parents=True, exist_ok=True)
                if target.exists():
                    if sha256(target) != sha256(source):
                        raise ValueError(f'Package files overlap with different bytes: {relative}')
                    continue
                if '.so' in source.name:
                    target.symlink_to(source)
                    links.append(relative.as_posix())
                else:
                    shutil.copy2(source, target)
                copied.append(relative.as_posix())
                count += 1
            reused.append({'package': expected['package'], 'version': dist.version, 'reported_version': expected.get('reported_version', expected['version']), 'files': count})
    finally:
        (site / '.vtm_kernel_links.json').write_text(json.dumps(links, indent=2), encoding='utf-8')
        (site / '.vtm_kernel_files.json').write_text(json.dumps(copied, indent=2), encoding='utf-8')
    print(f'Reused {len(reused)} kernel distributions in Python {target_version}', flush=True)
    return reused

def component_manifest(bundle, name):
    path = bundle / f'{name}_manifest.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {}

def private_python(python, site):
    """Use the same private CUDA library search order for probes and OCR workers."""
    libraries = sorted({str(path.parent) for path in (site / 'nvidia').rglob('*.so*') if path.is_file()})
    wrapper = python.with_name('vtm-python')
    prefix = shlex.quote(':'.join(libraries))
    script = '#!/bin/sh\nunset PYTHONPATH PYTHONHOME PYTHONUSERBASE\nexport PYTHONNOUSERSITE=1\nexport MPLBACKEND=Agg\n'
    if libraries:
        script += 'export LD_LIBRARY_PATH=' + prefix + '"${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\n'
    command = isolated_python_command(python, [])
    command[-1] = wrapper
    script += 'exec ' + ' '.join((shlex.quote(str(arg)) for arg in command)) + ' "$@"\n'
    wrapper.write_text(script, encoding='utf-8', newline='\n')
    wrapper.chmod(493)
    return wrapper

def native_environment(cuda=None):
    env = os.environ.copy()
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE', '_PYTHON_SYSCONFIGDATA_NAME', '_PYTHON_HOST_PLATFORM'):
        env.pop(name, None)
    env.update(PIP_NO_INDEX='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True', PYTHONNOUSERSITE='1', MPLBACKEND='Agg')
    if cuda:
        env.update(CUDA_HOME=str(cuda), CUDA_PATH=str(cuda), TORCH_CUDA_ARCH_LIST='12.0', MAX_JOBS=os.environ.get('MAX_JOBS', '2'))
        env['PATH'] = str(cuda / 'bin') + os.pathsep + env.get('PATH', '')
        env['LD_LIBRARY_PATH'] = str(cuda / 'lib64') + os.pathsep + env.get('LD_LIBRARY_PATH', '')
    return env

def repair_python_build_config(python_root, environment):
    """Relocate standalone Python's build paths in the writable runtime only."""
    python_root = Path(python_root).resolve()
    configs = list(python_root.glob('lib/python*/_sysconfigdata*.py'))
    if len(configs) != 1 or not configs[0].resolve().is_relative_to(python_root):
        raise ValueError(f'Cannot identify private Python build config: {python_root}')
    path = configs[0]
    tree = ast.parse(path.read_text(encoding='utf-8'))
    values = next((ast.literal_eval(n.value) for n in tree.body if isinstance(n, ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'build_time_vars' for t in n.targets))))
    previous = dict(values)
    old_prefix = values.get('prefix', '/install')
    if old_prefix != str(python_root):
        pattern = re.compile(re.escape(old_prefix) + '(?=/|$|[\\s:])')
        values = {key: pattern.sub(lambda _: str(python_root), value) if isinstance(value, str) else value for key, value in values.items()}
    values.update(CC=environment['CC'] + ' -pthread', CXX=environment['CXX'] + ' -pthread', LDSHARED=environment['LDSHARED'], LDCXXSHARED=environment['LDCXXSHARED'], BLDSHARED=environment['LDSHARED'])
    changed = sorted((key for key in values if values[key] != previous.get(key)))
    if changed:
        path.write_text('build_time_vars = ' + pprint.pformat(values, sort_dicts=True, width=110) + '\n', encoding='utf-8')
        for cached in (path.parent / '__pycache__').glob(path.stem + '.*.pyc'):
            if cached.is_file():
                cached.unlink()
    include = Path(values['INCLUDEPY'])
    if not include.is_relative_to(python_root) or not (include / 'Python.h').is_file() or (not (include / 'pyconfig.h').is_file()):
        raise ValueError(f'Private Python headers unavailable after relocation: {include}')
    return {'python_root': str(python_root), 'include': str(include), 'ext_suffix': values['EXT_SUFFIX'], 'CC': values['CC'], 'CXX': values['CXX'], 'changed_keys': changed}

def prepare_build_toolchain(python, runtime, environment):
    """Prove that the host compiler can build and import a private Python extension."""
    env = dict(environment)
    cc, cxx = (shutil.which(name, path=env.get('PATH')) for name in ('gcc', 'g++'))
    if not cc or not cxx:
        raise RuntimeError('Native offline builds need gcc and g++; compiler preflight stopped before source installs')
    env.update(CC=cc, CXX=cxx, LDSHARED=shlex.quote(cc) + ' -pthread -shared', LDCXXSHARED=shlex.quote(cxx) + ' -pthread -shared', NVCC_CCBIN=cxx, MAX_JOBS=env.get('MAX_JOBS', '2'), PIP_DISABLE_PIP_VERSION_CHECK='1', PYTHONUNBUFFERED='1')
    configuration = repair_python_build_config(Path(python).parent.parent, env)
    for label, compiler in [('CC_version', cc), ('CXX_version', cxx)]:
        configuration[label] = subprocess.check_output([compiler, '--version'], env=env, text=True).splitlines()[0]
        print(configuration[label], flush=True)
    probe = Path(runtime) / 'build_probe'
    probe.mkdir(parents=True, exist_ok=True)
    source = probe / '_vtm_build_probe.c'
    source.write_text('#include <Python.h>\nstatic PyObject *answer(PyObject *self, PyObject *args) { return PyLong_FromLong(42); }\nstatic PyMethodDef methods[] = {{"answer", answer, METH_NOARGS, "probe"}, {NULL, NULL, 0, NULL}};\nstatic struct PyModuleDef module = {PyModuleDef_HEAD_INIT, "_vtm_build_probe", NULL, -1, methods};\nPyMODINIT_FUNC PyInit__vtm_build_probe(void) { return PyModule_Create(&module); }\n', encoding='utf-8')
    extension = probe / ('_vtm_build_probe' + configuration['ext_suffix'])
    run([cc, '-shared', '-fPIC', '-O2', '-I' + configuration['include'], source, '-o', extension], env)
    run(isolated_python_command(python, ['-c', "import sys; sys.path.insert(0,sys.argv[1]); import _vtm_build_probe; assert _vtm_build_probe.answer()==42; print('C extension build/import preflight OK')", probe]), env)
    (probe / 'toolchain.json').write_text(json.dumps(configuration, indent=2), encoding='utf-8')
    return (env, configuration)

def patch_mmcv_build_source(source):
    """Apply narrowly scoped source fixes for the pinned Torch 2.10/CUDA 13 build."""
    source = Path(source)
    changed = []
    for relative, old, new in (('setup.py', '-std=c++14', '-std=c++17'), ('mmcv/ops/csrc/pytorch/spconv_utils.h', 'tensor.type().scalarType()', 'tensor.scalar_type()')):
        path = source / relative
        if not path.is_file():
            continue
        text = path.read_text(encoding='utf-8')
        if old in text:
            path.write_text(text.replace(old, new), encoding='utf-8')
            changed.append(relative)
    return changed

def build_offline_source_wheel(bundle, record, runtime, python, environment, profile):
    """Build once with verbose logs; cache only a checksummed, ABI-specific wheel."""
    identity = {'source_sha256': record['sha256'], 'package': record['package'], 'version': record['version'], 'profile': profile, 'build_schema': 2, 'build_environment': {name: environment.get(name) for name in ('CC', 'CXX', 'LDSHARED', 'LDCXXSHARED', 'CFLAGS', 'CPPFLAGS', 'CXXFLAGS', 'LDFLAGS', 'CUDA_HOME', 'NVCC_CCBIN', 'NVCC_FLAGS', 'MMCV_WITH_OPS', 'TORCH_CUDA_ARCH_LIST')}}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    name = re.sub('[-_.]+', '-', record['package']).lower()
    cache = Path(runtime) / 'built_source_wheels' / name / key
    cache.mkdir(parents=True, exist_ok=True)
    receipt = cache / 'receipt.json'
    if receipt.is_file():
        try:
            saved = json.loads(receipt.read_text(encoding='utf-8'))
        except (ValueError, OSError):
            saved = {}
    else:
        saved = {}
    if 'wheel' in saved:
        wheel = checked_path(cache, saved['wheel'])
        if saved.get('identity') == identity and wheel.is_file() and (sha256(wheel) == saved.get('sha256')):
            print(f'Reuse built wheel: {wheel}', flush=True)
            return wheel
    source = prepare_build_source(bundle, record, cache / 'source_runtime')
    if name == 'mmcv-full':
        print(f'MMCV build source patches: {patch_mmcv_build_source(source)}', flush=True)
    logs = Path(runtime) / 'build_logs'
    logs.mkdir(parents=True, exist_ok=True)
    log = logs / f'{name}-{key}.pip.log'
    print(f'Build {name}; full compiler/pip log: {log}', flush=True)
    with tempfile.TemporaryDirectory(prefix='wheel-', dir=cache) as temporary:
        temporary = Path(temporary).resolve()
        if not temporary.is_relative_to(cache.resolve()):
            raise ValueError('Build staging directory escapes private cache')
        try:
            run([python, '-u', '-m', 'pip', '--log', log, '-vvv', 'wheel', '--no-index', '--no-deps', '--no-build-isolation', '--no-cache-dir', '--no-clean', '--find-links', Path(bundle) / 'wheels', '--wheel-dir', temporary, source], environment)
        except subprocess.CalledProcessError as exc:
            if name == 'polygon3':
                print('Polygon3 direct compiler diagnostic (original pip failure is retained):', flush=True)
                try:
                    run([python, '-u', 'setup.py', 'build_ext', '--verbose'], environment, cwd=source)
                except subprocess.CalledProcessError:
                    pass
            raise RuntimeError(f'Build {name} failed. Full compiler log: {log}; source kept at {source}') from exc
        wheels = list(temporary.glob('*.whl'))
        if len(wheels) != 1:
            raise RuntimeError(f'Build {name} produced {len(wheels)} wheels; inspect {log}')
        with zipfile.ZipFile(wheels[0]) as archive:
            if archive.testzip() is not None:
                raise ValueError(f'Corrupt built wheel: {wheels[0]}')
        destination = cache / wheels[0].name
        wheels[0].replace(destination)
    receipt.write_text(json.dumps({'identity': identity, 'wheel': destination.name, 'sha256': sha256(destination)}, indent=2), encoding='utf-8')
    return destination

def install_paddle(bundle, runtime):
    manifest = component_manifest(bundle, 'paddle')
    directory = runtime / ('paddle_py313_cu130' if manifest.get('cuda') == '13.0' else 'paddle_py313')
    python = directory / 'bin/python'
    env = native_environment()
    if not python.exists():
        run(isolated_python_command(sys.executable, ['-m', 'venv', '--without-pip', directory]), env)
    if manifest.get('reused'):
        site, version = target_site(python, runtime)
        clear_reused_packages(site, private_kernel_cleanup_records(manifest))
        initial_runner = private_python(python, site)
    else:
        initial_runner = python
    run(host_pip_command(['--python', initial_runner, 'install', '--no-index', '--no-deps', '--find-links', bundle / 'wheels', '-r', bundle / 'requirements/paddle.txt']), env)
    reused = reuse_kernel_packages(manifest['reused'], site, version, bundle=bundle, manifest=manifest) if manifest.get('reused') else []
    runner = private_python(python, site) if manifest.get('reused') else python
    if manifest.get('reused'):
        ensure_private_opencv(bundle, manifest, site, runner, env, runtime)
        run(host_pip_command(['--python', runner, 'check']), env)
    probe = "import json, paddle\nfrom paddleocr import PaddleOCR, PaddleOCRVL\nassert paddle.is_compiled_with_cuda(), 'Paddle GPU wheel required'\nassert paddle.device.cuda.device_count() > 0, 'No CUDA device for Paddle'\npaddle.set_device('gpu:0')\nx = paddle.ones([2, 2]); y = paddle.matmul(x, x)\nprint(json.dumps({'paddle': paddle.__version__, 'cuda_devices': paddle.device.cuda.device_count(), 'gpu_sum': float(y.sum().item())}))\n"
    run([runner, '-c', probe], env)
    support = install_renderer(bundle, runtime)
    return {'paddle_python': str(runner), **support, 'paddle_gpu_import_verified': True, 'paddle_kernel_reused': reused}

def install_renderer(bundle, runtime):
    env = native_environment()
    common = runtime / 'common_site'
    renderer = next((bundle / 'wheels').glob('pypdfium2-5.15.0-*.whl'))
    run(host_pip_command(['install', '--no-index', '--no-deps', '--target', common, renderer]), env)
    return {'common_site': str(common)}

def install_tonggu(bundle, runtime, records):
    manifest = component_manifest(bundle, 'tonggu')
    if manifest.get('cuda') == '13.0':
        return install_tonggu_cu130(bundle, runtime, manifest)
    system = json.loads((bundle / 'system_assets_manifest.json').read_text(encoding='utf-8'))
    entries = {r['package']: r for r in system['records']}
    directory = runtime / 'tonggu_py310'
    python = directory / 'python/bin/python3.10'
    if not python.exists():
        prepare_python_runtime(bundle, entries['python-runtime'], directory)
    env = native_environment()
    bootstrap = ['pip==25.3', 'setuptools==80.10.2', 'wheel==0.48.0', 'Cython==3.0.12', 'ninja==1.13.2', 'numpy==2.2.6']
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', '--find-links', bundle / 'wheels', *bootstrap], env)
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', '--no-build-isolation', '--find-links', bundle / 'wheels', '--find-links', bundle / 'sources', '-r', bundle / 'requirements/tonggu.txt'], env)
    cuda = runtime / 'cuda12.8'
    for name in ('cuda_nvcc', 'cuda_cudart', 'cuda_cccl'):
        extracted = runtime / ('extract_' + name)
        extract_tar(checked_path(bundle, entries[name]['path']), extracted)
        roots = [p for p in extracted.iterdir() if p.is_dir()]
        if len(roots) != 1:
            raise ValueError(f'Unexpected CUDA archive layout: {name}')
        shutil.copytree(roots[0], cuda, dirs_exist_ok=True)
    env = native_environment(cuda)
    run([cuda / 'bin/nvcc', '--version'], env)
    if not (cuda / 'lib64').exists() and (cuda / 'lib').exists():
        (cuda / 'lib64').symlink_to('lib', target_is_directory=True)
    repo = prepare_tonggu_checkout(bundle, entries['tonggu-source'], runtime)
    built = runtime / 'built_wheels'
    built.mkdir(exist_ok=True)
    mmcv_source = checked_path(bundle, entries['mmcv-full']['path'])
    if not list(built.glob('mmcv_full-1.7.2-*.whl')):
        run([python, '-m', 'pip', 'wheel', '--no-index', '--no-deps', '--no-build-isolation', '--find-links', bundle / 'wheels', '--wheel-dir', built, mmcv_source], {**env, 'MMCV_WITH_OPS': '1', 'PIP_NO_BUILD_ISOLATION': '1'})
    mmcv = next(built.glob('mmcv_full-1.7.2-*.whl'))
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', mmcv], env)
    run([python, repo / 'tools/patch_openmmlab.py'], env)
    probe = "import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nassert torch.version.cuda.startswith('12.8'), torch.version.cuda\nboxes = torch.tensor([[5., 5., 2., 2., 0.]], device='cuda')\nscores = torch.tensor([.9], device='cuda')\n_, keep = mmcv.ops.nms_rotated(boxes, scores, .5)\nassert keep.numel() == 1\nprint(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'mmcv': mmcv.__version__, 'sm': list(torch.cuda.get_device_capability()), 'rotated_nms_gpu': 'OK'}))\n"
    run([python, '-c', probe], env)
    return {'tonggu_python': str(python), 'tonggu_repo': str(repo), 'tonggu_compiled_ops_verified': True, 'tonggu_cuda_home': str(cuda)}

def install_tonggu_cu130(bundle, runtime, manifest):
    system = component_manifest(bundle, 'system_assets')
    entries = {record['package']: record for record in system['records']}
    directory = runtime / 'tonggu_py310_cu130'
    python = directory / 'python/bin/python3.10'
    if not python.exists():
        prepare_python_runtime(bundle, entries['python-runtime'], directory)
    nvcc = shutil.which('nvcc')
    if not nvcc:
        raise RuntimeError('TongGu needs the CUDA 13.0 toolkit verified in the Kaggle report')
    candidates = [Path(value) for value in (os.environ.get('CUDA_HOME'), os.environ.get('CUDA_PATH')) if value]
    candidates += [Path(nvcc).resolve().parents[1], Path('/usr/local/cuda'), Path('/usr/local/cuda-13.0')]
    cuda = next((path.resolve() for path in candidates if (path / 'include/cuda.h').is_file()), candidates[0].resolve())
    compiler = subprocess.check_output([nvcc, '--version'], text=True)
    if 'release 13.0' not in compiler or not (cuda / 'include/cuda.h').is_file():
        raise RuntimeError(f'Expected a complete CUDA 13.0 toolkit, found {cuda}')
    env = native_environment(cuda)
    site, version = target_site(python, runtime)
    env, toolchain = prepare_build_toolchain(python, runtime, env)
    clear_reused_packages(site, private_kernel_cleanup_records(manifest))
    initial_runner = private_python(python, site)
    run(host_pip_command(['--python', initial_runner, 'install', '--no-index', '--no-deps', '--find-links', bundle / 'wheels', '-r', bundle / 'requirements/tonggu.txt']), env)
    reused = reuse_kernel_packages(manifest.get('reused', []), site, version, bundle=bundle, manifest=manifest)
    runner = private_python(python, site)
    ensure_private_opencv(bundle, manifest, site, runner, env, runtime)
    torch_probe = "import torch; assert torch.version.cuda == '13.0', torch.version.cuda; x=torch.ones((2,2),device='cuda'); assert float((x@x).sum().item()) == 8.0; print(torch.__version__,torch.cuda.get_device_capability())"
    run([runner, '-c', torch_probe], env)
    versions = {record['package']: record['version'] for record in [*manifest['records'], *reused]}
    profile = {'python': version, 'numpy': versions.get('numpy'), 'torch': versions.get('torch'), 'cuda': '13.0', 'arch': env['TORCH_CUDA_ARCH_LIST'], 'toolchain': {k: v for k, v in toolchain.items() if k != 'changed_keys'}}
    for record in manifest['records']:
        if record['kind'] != 'wheel' and record['package'] != 'mmcv-full':
            wheel = build_offline_source_wheel(bundle, record, runtime, runner, env, profile)
            run([runner, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', wheel], env)
    source_probe = "import Polygon, numpy as np, jieba, zhconv, cv2\nassert Polygon.withNumPy, 'Polygon must be built with the selected NumPy headers'\np = Polygon.Polygon(np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]], dtype=np.float64))\nassert abs(p.area()-4.) < 1e-9\nassert cv2.resize(np.zeros((2,2,3), dtype=np.uint8),(4,4)).shape == (4,4,3)\nprint('Polygon/NumPy, jieba, zhconv, OpenCV native imports/operations OK')\n"
    run([runner, '-c', source_probe], env)
    repo = prepare_tonggu_checkout(bundle, entries['tonggu-source'], runtime)
    mmcv_source = next((record for record in manifest['records'] if record['package'] == 'mmcv-full'))
    mmcv = build_offline_source_wheel(bundle, mmcv_source, runtime, runner, {**env, 'MMCV_WITH_OPS': '1', 'PIP_NO_BUILD_ISOLATION': '1'}, profile)
    run([runner, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', mmcv], env)
    run([runner, repo / 'tools/patch_openmmlab.py'], env)
    run([runner, '-m', 'pip', 'check'], env)
    probe = "import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nb = torch.tensor([[5.,5.,2.,2.,0.]], device='cuda')\ns = torch.tensor([.9], device='cuda')\n_, keep = mmcv.ops.nms_rotated(b,s,.5)\nassert keep.numel() == 1\nprint(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'mmcv':mmcv.__version__,'rotated_nms_gpu':'OK'}))\n"
    run([runner, '-c', probe], env)
    return {'tonggu_python': str(runner), 'tonggu_repo': str(repo), 'tonggu_cuda_home': str(cuda), 'tonggu_compiled_ops_verified': True, 'tonggu_kernel_reused': reused}

import json
import os
import subprocess
import sys
PROBE = "\nimport importlib.util, json, pathlib, sys\nengine, repo, custom_config = sys.argv[1:4]\npackages = {'hunyuan': ['torch', 'transformers'], 'paddle': ['paddle', 'paddleocr'],\n            'ppocr': ['paddle', 'paddleocr'],\n            'tonggu': ['torch', 'transformers', 'mmcv', 'mmdet', 'mmrotate', 'yaml', 'timm', 'cv2', 'Polygon']}[engine]\nmissing = [p for p in [*packages, 'pypdfium2', 'PIL', 'tqdm'] if importlib.util.find_spec(p) is None]\nchecks = []\nif engine == 'tonggu':\n    root = pathlib.Path(repo)\n    if not (root / 'tongguocr/pipeline.py').is_file(): missing.append('tonggu_repo/tongguocr/pipeline.py')\n    if custom_config:\n        if not pathlib.Path(custom_config).is_file(): missing.append('tonggu_config')\n    else:\n        for p in ['weights/layout/finetune_infer.pth', 'weights/ocr/config.json']:\n            if not (root / p).is_file(): missing.append(p)\nif not missing:\n    import pypdfium2 as pdfium\n    document = pdfium.PdfDocument.new()\n    page = document.new_page(8,8)\n    bitmap = page.render(scale=1)\n    image = bitmap.to_pil()\n    assert image.size == (8,8)\n    image.close(); bitmap.close(); page.close(); document.close()\n    checks.append('pdfium_render')\n    if engine == 'hunyuan':\n        from transformers import AutoProcessor\n        try:\n            from transformers import HunYuanVLForConditionalGeneration as Model\n        except ImportError:\n            from transformers import AutoModelForImageTextToText as Model\n        assert callable(AutoProcessor) and callable(Model)\n        checks.append('hunyuan_worker_api')\n    elif engine in {'paddle','ppocr'}:\n        from paddleocr import PaddleOCR, PaddleOCRVL\n        assert callable(PaddleOCR) and callable(PaddleOCRVL)\n        checks.append('paddle_worker_api')\n    else:\n        import cv2\n        assert callable(getattr(cv2,'resize',None)) and callable(getattr(cv2,'cvtColor',None)), 'Incomplete cv2 API'\n        from transformers import AutoModelForImageTextToText, AutoProcessor\n        import mmcv.ops, mmdet, mmrotate, Polygon, timm\n        sys.path.insert(0,str(root.resolve()))\n        from tongguocr.config import load_config\n        from tongguocr.pipeline import run_pipeline_on_image\n        assert callable(run_pipeline_on_image) and callable(AutoProcessor) and callable(AutoModelForImageTextToText)\n        config = load_config(custom_config or None)\n        for label, value, directory in (\n            ('tonggu_layout_weights',config['layout_inference']['detector_checkpoint_path'],False),\n            ('tonggu_layout_config',config['layout_inference']['detector_config_path'],False),\n            ('tonggu_ocr_weights',config['ocr']['transformers']['model_path'],True),\n        ):\n            path = pathlib.Path(value)\n            if not path.is_absolute(): path = root / path\n            if directory:\n                if not (path/'config.json').is_file(): missing.append(label+': '+str(path))\n            elif not path.is_file(): missing.append(label+': '+str(path))\n        checks.append('tonggu_worker_api')\nprint('VTM_PREFLIGHT_RESULT=' + json.dumps({'missing': missing, 'python': sys.executable, 'checks': checks}))\n"

def preflight_models(models):
    rows = []
    for name, cfg in models.items():
        if cfg.worker_wsl_distro:
            if os.name != 'nt' or not cfg.worker_python.startswith('/'):
                rows.append({'model': name, 'ready': False, 'detail': 'WSL cần Windows và worker_python là đường dẫn Linux tuyệt đối'})
                continue
            command = ['wsl.exe', '--distribution', cfg.worker_wsl_distro, '--exec', cfg.worker_python]
        else:
            command = [cfg.worker_python or sys.executable]
        command += ['-c', PROBE, name, cfg.tonggu_repo, cfg.tonggu_config]
        try:
            environment = os.environ.copy()
            environment['PYTHONPATH'] = os.pathsep.join([*cfg.worker_pythonpath, environment.get('PYTHONPATH', '')])
            if name in {'paddle', 'ppocr', 'tonggu'}:
                environment['MPLBACKEND'] = 'Agg'
            environment.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True', USE_TF='0', USE_FLAX='0')
            result = subprocess.run(command, capture_output=True, timeout=120, env=environment)
            if result.returncode:
                output = result.stderr or result.stdout
                detail = output.decode('utf-16-le' if b'\x00' in output else 'utf-8', errors='replace').strip()
                rows.append({'model': name, 'ready': False, 'detail': detail[-4000:] or f'exit={result.returncode}'})
                continue
            lines = result.stdout.decode('utf-8').strip().splitlines()
            marker = 'VTM_PREFLIGHT_RESULT='
            payload = next((line.split(marker, 1)[1] for line in reversed(lines) if marker in line), lines[-1])
            probe = json.loads(payload)
            rows.append({'model': name, 'ready': not probe['missing'], 'detail': ', '.join(probe['missing']) or probe['python']})
        except (OSError, subprocess.SubprocessError, ValueError, IndexError) as exc:
            rows.append({'model': name, 'ready': False, 'detail': str(exc)})
    return rows

NOTEBOOK_SETUP_SIGNATURE = 'de67c152b124b0ec5e174931c5931ecc8a9b746bb713a25a59f6a19a20bde303'

_INSTALL_OUTPUT = None

def run(args, environment=None, *, cwd=None):
    print("RUN:", command_description(args), flush=True)
    subprocess.run(list(map(str, args)), env=environment, cwd=cwd, check=True,
                   stdout=_INSTALL_OUTPUT, stderr=subprocess.STDOUT if _INSTALL_OUTPUT else None)

def install_offline_component(session, component):
    """Run setup functions defined above; dependency subprocesses write to a log."""
    global _INSTALL_OUTPUT
    log = session.log_dir / f"{component}.log"
    print(f"Setup {component}; log: {log}", flush=True)
    config = session.runtime_dir / "runtime_config.json"
    state = json.loads(config.read_text(encoding="utf-8")) if config.is_file() else {}
    receipt_path = session.runtime_dir / "notebook_setup_receipt.json"
    prior_receipt = json.loads(receipt_path.read_text(encoding="utf-8")) if receipt_path.is_file() else {}
    key = getattr(session, "bundle_key", None)
    completed = set(prior_receipt.get("components", [])) if prior_receipt.get("bundle_key") == key else set()
    pending = {"paddle", "tonggu"} if component == "all" else {component}
    completed.difference_update(pending)

    def checkpoint(name, result):
        state.update(result)
        state["bundle_root"] = str(session.bundle)
        state.pop("code_root", None)
        temporary = config.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")
        temporary.replace(config)
        completed.add(name)
        if key is not None:
            temporary = receipt_path.with_suffix(".json.tmp")
            temporary.write_text(json.dumps({"bundle_key": key, "components": sorted(completed)}, indent=2), encoding="utf-8")
            temporary.replace(receipt_path)
    previous_output = _INSTALL_OUTPUT
    try:
        with log.open("w", encoding="utf-8") as stream, redirect_stdout(stream), redirect_stderr(stream):
            _INSTALL_OUTPUT = stream
            records = verify(session.bundle, report_path=session.log_dir / "source_tree_report.json")
            if component == "app":
                checkpoint("app", install_renderer(session.bundle, session.runtime_dir))
            if component in {"paddle", "all"}:
                checkpoint("paddle", install_paddle(session.bundle, session.runtime_dir))
            if component in {"tonggu", "all"}:
                checkpoint("tonggu", install_tonggu(session.bundle, session.runtime_dir, records))
    except Exception as exc:
        tail = "\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-100:])
        diagnostic = session.log_dir / "source_tree_report.json"
        raise RuntimeError(f"Cài {component} lỗi; đã dừng trước OCR. {exc}\nLog: {log}\nChi tiết cây nguồn: {diagnostic}\n{tail}") from exc
    finally:
        _INSTALL_OUTPUT = previous_output

def activate_pdfium(session):
    if session.state.get("common_site"):
        sys.path.insert(0, session.state["common_site"])

def new_ocr_session(*, sources, packages_dir=None, input_dir=None, weights=None, books=None, max_books=1, pages='', limit=None, run_ocr=None, work_dir=None, advanced=None):
    session = SimpleNamespace()
    session.sources = sources
    session.options = advanced or {}
    session.on_kaggle = Path('/kaggle/working').is_dir()
    session.run_ocr = session.on_kaggle if run_ocr is None else run_ocr
    session.bundle = Path(packages_dir) if packages_dir else None
    session.auto_find_bundle = packages_dir is None
    session.input_dir = Path('/kaggle/input' if session.on_kaggle else 'data')
    if input_dir is not None:
        session.input_dir = Path(input_dir) if input_dir else None
    session.work_dir = Path(work_dir or ('/kaggle/working/four_model_consensus' if session.on_kaggle else 'work/four_model_consensus'))
    session.weights = weights or {}
    session.books = ['1'] if books is None else books
    session.max_books, session.pages, session.limit = (max_books, pages, limit)
    session.runtime_dir = Path(session.options.get('runtime_dir', '/tmp/vtm_kaggle_runtime'))
    session.log_dir = Path(session.options.get('log_dir', '/kaggle/working/offline_install_logs'))
    session.state = {}
    session.manifest = Path(session.options.get('existing_manifest', session.work_dir / 'manifest.json'))
    session.result = None
    return session

def validate_inputs(session):
    if not session.run_ocr:
        if not session.manifest.is_file():
            raise FileNotFoundError(f'Chưa có checkpoint: {session.manifest}. Chọn WORK_DIR có manifest.json hoặc bật RUN_OCR.')
        return
    if session.input_dir is None:
        raise ValueError('Chưa điền DATA_DIR ở bước 1: thư mục PDF trong Dataset đã gắn.')
    if not session.input_dir.is_dir():
        raise FileNotFoundError(f'Chưa có thư mục PDF: {session.input_dir}')
    missing = []
    required = ('hunyuan', 'ppocr_det', 'ppocr_rec', 'paddle_vl', 'paddle_layout')
    required += ('tonggu_config',) if session.weights.get('tonggu_config') else ('tonggu_ocr', 'tonggu_layout')
    for name in required:
        value = session.weights.get(name)
        if not value:
            if session.on_kaggle:
                missing.append(f'WEIGHTS[{name!r}] chưa điền')
            continue
        if name == 'tonggu_config' and os.name == 'nt' and (not session.on_kaggle):
            continue
        path = Path(value)
        exists = path.is_file() if name in {'tonggu_config', 'tonggu_layout'} else path.is_dir()
        if not exists:
            missing.append(f'{name}: {path}')
    if missing:
        raise FileNotFoundError('Thiếu weights cục bộ; kiểm tra MODELS_DIR và cây thư mục models ở bước 1:\n' + '\n'.join(missing))

def find_packages(session):
    if session.bundle is not None:
        bundle = session.bundle.resolve()
        if not (bundle / 'paddle_manifest.json').is_file() and (bundle / 'packages/paddle_manifest.json').is_file():
            bundle = bundle / 'packages'
        for name in ('paddle_manifest.json', 'tonggu_manifest.json', 'system_assets_manifest.json'):
            if not (bundle / name).is_file():
                raise FileNotFoundError(f'PACKAGES_DIR chưa đúng: thiếu {bundle / name}. Điền thư mục chứa manifest ở bước 1.')
        return bundle
    if not session.auto_find_bundle:
        raise ValueError('Chưa điền PACKAGES_DIR ở bước 1: thư mục chứa paddle_manifest.json và system_assets_manifest.json.')
    matches = []
    for visited, (parent, dirs, files) in enumerate(os.walk('/kaggle/input'), 1):
        if {'paddle_manifest.json', 'system_assets_manifest.json'} <= set(files):
            matches.append(Path(parent))
            dirs[:] = []
        if len(Path(parent).relative_to('/kaggle/input').parts) >= 6:
            dirs[:] = []
        if visited >= 500:
            break
    if len(matches) != 1:
        raise RuntimeError(f'Tìm thấy {len(matches)} Dataset packages. Điền PACKAGES_DIR ở bước 1.')
    return matches[0].resolve()

def runtime_files_ready(component, state):
    if component == 'paddle':
        return state.get('paddle_gpu_import_verified') is True and Path(state.get('paddle_python', '')).is_file() and (Path(state.get('common_site', '')) / 'pypdfium2/__init__.py').is_file()
    return state.get('tonggu_compiled_ops_verified') is True and Path(state.get('tonggu_python', '')).is_file() and (Path(state.get('tonggu_repo', '')) / 'tongguocr/pipeline.py').is_file()

def runtime_healthy(session, component):
    if not runtime_files_ready(component, session.state):
        return False
    code = "import paddle; from paddleocr import PaddleOCR, PaddleOCRVL; paddle.set_device('gpu:0'); x=paddle.ones([2,2]); assert float(paddle.matmul(x,x).sum().item()) == 8.0" if component == 'paddle' else "import torch, mmcv.ops, mmdet, mmrotate; b=torch.tensor([[5.,5.,2.,2.,0.]],device='cuda'); s=torch.tensor([.9],device='cuda'); assert mmcv.ops.nms_rotated(b,s,.5)[1].numel() == 1"
    try:
        result = subprocess.run([session.state[component + '_python'], '-c', code], capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=120, env={**os.environ, 'MPLBACKEND': 'Agg'})
        (session.log_dir / f'cache_{component}.log').write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
        return result.returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False

def prepare_offline_runtime(session):
    if not session.on_kaggle:
        return
    session.log_dir.mkdir(parents=True, exist_ok=True)
    session.bundle = find_packages(session)
    session.runtime_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256()
    for name in ('paddle_manifest.json', 'tonggu_manifest.json', 'system_assets_manifest.json', 'app_manifest.json'):
        path = session.bundle / name
        if path.is_file():
            fingerprint.update(name.encode())
            fingerprint.update(path.read_bytes())
    fingerprint.update(session.sources['installer'])
    fingerprint.update(str(sys.version_info[:2]).encode())
    key = fingerprint.hexdigest()
    session.bundle_key = key
    config = session.runtime_dir / 'runtime_config.json'
    receipt_path = session.runtime_dir / 'notebook_setup_receipt.json'
    session.state = json.loads(config.read_text(encoding='utf-8')) if config.is_file() else {}
    receipt = json.loads(receipt_path.read_text(encoding='utf-8')) if receipt_path.is_file() else {}
    requested = ['paddle', 'tonggu'] if session.run_ocr else []
    reuse = session.options.get('reuse_runtime', True) and receipt.get('bundle_key') == key
    missing = [name for name in requested if not (reuse and name in receipt.get('components', []) and runtime_healthy(session, name))]
    support = (Path(session.state.get('common_site', '')) / 'pypdfium2/__init__.py').is_file()
    if missing or not support or (not requested and (not reuse)):
        install_offline_component(session, 'all' if len(missing) == 2 else missing[0] if missing else 'app')
        session.state = json.loads(config.read_text(encoding='utf-8'))
    if any((not runtime_files_ready(name, session.state) for name in requested)):
        raise RuntimeError('Runtime chưa đủ sau bước cài. Xem log setup. Đã dừng trước OCR.')
    receipt_path.write_text(json.dumps({'bundle_key': key, 'components': requested}, indent=2), encoding='utf-8')
    config.write_text(json.dumps(session.state, indent=2), encoding='utf-8')
    (session.log_dir.parent / 'runtime_config.json').write_bytes(config.read_bytes())
    print('Packages sẵn sàng.')

def check_hunyuan_api(session):
    if not session.on_kaggle or not session.run_ocr or (not session.options.get('check_hunyuan_import', True)):
        return
    code = "from transformers import AutoProcessor, HunYuanVLForConditionalGeneration; print('Hunyuan API OK')"
    try:
        probe = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=180, env=os.environ.copy())
        check = {'ok': probe.returncode == 0, 'stdout': probe.stdout, 'stderr': probe.stderr}
    except subprocess.TimeoutExpired:
        check = {'ok': False, 'detail': 'Hunyuan import timeout after 180 seconds'}
    log = session.log_dir.parent / 'hunyuan_import_check.json'
    log.write_text(json.dumps(check, ensure_ascii=False, indent=2), encoding='utf-8')
    if not check['ok']:
        raise RuntimeError(f'Import Hunyuan chưa thành công; xem {log}. Đã dừng trước OCR.')

def configure_models(session):
    options = session.options
    common = dict(input_dir=str(session.input_dir), workdir=str(session.work_dir), books=session.books, pages=session.pages, limit=session.limit, n_gpu=1, worker_python=sys.executable, n_layout=0, page_timeout=600, budget_h=24, max_restarts=3, export=True, dpi=200, max_side=1280, max_new_tokens=6000, segmentation='auto')
    common.update(options.get('ocr', {}))
    session.models = {'ppocr': OCRConfig(**common, engine='ppocr', model='PP-OCRv6_medium', run_name='ppocrv6_medium_charboxes'), 'tonggu': OCRConfig(**common, engine='tonggu', model='TongGuOCR-Stage3', run_name='tonggu_stage3'), 'paddle': OCRConfig(**common, engine='paddle', model='PaddleOCR-VL-1.6', paddle_version='v1.6', run_name='paddle_vl16'), 'hunyuan': OCRConfig(**common, engine='hunyuan', model='tencent/HunyuanOCR', run_name='hunyuan15', model_revision='47644ecc4fc854efa4f505155158831f36773ee4')}
    h, p, pp, t = (session.models[name] for name in ('hunyuan', 'paddle', 'ppocr', 'tonggu'))
    for cfg, minimum, maximum in ((h, 262144, 4194304), (p, 200704, 1048576), (t, 1024, 1048576)):
        if 'model_min_pixels' not in options.get('ocr', {}):
            cfg.model_min_pixels = minimum
        if 'model_max_pixels' not in options.get('ocr', {}):
            cfg.model_max_pixels = maximum
    if 'page_timeout' not in options.get('ocr', {}):
        t.page_timeout = 1800
    h.dtype = 'bf16' if session.on_kaggle else 'fp16'
    if session.weights.get('hunyuan'):
        h.model, h.model_revision = (str(session.weights['hunyuan']), '')
    for key, field in (('ppocr_det', 'text_detection_model_dir'), ('ppocr_rec', 'text_recognition_model_dir')):
        if session.weights.get(key):
            pp.paddle_args[field] = str(session.weights[key])
    for key, field in (('paddle_vl', 'vl_rec_model_dir'), ('paddle_layout', 'layout_detection_model_dir')):
        if session.weights.get(key):
            p.paddle_args[field] = str(session.weights[key])
    p.paddle_args.update(use_doc_orientation_classify=False, use_doc_unwarping=False)
    t.tonggu_config = str(session.weights.get('tonggu_config') or options.get('tonggu_config', ''))
    if session.on_kaggle and (not t.tonggu_config) and session.weights.get('tonggu_ocr') and session.weights.get('tonggu_layout'):
        session.work_dir.mkdir(parents=True, exist_ok=True)
        config = session.work_dir / 'tonggu_paths.json'
        config.write_text(json.dumps({'layout_inference': {'detector_checkpoint_path': str(Path(session.weights['tonggu_layout']).resolve())}, 'ocr': {'transformers': {'model_path': str(Path(session.weights['tonggu_ocr']).resolve())}}}, indent=2) + '\n', encoding='utf-8')
        t.tonggu_config = str(config)
    if session.on_kaggle:
        p.worker_python = pp.worker_python = session.state['paddle_python'] if session.run_ocr else session.state.get('paddle_python', sys.executable)
        h.worker_pythonpath = [session.state['common_site']] if session.state.get('common_site') else []
        t.worker_python = session.state.get('tonggu_python', sys.executable)
        t.tonggu_repo = session.state.get('tonggu_repo', '')
    else:
        t.worker_wsl_distro = options.get('tonggu_wsl_distro', 'Ubuntu' if os.name == 'nt' else '')
        t.worker_python = options.get('tonggu_python', '/home/YOUR_WSL_USER/miniconda3/envs/tongguocr/bin/python')
        t.tonggu_repo = options.get('tonggu_repo', '/home/YOUR_WSL_USER/TongGuOCR')
    for name, overrides in options.get('model_overrides', {}).items():
        for field, value in overrides.items():
            if not hasattr(session.models[name], field):
                raise ValueError(f'Unknown OCR setting: {name}.{field}')
            setattr(session.models[name], field, value)
    session.profile_dir = Path(options.get('profile_dir') or NOTEBOOK_PROFILE_DIR)

def setup_session(session):
    if session.on_kaggle:
        os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True', USE_TF='0', USE_FLAX='0', CUDA_VISIBLE_DEVICES=str(session.options.get('gpu', '0')))
    validate_inputs(session)
    prepare_offline_runtime(session)
    activate_pdfium(session)
    check_hunyuan_api(session)
    configure_models(session)
    if session.run_ocr:
        checks = preflight_models(session.models)
        failed = [f"{row['model']}: {row['detail']}" for row in checks if not row['ready']]
        if failed:
            raise RuntimeError('Chưa đủ môi trường cho bốn model:\n' + '\n'.join(failed))
    else:
        check_manifest(session)
    print(f"Setup xong: 4 model; {('OCR' if session.run_ocr else 'đọc checkpoint')}; PAGES={session.pages!r}, LIMIT={session.limit}.")
    return session

def check_manifest(session):
    campaign = json.loads(session.manifest.read_text(encoding='utf-8'))
    if set(campaign.get('models', [])) != set(session.models):
        raise ValueError('Checkpoint phải có đúng bốn model.')
    return campaign

def run_ocr_models(session):
    if session.run_ocr:
        session.manifest = prepare_multi_corpus(session.models, session.profile_dir if session.options.get('use_book_profiles', True) else None, max_books=session.max_books, cache_renders=session.options.get('cache_renders', True))
    campaign = check_manifest(session)
    print(f"{campaign['total_books']} PDF, {campaign['total_pages']} trang/model, 4 model.")
    results, stage_errors = ({}, {})
    if session.run_ocr:
        for name in session.models:
            try:
                results[name] = run_corpus_engine(session.manifest, name)
            except Exception as exc:
                stage_errors[name] = {'error': type(exc).__name__, 'message': str(exc), 'updated': now()}
                results[name] = {**engine_progress(session.manifest, name), 'status': 'error'}
                print(f'{name}: stage lỗi {exc}; giữ checkpoint, chuyển sang model tiếp theo.')
            write_json(session.manifest.parent / 'stage_errors.json', stage_errors)
    return results

def compare_ocr_results(session):
    session.result = compare_multi_corpus(session.manifest, max_alignment_cells=session.options.get('max_alignment_cells', 8000000), review_threshold=session.options.get('review_threshold', 0.05), reading_order=session.options.get('reading_order', 'auto'), page_consensus=session.options.get('page_consensus', True), page_consensus_options=session.options.get('page_consensus_options'))
    session.pages_table, session.conflicts_table = multi_tables(session.result)
    session.review_queue = session.pages_table[session.pages_table['Xử lý'] != 'agreement_only']
    display(session.pages_table.round(4))
    print(f'{len(session.review_queue)} trang cần rà soát; {len(session.conflicts_table)} cột ký tự bất đồng.')
    structural = sum((bool((r.get('page_consensus') or {}).get('findings')) for r in session.result['rows']))
    print(f'{structural} trang có dấu hiệu thiếu/lặp/dư vùng hoặc sai thứ tự đọc; xem cột Dấu hiệu cấp trang.')
    for warning in session.result['warnings']:
        print('Lưu ý:', warning)
    if session.options.get('export_consensus', False):
        write_json(session.manifest.parent / 'character_consensus.json', session.result)
    return session.result

def review_character_conflicts(session, *, book=None, page=None, max_characters=20, view='regions'):
    if session.result is None:
        raise RuntimeError('Chạy bước xem kết quả trước.')
    candidates = [row for row in session.result['rows'] if (book is None or row['book'] == book) and (page is None or row['page'] == page) and (book is not None or page is not None or row['route'] != 'agreement_only')]
    selected = next((row for row in candidates if (row.get('page_consensus') or {}).get('findings')), None)
    selected = selected or next((row for row in candidates if row['consensus']['conflicts']), candidates[0] if candidates else None)
    if selected:
        return show_character_conflicts(session.result, selected['book'], selected['page'], max_characters=max_characters, view=view)
    else:
        print('Không có trang cần xem hoặc lựa chọn không khớp.')

def export_ocr_results(session, *, output_dir=None, include_crops=True, create_zip=True):
    if session.result is None:
        compare_ocr_results(session)
    session.exports = export_corpus_results(session.result, output_dir or session.work_dir / 'output', include_crops=include_crops, create_zip=create_zip, dpi=session.options.get('crop_dpi', 250))
    for artifact in session.exports:
        print(f"{artifact['book']}: {artifact['pages']} trang, {artifact['status']}; {artifact['crop_regions']} vùng ảnh; xuất trong {artifact['export_seconds']:.1f}s.")
        print('HTML:', artifact['index_html'])
        if artifact['zip_path']:
            print('ZIP:', artifact['zip_path'])
    return session.exports

def setup_ocr(**settings):
    session = new_ocr_session(sources={"installer": NOTEBOOK_SETUP_SIGNATURE.encode()}, **settings)
    if session.run_ocr and session.options.get("use_book_profiles", True):
        NOTEBOOK_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
        for name, profile in BOOK_PROFILES.items():
            write_json(NOTEBOOK_PROFILE_DIR / name, profile)
    return setup_session(session)


### Worker chạy riêng

Chỉ ô này dùng `%%writefile`: nội dung là một worker độc lập, không import mã dự án.
Mỗi model chạy trong một tiến trình rồi thoát để nhả GPU. Không chạy ô này là chưa có worker.


In [ ]:
%%writefile {NOTEBOOK_WORKER_FILE}
from __future__ import annotations

import dataclasses
import hashlib
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
HUNYUAN_PROMPT = '提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，表格用html格式表达，文档中公式用latex格式表示，按照阅读顺序组织进行解析。'
LAYOUT_PROMPT = '按照阅读顺序解析图中的版式信息。'

def _json_default(o: Any):
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

def config_hash(cfg: Any, exclude: set[str]) -> str:
    data = {k: v for k, v in dataclasses.asdict(cfg).items() if k not in exclude}
    raw = json.dumps(data, ensure_ascii=False, sort_keys=True, default=_json_default)
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()

@dataclass
class OCRConfig:
    """Cấu hình bước OCR. Mọi trường đều có thể sửa từ notebook."""
    engine: str = 'hunyuan'
    input_dir: str = 'data'
    books: list[str] = field(default_factory=list)
    workdir: str = 'work'
    run_name: str = ''
    pages: str = ''
    dpi: int = 200
    max_side: int = 1280
    segmentation: str = 'none'
    reading_direction: str = 'vertical_rtl'
    segmentation_args: dict = field(default_factory=dict)
    region_max_pixels: int = 0
    region_max_side: int = 0
    region_upscale: float = 1.0
    resolution_strict: bool = False
    model_min_pixels: int | None = None
    model_max_pixels: int | None = None
    image_dataset: str = ''
    image_dataset_key: str = ''
    image_variant: str = 'original'
    image_crop_type: str = 'group'
    image_dataset_fallback: bool = False
    max_new_tokens: int = 4096
    dtype: str = 'fp16'
    model: str = 'tencent/HunyuanOCR'
    model_revision: str = ''
    prompt: str = ''
    repetition_penalty: float = 1.08
    device: str = 'auto'
    paddle_version: str = 'v1.6'
    paddle_args: dict = field(default_factory=dict)
    tonggu_repo: str = ''
    tonggu_config: str = ''
    limit: int | None = None
    page_timeout: int = 300
    n_gpu: int = 0
    worker_python: str = ''
    worker_wsl_distro: str = ''
    worker_pythonpath: list[str] = field(default_factory=list)
    budget_h: float = 10.5
    stall_min: int = 15
    max_restarts: int = 5
    n_layout: int = 0
    layout_budget_h: float = 1.0
    layout_prompt: str = LAYOUT_PROMPT
    export: bool = True
    fresh: bool = False
    resume_from: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        if self.engine not in ('hunyuan', 'paddle', 'ppocr', 'tonggu', 'dummy'):
            raise ValueError(f'engine không hợp lệ: {self.engine}')
        if self.dtype not in ('fp32', 'fp16', 'bf16'):
            raise ValueError(f'dtype không hợp lệ: {self.dtype}')
        if self.dpi <= 0 or self.max_side <= 0 or self.max_new_tokens <= 0:
            raise ValueError('dpi, max_side, max_new_tokens phải > 0')
        if self.segmentation not in {'none', 'auto', 'columns'}:
            raise ValueError('segmentation must be none, auto or columns')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('reading_direction must be vertical_rtl or horizontal_ltr')
        if self.region_max_pixels < 0 or self.region_max_side < 0 or (not 1 <= self.region_upscale <= 4):
            raise ValueError('region caps must be >= 0; region_upscale must be in [1, 4]')
        if any((v is not None and v <= 0 for v in (self.model_min_pixels, self.model_max_pixels))):
            raise ValueError('model pixel budgets must be positive or None')
        if self.model_min_pixels is not None and self.model_max_pixels is not None and (self.model_min_pixels > self.model_max_pixels):
            raise ValueError('model_min_pixels must not exceed model_max_pixels')
        if not isinstance(self.segmentation_args, dict):
            raise ValueError('segmentation_args must be a dictionary')
        if self.image_crop_type not in {'page', 'group', 'column', 'native'}:
            raise ValueError('image_crop_type must be page, group, column or native')
        if self.image_variant not in {'original', 'clahe', 'bicubic_x2'}:
            raise ValueError('Unsupported image_variant')
        if not self.image_dataset and (self.image_variant != 'original' or self.image_dataset_key):
            raise ValueError('Image variants/snapshot identity require image_dataset')

    def cache_key(self) -> str:
        exclude = {'input_dir', 'workdir', 'run_name', 'limit', 'page_timeout', 'n_gpu', 'worker_python', 'worker_wsl_distro', 'worker_pythonpath', 'budget_h', 'stall_min', 'max_restarts', 'n_layout', 'layout_budget_h', 'export', 'fresh', 'resume_from'}
        if not self.image_dataset:
            exclude.update({'image_dataset', 'image_dataset_key', 'image_variant', 'image_crop_type', 'image_dataset_fallback'})
        return config_hash(self, exclude)

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)


import unicodedata

def comparison_text(text: str, mode: str='visible') -> str:
    if mode not in {'han', 'visible'}:
        raise ValueError('mode must be han or visible')
    if mode == 'visible':
        return ''.join((c for c in text if not c.isspace()))
    return ''.join((c for c in text if ord(c) >= 13312 and unicodedata.name(c, '').startswith(('CJK UNIFIED IDEOGRAPH-', 'CJK COMPATIBILITY IDEOGRAPH-'))))


from bisect import bisect_right
from math import isfinite
from statistics import median

def line_rectangle(polygon):
    if not isinstance(polygon, (list, tuple)):
        return None
    if len(polygon) == 4 and all((isinstance(v, (int, float)) for v in polygon)):
        x1, y1, x2, y2 = polygon
    else:
        if len(polygon) < 4 or any((not isinstance(p, (list, tuple)) or len(p) != 2 for p in polygon)):
            return None
        if any((not isinstance(v, (int, float)) or not isfinite(v) for p in polygon for v in p)):
            return None
        x1, y1 = (min((p[0] for p in polygon)), min((p[1] for p in polygon)))
        x2, y2 = (max((p[0] for p in polygon)), max((p[1] for p in polygon)))
    if not all((isfinite(v) for v in (x1, y1, x2, y2))) or x1 >= x2 or y1 >= y2:
        return None
    return (x1, y1, x2, y2)

def order_ppocr_columns(raw, *, mode='auto'):
    """Return comparison geometry; the original checkpoint remains untouched.

    Auto requires at least two tall detections and 70% of detected characters
    in tall detections. Unknown/missing geometry keeps detection order. Within
    a shared column, fragments are read top to bottom. Every character offset
    is moved with its original box, even when the same glyph occurs repeatedly.
    """
    if mode not in {'auto', 'vertical_rtl', 'detection'}:
        raise ValueError('reading_order must be auto, vertical_rtl or detection')
    info = {'mode': 'detection', 'changed': False}
    lines = raw.get('text_lines', [])
    if mode == 'detection' or not lines:
        return (raw, info)
    texts = [comparison_text(line.get('text', '')) for line in lines]
    if ''.join(texts) != comparison_text(raw.get('text', '')):
        return (raw, {**info, 'reason': 'line_text_mismatch'})
    boxes = [line_rectangle(line.get('polygon')) for line in lines]
    if any((box is None for box in boxes)):
        return (raw, {**info, 'reason': 'incomplete_line_geometry'})
    vertical = [i for i, (x1, y1, x2, y2) in enumerate(boxes) if y2 - y1 >= 1.5 * (x2 - x1)]
    total = sum(map(len, texts))
    if mode == 'auto' and (len(vertical) < 2 or not total or sum((len(texts[i]) for i in vertical)) < 0.7 * total):
        return (raw, info)
    tolerance = 0.45 * median((box[2] - box[0] for box in boxes))
    groups = []
    for index in sorted(range(len(lines)), key=lambda i: -(boxes[i][0] + boxes[i][2]) / 2):
        center = (boxes[index][0] + boxes[index][2]) / 2
        if groups and abs(groups[-1][0] - center) <= tolerance:
            groups[-1][1].append(index)
        else:
            groups.append((center, [index]))
    order = [index for _, group in groups for index in sorted(group, key=lambda i: boxes[i][1])]
    starts, offset = ([], 0)
    for text in texts:
        starts.append(offset)
        offset += len(text)
    new_starts, offset = ({}, 0)
    for index in order:
        new_starts[index] = offset
        offset += len(texts[index])
    characters = []
    for entry in raw.get('character_boxes', []):
        old = entry.get('offset')
        if not isinstance(old, int) or old < 0 or old >= total:
            continue
        index = bisect_right(starts, old) - 1
        local = old - starts[index]
        if texts[index][local:local + 1] == entry.get('text'):
            characters.append({**entry, 'offset': new_starts[index] + local})
    geometry = {**raw, 'text': '\n'.join((lines[i]['text'] for i in order)), 'text_lines': [lines[i] for i in order], 'character_boxes': sorted(characters, key=lambda entry: entry['offset'])}
    return (geometry, {'mode': 'vertical_rtl', 'changed': order != list(range(len(lines))), 'line_order': order})


from collections import Counter
import re
from statistics import median

def ocr_quality_issues(text, raw):
    issues = list(raw.get('quality_issues', []))
    if raw.get('hit_max'):
        issues.append('token_limit')
    visible = comparison_text(text)
    if re.search('(.{1,40}?)\\1{11,}', visible):
        match = max((m.end() - m.start() for m in re.finditer('(.{1,40}?)\\1{11,}', visible)), default=0)
        if match >= 80:
            issues.append('repetitive_output')
    if len(visible) >= 80:
        lines = [comparison_text(line) for line in text.splitlines()]
        signatures = [re.sub('[0-9０-９]+', '0', line) for line in lines if len(line) >= 12]
        if signatures and max(Counter(signatures).values()) >= 6:
            issues.append('repetitive_output')
        symbols = sum((not c.isalnum() for c in visible))
        if symbols >= 80 and symbols / len(visible) > 0.5:
            issues.append('symbol_dominated_output')
    return sorted(set(issues))

def page_quality_issues(records):
    issues = {name: list(record.get('quality_issues', [])) for name, record in records.items() if record is not None}
    lengths = {name: record['visible'] for name, record in records.items() if record is not None and (not issues[name]) and (record['visible'] > 0)}
    for name, length in lengths.items():
        peers = [value for other, value in lengths.items() if other != name]
        if len(peers) < 2 or max(peers) > 1.6 * min(peers):
            continue
        typical = median(peers)
        if typical >= 80 and length < 0.35 * typical:
            issues[name].append('suspected_incomplete_output')
        elif length > max(3 * typical, typical + 200):
            issues[name].append('length_outlier')
    return {name: sorted(set(reasons)) for name, reasons in issues.items() if reasons}


from math import floor, isfinite, sqrt
from pathlib import Path

def image_budget(cfg):
    defaults = {'hunyuan': 4194304, 'paddle': 1048576, 'tonggu': 1048576, 'ppocr': 4194304, 'dummy': 4194304}
    pixels = cfg.region_max_pixels or defaults[cfg.engine]
    if cfg.model_max_pixels and cfg.engine == 'hunyuan':
        pixels = min(pixels, cfg.model_max_pixels)
    return (pixels, cfg.region_max_side)

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert('L')
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get('ink_threshold', 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    left, right = (max(1, int(w * 0.03)), min(w - 1, int(w * 0.97)))
    top, bottom = (max(1, int(h * 0.03)), min(h - 1, int(h * 0.97)))
    if left >= right or top >= bottom:
        return []
    rows = [sum((v > 0 for v in data[y * w + left:y * w + right])) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < 0.7 * (right - left)]
    if not valid_rows:
        return []
    cols = [sum((data[y * w + x] > 0 for y in valid_rows)) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < 0.7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == 'x':
        values = [sum((data[y * w + x] > 0 for y in valid_rows)) / len(valid_rows) for x in range(w)]
        extent, length = (image.width, w)
    else:
        values = [sum((data[y * w + x] > 0 for x in valid_cols)) / len(valid_cols) for y in range(h)]
        extent, length = (image.height, h)
    values = [0.0 if value >= 0.7 else value for value in values]
    threshold = options.get('gap_ink_fraction', 0.006)
    minimum = max(2, round(length * options.get('min_gap_fraction', 0.006)))
    edge = max(minimum, round(length * 0.06))
    gaps, start = ([], None)
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                span = max(minimum * 3, round(length * 0.035))
                before = values[max(edge, start - span):start]
                after = values[i:min(length - edge, i + span)]
                support = max(3, round(length * 0.006))
                if sum((v > threshold * 2 for v in before)) >= support and sum((v > threshold * 2 for v in after)) >= support:
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2 - x1 if axis == 'x' else y2 - y1]
    result = [(x1 + a, y1, x1 + b, y2) for a, b in zip(ends, ends[1:])] if axis == 'x' else [(x1, y1 + a, x2, y1 + b) for a, b in zip(ends, ends[1:])]
    return result[::-1] if axis == 'x' and direction == 'vertical_rtl' else result

def plan_regions(image, cfg, *, page=0):
    """Return a disjoint, complete partition of the rendered page in read order."""
    cfg.__post_init__()
    full = (0, 0, image.width, image.height)
    info = {'method': 'whole_page', 'warnings': [], 'coverage': 1.0, 'overlap': False}
    if cfg.segmentation == 'none':
        return ([full], info)
    options = cfg.segmentation_args
    allowed = {'column_cuts', 'page_column_cuts', 'columns_per_region', 'max_regions', 'ink_threshold', 'gap_ink_fraction', 'min_gap_fraction'}
    if set(options) - allowed:
        raise ValueError(f'Unknown segmentation options: {sorted(set(options) - allowed)}')
    cuts = options.get('page_column_cuts', {}).get(str(page + 1), options.get('column_cuts'))
    native = cfg.segmentation == 'auto' and cfg.engine in {'paddle', 'tonggu'} and (cuts is None)
    if native:
        return ([full], {**info, 'method': 'native_layout', 'warnings': []})
    max_regions = int(options.get('max_regions', 64))
    group = int(options.get('columns_per_region', 3))
    if max_regions < 1 or group < 1:
        raise ValueError('max_regions and columns_per_region must be >= 1')
    boxes = [full]
    if cuts is not None:
        if not isinstance(cuts, list) or any((not isinstance(v, (int, float)) or not isfinite(v) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts)):
            raise ValueError('column_cuts must be unique increasing fractions in (0, 1)')
        pixel_cuts = [round(x * image.width) for x in cuts]
        if len(set([0, *pixel_cuts, image.width])) != len(cuts) + 2:
            raise ValueError('column_cuts collapse at this image resolution')
        boxes = _partition_box(full, pixel_cuts, 'x', cfg.reading_direction)
        info['method'] = 'manual_columns'
    elif cfg.segmentation == 'columns' or (cfg.engine == 'hunyuan' and cfg.reading_direction == 'vertical_rtl'):
        found = projection_gaps(image, 'x', options)
        if cfg.reading_direction == 'vertical_rtl':
            selected = list(reversed(list(reversed(found))[group - 1::group]))
        else:
            selected = found[group - 1::group]
        if cfg.segmentation == 'auto' and len(found) < 2 * group - 1:
            selected = []
            info['warnings'].append('too_few_gutters_for_context_groups; kept whole page')
        boxes = _partition_box(full, selected, 'x', cfg.reading_direction)
        info.update(method='whitespace_columns', detected_gutters=found)
        info['warnings'].append('heuristic_layout; preview cuts on text/annotation pages before OCR')
        if not found:
            info['warnings'].append('no_confident_column_gutter; kept whole page')
    pixels, side = image_budget(cfg)

    def too_big(box):
        w, h = (box[2] - box[0], box[3] - box[1])
        return w * h > pixels or (side > 0 and max(w, h) > side)
    pending, boxes = (boxes, [])
    while pending:
        box = pending.pop(0)
        if too_big(box):
            crop = image.crop(box)
            axes = (['x', 'y'] if crop.height >= 4 * crop.width else ['x']) if cfg.reading_direction == 'vertical_rtl' else ['y', 'x']
            children = None
            for axis in axes:
                candidates = projection_gaps(crop, axis, options)
                size = crop.width if axis == 'x' else crop.height
                candidates = [c for c in candidates if 0.15 * size <= c <= 0.85 * size]
                if candidates:
                    cut = min(candidates, key=lambda x: abs(x - size / 2))
                    children = _partition_box(box, [cut], axis, cfg.reading_direction)
                    break
            if children and len(boxes) + len(pending) + 2 <= max_regions:
                pending[0:0] = children
                continue
            info['warnings'].append('unsplittable_region_exceeds_budget')
        boxes.append(box)
    if len(boxes) > max_regions:
        raise ValueError('Region plan exceeds max_regions; increase columns_per_region')
    info.update(regions=len(boxes), coverage=1.0, overlap=False)
    return (boxes, info)

def prepare_region(image, box, cfg, *, native=False):
    from PIL import Image
    crop = image.crop(box)
    original = crop.size
    pixels, side = image_budget(cfg)
    if native:
        return (crop, {'crop_size': list(original), 'input_size': list(original), 'resized': False, 'scale_xy': [1.0, 1.0], 'budget_scope': 'native_blocks'})
    scale = min(cfg.region_upscale, sqrt(pixels / (crop.width * crop.height)))
    if side:
        scale = min(scale, side / max(crop.size))
    if cfg.resolution_strict and scale < 1:
        raise ValueError(f'Cannot preserve region {box}: {crop.size} exceeds pixel/side budget')
    target = (max(1, floor(crop.width * scale)), max(1, floor(crop.height * scale)))
    if target != original:
        crop = crop.resize(target, Image.Resampling.LANCZOS)
    return (crop, {'crop_size': list(original), 'input_size': list(target), 'resized': target != original, 'downsampled': target[0] < original[0] or target[1] < original[1], 'scale_xy': [target[0] / original[0], target[1] / original[1]], 'max_pixels': pixels, 'max_side': side, 'budget_scope': 'backend_input'})

def _page_geometry(extra, box, input_size, offset, region_index):
    """Map genuine crop detections back to the page; preserve their provenance."""
    from copy import deepcopy
    extra = deepcopy(extra)
    if 'blocks' in extra:
        extra['blocks'] = ordered_page_blocks(extra['blocks'])
    sx, sy = ((box[2] - box[0]) / input_size[0], (box[3] - box[1]) / input_size[1])

    def transform(points):
        if not isinstance(points, (list, tuple)):
            return points
        if len(points) == 4 and all((isinstance(v, (int, float)) for v in points)):
            return [points[0] * sx + box[0], points[1] * sy + box[1], points[2] * sx + box[0], points[3] * sy + box[1]]
        return [[x * sx + box[0], y * sy + box[1]] for x, y in points]
    for entry in extra.get('character_boxes', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['offset'] += offset
        entry['region_index'] = region_index
    for entry in extra.get('text_lines', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['region_index'] = region_index
    for field in ('blocks', 'boxes'):
        for entry in extra.get(field, []):
            entry['box'] = transform(entry.get('box'))
            entry['region_index'] = region_index
    return extra

def run_prepared_page(backend, image, image_path, cfg, *, page=0):
    """Run all regions transactionally; partial region text never becomes a page."""
    import tempfile
    if cfg.segmentation == 'none' and (not image.info.get('prepared_dataset')):
        return backend.run(str(image_path))
    with tempfile.TemporaryDirectory(prefix='vtm-regions-') as directory:
        return _run_prepared_regions(backend, image, cfg, page=page, directory=Path(directory))

def _run_prepared_regions(backend, image, cfg, *, page, directory):
    prepared = image.info.get('prepared_dataset')
    if prepared:
        boxes = [tuple(r['source_box']) for r in prepared['regions']]
        layout = {**prepared['layout'], 'warnings': list(prepared['layout']['warnings']), 'method': 'prepared_' + prepared['crop_type']}
    else:
        boxes, layout = plan_regions(image, cfg, page=page)
    if image.info.get('render', {}).get('downsampled'):
        layout['warnings'].append('page_render_downsampled_before_segmentation; raise max_side')
        if cfg.resolution_strict:
            raise ValueError('Page render was downsampled before segmentation; raise max_side')
    records, texts, merged, offset = ([], [], {}, 0)
    native = prepared['crop_type'] == 'native' and cfg.engine in {'paddle', 'tonggu'} if prepared else layout['method'] == 'native_layout'
    for i, box in enumerate(boxes):
        if prepared:
            entry = prepared['regions'][i]['images'][prepared['variant']]
            variant = dataset_load_image(prepared['root'], entry)
            crop, meta = prepare_region(variant, (0, 0, variant.width, variant.height), cfg, native=native)
            meta.update(variant_scale_xy=meta['scale_xy'], variant_downsampled=meta.get('downsampled', False), resized_from_variant=meta['resized'])
            meta.update(crop_size=[box[2] - box[0], box[3] - box[1]], variant_size=list(variant.size), scale_xy=[crop.width / (box[2] - box[0]), crop.height / (box[3] - box[1])], downsampled=crop.width < box[2] - box[0] or crop.height < box[3] - box[1], resized=crop.size != (box[2] - box[0], box[3] - box[1]), variant=prepared['variant'], variant_sha256=entry['sha256'], operation=entry['operation'], canonical_region_id=prepared['regions'][i]['id'], input_to_source=[(box[2] - box[0]) / crop.width, 0, 0, (box[3] - box[1]) / crop.height, box[0], box[1]])
        else:
            crop, meta = prepare_region(image, box, cfg, native=native)
        path = directory / f'region_{i:03d}.png'
        crop.save(path, format='PNG')
        try:
            text, extra = backend.run(str(path))
        finally:
            path.unlink(missing_ok=True)
        observed = extra.get('model_input', {}).get('processed_sizes_wh', [])
        if len(observed) == 1:
            actual_w, actual_h = observed[0]
            meta['processor_size'] = [actual_w, actual_h]
            meta['processor_scale_xy'] = [actual_w / crop.width, actual_h / crop.height]
            meta['processor_downsampled'] = actual_w < crop.width or actual_h < crop.height
            if meta['processor_downsampled']:
                layout['warnings'].append(f'region_{i}: processor_downsampled (includes patch-grid rounding)')
                if cfg.resolution_strict:
                    raise ValueError(f'Processor downsampled region {i}: {crop.size} -> {observed[0]}')
        if cfg.engine == 'ppocr':
            ordered, order_info = order_ppocr_columns({'text': text, **extra}, mode=('auto' if cfg.segmentation == 'auto' else 'vertical_rtl') if cfg.reading_direction == 'vertical_rtl' else 'detection')
            text = ordered.pop('text')
            extra = {**ordered, 'region_reading_order': order_info}
        records.append({'index': i, 'page_box': list(box), **meta, 'text': text, 'raw': extra})
        mapped = _page_geometry(extra, box, crop.size, offset, i)
        for field in ('character_boxes', 'text_lines', 'blocks', 'boxes'):
            if field in mapped:
                merged.setdefault(field, []).extend(mapped[field])
        offset += len(comparison_text(text))
        texts.append(text)
    for i, block in enumerate(merged.get('blocks', [])):
        block['order'] = i
    issues = sorted({issue for rec in records for issue in rec['raw'].get('quality_issues', [])})
    hit_flags = [rec['raw'].get('hit_max') for rec in records]
    hit_max = True if any((v is True for v in hit_flags)) else None if any((v is None for v in hit_flags)) else False
    if any((rec.get('downsampled') for rec in records)):
        layout['warnings'].append('backend_input_downsampled; inspect regions or enable resolution_strict')
    if any((rec.get('variant_downsampled') for rec in records)):
        layout['warnings'].append('backend_variant_downsampled; inspect selected variant and input budget')
    observation = {'page_size': list(image.size), 'render': image.info.get('render', {}), 'layout': layout, 'regions': records, 'model_min_pixels': cfg.model_min_pixels, 'model_max_pixels': cfg.model_max_pixels, 'processor_observation': 'see region raw.model_input; native internal tensors may be unavailable'}
    if prepared:
        observation['dataset'] = {k: prepared[k] for k in ('snapshot_id', 'source_sha256', 'source', 'variant', 'crop_type', 'acquisition')}
    return ('\n'.join(texts), {**merged, 'hit_max': hit_max, 'quality_issues': issues, 'coordinate_frame': 'input_image', 'image_preparation': observation, 'reading_direction': cfg.reading_direction, 'region_order_applied': True})

def preview_preparation(src, page, cfg, *, output=None):
    """Model-free preview using 1-based PDF page numbers; never starts OCR."""
    from PIL import ImageDraw
    image = render(str(src), page - 1, cfg.dpi, cfg.max_side)
    boxes, layout = plan_regions(image, cfg, page=page - 1)
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    rows = []
    for i, box in enumerate(boxes):
        _, meta = prepare_region(image, box, cfg, native=layout['method'] == 'native_layout')
        draw.rectangle((box[0], box[1], box[2] - 1, box[3] - 1), outline='red', width=max(1, image.width // 600))
        draw.text((box[0] + 3, box[1] + 3), str(i + 1), fill='red')
        rows.append({'region': i + 1, 'page_box': list(box), **meta})
    if output:
        Path(output).parent.mkdir(parents=True, exist_ok=True)
        preview.save(output)
    return (preview, {'source': str(src), 'page': page, 'page_size': list(image.size), 'render': image.info.get('render', {}), 'engine': cfg.engine, 'layout': layout, 'regions': rows})


import math
PDF_IMAGE_VERSION = 'pdfium-visible-v1'

def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t - b) / w, (r - l) / h, 0, l, b]
    if rotation == 180:
        return [-(r - l) / w, 0, 0, (t - b) / h, r, b]
    if rotation == 270:
        return [0, -(t - b) / w, -(r - l) / h, 0, r, t]
    return [(r - l) / w, 0, 0, -(t - b) / h, l, t]

def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w * scale), math.ceil(h * scale)]
    if min(size) < 1 or size[0] * size[1] > max_pixels:
        raise ValueError(f'Page render {size} exceeds max_source_pixels={max_pixels}; review acquisition settings (source is never silently reduced)')
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert('RGB')
    finally:
        bitmap.close()

def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100000000, strategy='auto'):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops
    if strategy not in {'auto', 'render'} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError('Invalid PDF acquisition options')
    width, height = page.get_size()
    bbox, rotation = (list(page.get_bbox()), page.get_rotation())
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = ([], 0.0)
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x, y) for x, y in ((0, 0), (1, 0), (1, 1), (0, 1))]
        bounds = [min((c[0] for c in corners)), min((c[1] for c in corners)), max((c[0] for c in corners)), max((c[1] for c in corners))]
        full = max(0, min(bbox[2], bounds[2]) - max(bbox[0], bounds[0])) * max(0, min(bbox[3], bounds[3]) - max(bbox[1], bounds[1])) >= 0.8 * (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        native = max(pixels[0] / axes[0], pixels[1] / axes[1]) if min(axes) > 0 else 0
        density = max(density, native)
        images.append({'native_size': list(pixels), 'bounds_pdf': bounds, 'matrix_pdf': [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f], 'level': obj.level, 'full_page_scan': full, 'pixels_per_72_canvas_units': 72 * native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append('not_one_top_level_image; preserve_composition')
    if rotation:
        reasons.append('page_rotation')
    if annotations != 0:
        reasons.append('page_annotations_or_unknown')
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if obj.level or abs(m.b) > 1e-06 or abs(m.c) > 1e-06 or (m.a <= 0) or (m.d <= 0) or any((abs(a - v) > 0.001 for a, v in zip(obj.get_bounds(), bbox))) or any((abs(a - v) > 0.001 for a, v in zip([m.e, m.f, m.a, m.d], [l, b, r - l, t - b]))):
            reasons.append('placement_crop_rotation_or_reflection')
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append('image_clip_path')
    image, method, validation = (None, 'pdf_render', None)
    if strategy == 'auto' and (not reasons):
        nw, nh = obj.get_px_size()
        if nw * nh > max_source_pixels:
            raise ValueError('Native image exceeds max_source_pixels')
        scale = math.nextafter(min(nw / width, nh / height), 0)
        if abs(nw / width - nh / height) > 0.0001 * max(nw / width, nh / height):
            reasons.append('anisotropic_native_geometry')
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert('RGB')
            finally:
                if hasattr(bitmap, 'warn_on_close'):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {'method': 'full_pixel_equality_to_visible_render', 'equal': equal, 'render_size': list(visible.size), 'scale': scale}
            if equal:
                image, method = (candidate, 'native_validated')
            else:
                reasons.append('native_pixels_differ_from_visible_page')
    if image is None:
        raster_only = bool(images) and all((o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM} for o in objects)) and (annotations == 0)
        scale = density if raster_only and density else max(dpi / 72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation['scale']
    meta = {'version': PDF_IMAGE_VERSION, 'method': method, 'size': list(image.size), 'page_size_pdf': [width, height], 'page_bbox_pdf': bbox, 'media_box_pdf': list(page.get_mediabox()), 'crop_box_pdf': list(page.get_cropbox()), 'rotation': rotation, 'object_types': [o.type for o in objects], 'annotations': annotations, 'embedded_images': images, 'requested_dpi': dpi, 'render_scale': scale, 'native_validation': validation, 'fallback_reasons': reasons, 'source_downsampled': False, 'source_to_pdf': pdf_pixel_to_page(image.size, bbox, rotation), 'resolution_unit': 'pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable', 'warnings': ['low_native_resolution; interpolation_cannot_recover_detail'] if images and max((max(i['native_size']) for i in images)) <= 750 else []}
    return (image, meta)


from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import time
IMAGE_DATASET_FORMAT = 'vtm-image-dataset'
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = ''

@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = 'auto'
    max_source_pixels: int = 100000000
    analysis_max_side: int = 1200
    reading_direction: str = 'vertical_rtl'
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = 0.006
    min_gap_fraction: float = 0.006
    variants: list[str] = field(default_factory=lambda: ['original'])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.dpi <= 0 or self.max_source_pixels < 1 or (not 32 <= self.analysis_max_side <= 1600) or (self.columns_per_group < 1) or (self.max_columns < 1):
            raise ValueError('Invalid preparation dimensions/group limits')
        if self.acquisition not in {'auto', 'render'}:
            raise ValueError('acquisition must be auto or render')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('Invalid reading_direction')
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or (not 0 < self.min_gap_fraction < 0.5):
            raise ValueError('Invalid projection thresholds')
        if not isinstance(self.variants, list) or 'original' not in self.variants or len(set(self.variants)) != len(self.variants) or set(self.variants) - {'original', 'bicubic_x2', 'clahe'}:
            raise ValueError('variants require original; supported experiments: bicubic_x2, clahe. Denoising/SR need separate evidence and are not implemented')
        if set(self.variant_options) - {'clahe'}:
            raise ValueError('Only clahe accepts variant_options')
        for name, options in self.variant_options.items():
            if set(options) - {'clip_limit', 'tile_grid'}:
                raise ValueError(f'Unknown {name} parameters')
            if not 0 < options.get('clip_limit', 1.5) <= 4 or not 2 <= options.get('tile_grid', 8) <= 32:
                raise ValueError('CLAHE clip_limit must be (0,4], tile_grid [2,32]')
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError('page_column_cuts uses 1-based page keys')
            dataset_validate_cuts(cuts)

def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')

def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()

def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root / name) for name in ('image_dataset.py', 'image_preparation.py', 'pdf_images.py')})

def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {'pillow': PIL.__version__, 'pypdfium2': str(pdfium.PYPDFIUM_INFO), 'pdfium': str(pdfium.PDFIUM_INFO), 'python': sys.version.split()[0]}
    return result

def dataset_path(root, relative):
    relative = str(relative)
    if '\\' in relative or ':' in relative:
        raise ValueError(f'Dataset requires portable relative paths: {relative}')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or '..' in parts.parts or (not parts.parts):
        raise ValueError(f'Invalid dataset path: {relative}')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Dataset path escapes root: {relative}')
    return path

def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_bytes(data)
    temporary.replace(path)
    return {'path': str(relative), 'sha256': hashlib.sha256(data).hexdigest()}

def dataset_read(root, reference):
    data = dataset_path(root, reference['path']).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference['sha256']:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data

def dataset_store_image(root, image):
    image = image.convert('RGB')
    output = io.BytesIO()
    image.save(output, format='PNG', compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': hashlib.sha256(image.tobytes()).hexdigest()}

def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert('RGB')
    if list(image.size) != reference['size'] or hashlib.sha256(image.tobytes()).hexdigest() != reference['pixel_sha256']:
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image

@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root) / 'prepare.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            if os.name == 'nt':
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream, fcntl.LOCK_UN)

def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts))):
        raise ValueError('column_cuts must be unique increasing fractions in (0,1)')

def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == 'horizontal_ltr' else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region['source_box']
        if not all((isinstance(v, int) for v in (x1, y1, x2, y2))) or not 0 <= x1 < x2 <= w or (y1, y2) != (0, h) or (region['reading_order'] != index):
            raise ValueError('Invalid canonical crop bounds/order')
        start, end = (x1, x2) if direction == 'horizontal_ltr' else (x2, x1)
        if start != expected:
            raise ValueError('Canonical partition has a gap, overlap or inconsistent reading order')
        expected = end
    if not regions or expected != (w if direction == 'horizontal_ltr' else 0):
        raise ValueError('Canonical partition omits page content')

def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ('ink_threshold', 'gap_ink_fraction', 'min_gap_fraction')}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page + 1), config.column_cuts)
    if manual is not None:
        cuts = [round(v * w) for v in manual]
        method = 'manual_columns'
    elif config.reading_direction == 'vertical_rtl':
        cuts = sorted(set((round(v * w / aw) for v in projection_gaps(analysis, 'x', options))))
        method = 'whitespace_projection'
    else:
        cuts, method = ([], 'whole_page_horizontal')
    if len(set([0, *cuts, w])) != len(cuts) + 2:
        raise ValueError('Column cuts collapse in source space')
    warnings = ['heuristic_layout; geometric_coverage_is_not_text_coverage']
    if len(cuts) + 1 > config.max_columns:
        if manual is not None:
            raise ValueError('Manual layout exceeds max_columns')
        warnings.append('too_many_gutters; retained_whole_page')
        cuts = []
    if not cuts:
        warnings.append('no_partition; whole_page_retained')
    thresholds = sorted(set([max(1, config.ink_threshold - 8), config.ink_threshold, min(254, config.ink_threshold + 8)]))
    found = [projection_gaps(analysis, 'x', {**options, 'ink_threshold': t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = (int(0.03 * ah), max(1, int(0.97 * ah)))
    projection = [sum((ink[y * aw + x] > 0 for y in range(top, bottom))) / (bottom - top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw - 1, max(0, round(cut * aw / w)))
        lo, hi = (x, x)
        while lo > 0 and projection[lo - 1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw - 1 and projection[hi + 1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2, round(aw * 0.006))
        stability = sum((any((abs(c - x) <= tolerance for c in cs)) for cs in found)) / len(found)
        radius = max(2, round(aw * 0.035))
        left, right = (projection[max(0, x - radius):x], projection[x + 1:min(aw, x + radius + 1)])
        risks = []
        if projection[x] > 0.02:
            risks.append('cut_crosses_analysis_ink; rules_or_strokes_need_review')
        if stability < 2 / 3:
            risks.append('unstable_across_ink_thresholds')
        if not left or not right or min(max(left), max(right)) <= config.gap_ink_fraction * 2:
            risks.append('weak_text_support_on_one_side')
        evidence.append({'source_x': cut, 'analysis_x': x, 'gutter_width_analysis': hi - lo + 1, 'ink_fraction': projection[x], 'threshold_stability': stability, 'warnings': risks})
    full = (0, 0, w, h)
    columns = _partition_box(full, cuts, 'x', config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0, len(columns), group):
        chunk = columns[start:start + group]
        groups.append((min((b[0] for b in chunk)), 0, max((b[2] for b in chunk)), h))
    regions = []
    for kind, boxes in (('page', [full]), ('group', groups), ('column', columns)):
        for index, box in enumerate(boxes):
            regions.append({'id': f'{kind}_{index:03d}', 'crop_type': kind, 'reading_order': index, 'source_box': list(box), 'overlap': False})
        dataset_partition_qc([r for r in regions if r['crop_type'] == kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r['crop_type'] == 'column']:
        box = region['source_box']
        x1, x2 = (round(box[0] * aw / w), round(box[2] * aw / w))
        draw.rectangle((x1, 0, max(x1, x2 - 1), ah - 1), outline='red', width=1)
        draw.text((x1 + 2, 3), str(region['reading_order'] + 1), fill='red')
    for item in evidence:
        if item['warnings']:
            draw.line((item['analysis_x'], 0, item['analysis_x'], ah - 1), fill='orange', width=2)
    draw.text((3, ah - 14), 'RTL <-' if config.reading_direction == 'vertical_rtl' else 'LTR ->', fill='red')
    layout = {'method': method, 'source_size': [w, h], 'analysis_size': [aw, ah], 'analysis_to_source': [w / aw, 0, 0, h / ah, 0, 0], 'reading_direction': config.reading_direction, 'regions': regions, 'cuts': evidence, 'excluded_regions': [], 'uncertain_regions': [r['id'] for r in regions if r['crop_type'] == 'column'], 'geometric_coverage': 1.0, 'text_coverage': 'unverified', 'warnings': warnings, 'thresholds_tested': thresholds, 'review_status': 'needs_visual_review'}
    return (layout, mask.convert('RGB'), preview)

def dataset_variant(image, name, options):
    from PIL import Image
    if name == 'original':
        return (image, {'method': 'identity', 'parameters': {}, 'warnings': []})
    if name == 'bicubic_x2':
        return (image.resize((image.width * 2, image.height * 2), Image.Resampling.BICUBIC), {'method': 'bicubic', 'parameters': {'scale': 2}, 'warnings': ['experimental_interpolation; no_recovered_source_detail']})
    if name == 'clahe':
        try:
            import cv2
            import numpy as np
        except ImportError as exc:
            raise RuntimeError('clahe requires an already available OpenCV/NumPy installation; use original offline') from exc
        params = {'clip_limit': 1.5, 'tile_grid': 8, **options}
        lab = cv2.cvtColor(np.asarray(image.convert('RGB')), cv2.COLOR_RGB2LAB)
        grid = int(params['tile_grid'])
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=float(params['clip_limit']), tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        return (Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)), {'method': 'clahe_lab_l', 'parameters': params, 'opencv': cv2.__version__, 'warnings': ['experimental_contrast; inspect_weak_strokes']})
    raise ValueError(f'Unsupported image variant: {name}')

def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root, record['source']) if check_pixels else None
    acquisition = json.loads(dataset_read(root, record['acquisition']))
    layout = json.loads(dataset_read(root, record['layout']))
    if record['source']['size'] != layout['source_size'] or acquisition['size'] != layout['source_size']:
        raise ValueError('Source, acquisition and layout size mismatch')
    dataset_read(root, layout['analysis_mask'])
    dataset_read(root, layout['preview'])
    for kind in ('page', 'group', 'column'):
        regions = [r for r in layout['regions'] if r['crop_type'] == kind]
        dataset_partition_qc(regions, layout['source_size'], layout['reading_direction'])
    if [r['id'] for r in record['regions']] != [r['id'] for r in layout['regions']]:
        raise ValueError('Image dataset omits canonical regions')
    for region, geometry in zip(record['regions'], layout['regions']):
        if any((region[k] != geometry[k] for k in ('source_box', 'crop_type', 'reading_order'))):
            raise ValueError('Region geometry differs from canonical layout')
        if 'original' not in region['images']:
            raise ValueError('Region original variant is missing')
        x1, y1, x2, y2 = region['source_box']
        for name, entry in region['images'].items():
            expected = [(x2 - x1) * (2 if name == 'bicubic_x2' else 1), (y2 - y1) * (2 if name == 'bicubic_x2' else 1)]
            if entry['size'] != expected or entry['image_to_source'] != [(x2 - x1) / expected[0], 0, 0, (y2 - y1) / expected[1], x1, y1]:
                raise ValueError('Variant geometry/transform mismatch')
            if check_pixels:
                image = dataset_load_image(root, entry)
                if name == 'original' and image.tobytes() != source.crop(region['source_box']).tobytes():
                    raise ValueError('Original crop is not pixel-identical to source')
    return layout

def dataset_open(directory):
    path = runtime_path(directory)
    descriptor = path / 'dataset.json' if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name == 'snapshots' else descriptor.parent
    data = json.loads(descriptor.read_text(encoding='utf-8'))
    if data.get('format') != IMAGE_DATASET_FORMAT or data.get('version') != IMAGE_DATASET_VERSION:
        raise ValueError('Unsupported image dataset format/version')
    body = {k: v for k, v in data.items() if k != 'snapshot_id'}
    if dataset_hash(body) != data['snapshot_id']:
        raise ValueError('Dataset snapshot identity mismatch')
    return (root, data)

def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor['pages'] if p['source_sha256'] == source_sha and p['page'] == page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root, e['record'])) for e in entries]
    if len({r['input_key'] for r in records}) != 1:
        raise ValueError('Ambiguous prepared layouts for the same source/page')
    record = records[0]
    if record['source_sha256'] != source_sha or record['page'] != page:
        raise ValueError('Dataset page identity mismatch')
    return record

def bind_image_dataset(cfg, source_sha, pages):
    """Bind an immutable snapshot before looking up OCR checkpoints."""
    if not cfg.image_dataset:
        return
    root, descriptor = dataset_open(cfg.image_dataset)
    snapshot_path = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot_path.is_file():
        raise ValueError('Dataset immutable snapshot file is missing')
    cfg.image_dataset = str(runtime_path(snapshot_path).resolve())
    cfg.image_dataset_key = descriptor['snapshot_id']
    for page in pages:
        record = dataset_page(root, descriptor, source_sha, page)
        if record is None:
            if cfg.image_dataset_fallback:
                continue
            raise ValueError(f'Prepared dataset missing source {source_sha}, page {page + 1}')
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ; choose a matching preparation profile')
        kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
        for region in record['regions']:
            if region['crop_type'] == kind and cfg.image_variant not in region['images']:
                raise ValueError(f"Missing variant {cfg.image_variant} for {region['id']}")

def dataset_ocr_page(cfg, source_sha, page):
    """Load prepared source with inference selection attached; never render a PDF."""
    root, descriptor = dataset_open(cfg.image_dataset)
    if cfg.image_dataset_key and descriptor['snapshot_id'] != cfg.image_dataset_key:
        raise ValueError('Prepared input snapshot changed since OCR plan')
    record = dataset_page(root, descriptor, source_sha, page)
    if record is None:
        if cfg.image_dataset_fallback:
            return None
        raise ValueError(f'Prepared dataset missing page {page + 1}')
    layout = dataset_validate_page(root, record, check_pixels=False)
    image = dataset_load_image(root, record['source'])
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    regions = [r for r in record['regions'] if r['crop_type'] == kind]
    if not regions:
        raise ValueError(f'Prepared crop type missing: {kind}')
    for region in regions:
        dataset_load_image(root, region['images'][cfg.image_variant])
    image.info['prepared_dataset'] = {'root': str(root), 'snapshot_id': descriptor['snapshot_id'], 'source_sha256': source_sha, 'source': record['source'], 'layout': layout, 'regions': regions, 'variant': cfg.image_variant, 'crop_type': cfg.image_crop_type, 'acquisition': json.loads(dataset_read(root, record['acquisition']))}
    return (image, dataset_path(root, record['source']['path']))

def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages='1-2', limit=2, max_books=1, profile_dir=None):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or (max_books is not None and max_books < 1):
        raise ValueError('limit/max_books must be positive or None')
    entries = select_entries(list_pdfs(input_dir), books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = ([], limit)
    for entry in entries:
        wanted = parse_pages(pages, entry['n_pages'])
        if remaining is not None:
            wanted = wanted[:remaining]
            remaining -= len(wanted)
        if wanted:
            selected.append((entry, wanted))
    if not selected:
        raise ValueError('No pages selected')
    root = Path(output_dir).resolve()
    stats = {'pages': 0, 'source_reused': 0, 'layout_reused': 0, 'variants_reused': 0}
    started, environment, code = (time.monotonic(), dataset_environment(), dataset_code_signature())
    page_entries, manifest_rows, configurations = ([], [], {})
    with dataset_lock(root):
        marker = root / 'dataset.json'
        if marker.exists():
            dataset_open(root)
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir) / (entry['stem'] + '.json')).read_text(encoding='utf-8-sig'))
                for key in ('dpi', 'reading_direction'):
                    if key in profile:
                        setattr(cfg, key, profile[key])
            cfg.__post_init__()
            configurations[entry['name']] = asdict(cfg)
            source_sha = dataset_file_hash(entry['path'])
            doc = pdfium.PdfDocument(entry['path'])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({'source_sha256': source_sha, 'page': page, 'acquisition': cfg.acquisition, 'dpi': cfg.dpi, 'max_source_pixels': cfg.max_source_pixels, 'code': code, 'environment': environment})
                    source_receipt = f'sources/{source_key}.json'
                    try:
                        cached = json.loads(dataset_path(root, source_receipt).read_bytes())
                        if cached['key'] != source_key:
                            raise ValueError('Source cache key mismatch')
                        image = dataset_load_image(root, cached['source'])
                        acquisition = json.loads(dataset_read(root, cached['acquisition']))
                        stats['source_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page, dpi=cfg.dpi, max_source_pixels=cfg.max_source_pixels, strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root, image)
                        acquisition_ref = dataset_write(root, f'sources/{source_key}.acquisition.json', dataset_json_bytes(acquisition))
                        cached = {'key': source_key, 'source': source_asset, 'acquisition': acquisition_ref}
                        dataset_write(root, source_receipt, dataset_json_bytes(cached))
                    layout_options = {k: v for k, v in asdict(cfg).items() if k not in {'dpi', 'acquisition', 'max_source_pixels', 'variants', 'variant_options'}}
                    layout_options['page_column_cuts'] = {str(page + 1): cfg.page_column_cuts.get(str(page + 1), cfg.column_cuts)}
                    layout_key = dataset_hash({'source': cached['source']['sha256'], 'options': layout_options, 'code': code})
                    layout_relative = f'layouts/{layout_key}.json'
                    try:
                        layout_data = dataset_path(root, layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout['key'] != layout_key:
                            raise ValueError('Layout key mismatch')
                        dataset_read(root, layout['analysis_mask'])
                        dataset_read(root, layout['preview'])
                        stats['layout_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        layout, mask, preview = dataset_plan_layout(image, cfg, page=page)
                        layout.update(key=layout_key, analysis_mask=dataset_store_image(root, mask), preview=dataset_store_image(root, preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root, layout_relative, layout_data)
                    layout_ref = {'path': layout_relative, 'sha256': hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout['regions']:
                        crop = image.crop(geometry['source_box'])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name, {})
                            variant_environment = dict(environment)
                            if name == 'clahe':
                                import cv2
                                variant_environment['opencv'] = cv2.__version__
                            key = dataset_hash({'source': cached['source']['sha256'], 'box': geometry['source_box'], 'variant': name, 'options': options, 'code': code, 'environment': variant_environment})
                            receipt = f'variants/{key}.json'
                            try:
                                variant = json.loads(dataset_path(root, receipt).read_bytes())
                                if variant['key'] != key:
                                    raise ValueError('Variant cache key mismatch')
                                dataset_load_image(root, variant)
                                stats['variants_reused'] += 1
                            except (OSError, ValueError, KeyError):
                                prepared, operation = dataset_variant(crop, name, options)
                                x1, y1, x2, y2 = geometry['source_box']
                                variant = {**dataset_store_image(root, prepared), 'key': key, 'operation': operation, 'image_to_source': [(x2 - x1) / prepared.width, 0, 0, (y2 - y1) / prepared.height, x1, y1]}
                                dataset_write(root, receipt, dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page + 1, 'page_index': page, 'acquisition_method': acquisition['method'], 'source_size': cached['source']['size'], 'source_image': cached['source'], 'analysis_size': layout['analysis_size'], 'analysis_to_source': layout['analysis_to_source'], 'source_to_pdf': acquisition['source_to_pdf'], 'region_id': geometry['id'], 'crop_type': geometry['crop_type'], 'source_box': geometry['source_box'], 'reading_order': geometry['reading_order'], 'reading_direction': cfg.reading_direction, 'variant': name, **variant, 'qc_warnings': acquisition['warnings'] + layout['warnings'] + [w for c in layout['cuts'] for w in c['warnings']], 'layout': layout_ref, 'acquisition': cached['acquisition'], 'code_sha256': code, 'processing_version': IMAGE_DATASET_VERSION})
                        regions.append({**geometry, 'images': variants})
                    record = {'input_key': dataset_hash({'source': source_key, 'layout': layout_key, 'variants': cfg.variants, 'options': cfg.variant_options}), 'source_sha256': source_sha, 'page': page, 'source': cached['source'], 'acquisition': cached['acquisition'], 'layout': layout_ref, 'regions': regions}
                    dataset_validate_page(root, record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root, f'pages/{record_key}.json', record_bytes)
                    page_entries.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page, 'record': record_ref})
                    stats['pages'] += 1
            finally:
                doc.close()
        manifest_data = b''.join((dataset_json_bytes(row) for row in manifest_rows))
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root, f'manifests/{manifest_sha}.jsonl', manifest_data)
        descriptor = {'format': IMAGE_DATASET_FORMAT, 'version': IMAGE_DATASET_VERSION, 'pages': page_entries, 'expected_pages': [[e['book'], e['source_sha256'], e['page']] for e in page_entries], 'manifest': manifest_ref, 'configurations': configurations, 'environment': environment, 'code_sha256': code, 'geometry': 'top_left_source_pixels; half_open_boxes', 'ocr_text_coverage': 'unverified'}
        descriptor['snapshot_id'] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root, snapshot, data)
        dataset_write(root, 'dataset.json', data)
    stats.update(snapshot_id=descriptor['snapshot_id'], dataset=str(root / snapshot), seconds=round(time.monotonic() - started, 3), manifest_rows=len(manifest_rows))
    return stats

def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    dataset_read(root, descriptor['manifest'])
    actual = [[e['book'], e['source_sha256'], e['page']] for e in descriptor['pages']]
    if actual != descriptor['expected_pages'] or len(set(map(tuple, actual))) != len(actual):
        raise ValueError('Dataset expected-page coverage mismatch')
    warnings = []
    for entry in descriptor['pages']:
        record = dataset_page(root, descriptor, entry['source_sha256'], entry['page'])
        layout = dataset_validate_page(root, record)
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}: {w}" for w in layout['warnings']))
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}/x{c['source_x']}: {w}" for c in layout['cuts'] for w in c['warnings']))
    return {'pages': len(actual), 'snapshot_id': descriptor['snapshot_id'], 'warnings': warnings, 'geometric_coverage': 1.0, 'ocr_text_coverage': 'unverified'}

def package_image_dataset(directory, archive):
    """ZIP_STORED PNGs; include only this validated snapshot's referenced files."""
    import zipfile
    root, descriptor = dataset_open(directory)
    validate_image_dataset(directory)
    archive = Path(archive).resolve()
    if archive.exists():
        raise FileExistsError(f'Archive already exists: {archive}')
    references = {descriptor['manifest']['path']}
    for entry in descriptor['pages']:
        references.add(entry['record']['path'])
        record = json.loads(dataset_read(root, entry['record']))
        references.update((record[k]['path'] for k in ('source', 'acquisition', 'layout')))
        layout = json.loads(dataset_read(root, record['layout']))
        references.update((layout[k]['path'] for k in ('analysis_mask', 'preview')))
        references.update((v['path'] for r in record['regions'] for v in r['images'].values()))
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
        for name in ['dataset.json', f"snapshots/{descriptor['snapshot_id']}.json", *sorted(references)]:
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.external_attr = 33188 << 16
            data = dataset_json_bytes(descriptor) if name == 'dataset.json' or name.startswith('snapshots/') else dataset_path(root, name).read_bytes()
            output.writestr(info, data)
    return {'archive': str(archive), 'files': len(references) + 2, 'bytes': archive.stat().st_size, 'snapshot_id': descriptor['snapshot_id']}

def add_image_dataset_commands(parser):
    actions = parser.add_subparsers(dest='image_action', required=True)
    prepare = actions.add_parser('prepare', help='prepare once; repeat to resume (default: one PDF/two pages)')
    prepare.add_argument('--input-dir', default='data')
    prepare.add_argument('--output-dir', required=True)
    prepare.add_argument('--config', help='JSON ImageDatasetConfig; no OCR/model settings')
    prepare.add_argument('--books', nargs='+', default=[])
    prepare.add_argument('--pages', default='1-2', help='1-based selection; empty string selects all pages')
    prepare.add_argument('--limit', type=int, default=2, help='total page limit; 0 means unlimited')
    prepare.add_argument('--max-books', type=int, default=1, help='0 means all selected PDFs')
    prepare.add_argument('--profile-dir', help='reuse only dpi/reading_direction from OCR book profiles')
    validate = actions.add_parser('validate', help='verify hashes, pixels, geometry, pages and references')
    validate.add_argument('dataset')
    package = actions.add_parser('package', help='package one validated snapshot for offline Kaggle input')
    package.add_argument('dataset')
    package.add_argument('archive')

def run_image_dataset_command(args):
    if args.image_action == 'prepare':
        config = ImageDatasetConfig(**json.loads(Path(args.config).read_text(encoding='utf-8-sig'))) if args.config else ImageDatasetConfig()
        result = prepare_image_dataset(args.input_dir, args.output_dir, config=config, books=args.books, pages=args.pages, limit=None if args.limit == 0 else args.limit, max_books=None if args.max_books == 0 else args.max_books, profile_dir=args.profile_dir)
    elif args.image_action == 'validate':
        result = validate_image_dataset(args.dataset)
    else:
        result = package_image_dataset(args.dataset, args.archive)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


IMAGE_DATASET_CODE_SIGNATURE = '77a207cd108f3a591b0c28a7510a3bd50968204d71e5bfb730962067af10ca10'

import argparse
import json
import os
import re
import signal
import tempfile
import threading
import time
from contextlib import contextmanager
from pathlib import Path
PADDLE_IGNORE = {'header', 'header_image', 'footer', 'footer_image', 'number', 'footnote', 'aside_text', 'image', 'chart', 'seal'}

def clean_degenerate(s: str, max_unit: int=40, min_chars: int=60, min_cycles: int=4) -> str:
    """Cắt đuôi lặp vô hạn (VLM kẹt vào 1 chu kỳ ngắn rồi lặp tới hết token)."""
    st = s.rstrip()
    for u in range(1, max_unit + 1):
        unit = st[-u:]
        if len(st) < min_chars or not unit.strip():
            continue
        n, i = (0, len(st))
        while i >= u and st[i - u:i] == unit:
            n, i = (n + 1, i - u)
        if n * u >= min_chars and n >= min_cycles:
            return st[:i].rstrip()
    return s

def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub('</t[dh]>', ' ', md)
    md = re.sub('</tr>|<br\\s*/?>', '\n', md)
    md = re.sub('<[^>]+>', '', md)
    md = re.sub('!\\[[^\\]]*\\]\\([^)]*\\)', '', md)
    out = []
    for ln in md.splitlines():
        ln = re.sub('^\\s{0,3}#{1,6}\\s*', '', ln)
        ln = re.sub('^\\s*[-*+]\\s+(?=\\S)', '', ln)
        ln = re.sub('(?:[ \\t,，]*\\(\\d{1,4},\\d{1,4}\\),\\(\\d{1,4},\\d{1,4}\\))+\\s*$', '', ln)
        ln = ln.replace('**', '').replace('__', '').strip()
        if ln.startswith('|') and ln.endswith('|'):
            ln = ' '.join((c.strip() for c in ln.strip('|').split('|') if c.strip()))
        if ln and (not re.fullmatch('[-|:\\s]+', ln)):
            out.append(ln)
    return out

def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if 'blocks' not in rec:
        return md_to_lines(rec['text'])
    return [ln for b in ordered_page_blocks(rec['blocks']) for ln in md_to_lines(b['text'])]

def ordered_page_blocks(blocks):
    """Use the same block sequence for exported text and segmented geometry."""
    ordered = sorted((b for b in blocks if b['order'] is not None), key=lambda b: b['order'])
    center = lambda b: ((b['box'][0] + b['box'][2]) / 2, (b['box'][1] + b['box'][3]) / 2)
    for b in (b for b in blocks if b['order'] is None):
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)
    return ordered
_DOCS = {}

def render(src: str, page: int, dpi: int=200, max_side: int=1600):
    """Trang PDF -> ảnh PIL (cạnh dài tối đa max_side)."""
    import pypdfium2 as pdfium
    if src not in _DOCS:
        _DOCS[src] = pdfium.PdfDocument(src)
    pdf_page = _DOCS[src][page]
    bitmap = None
    try:
        bitmap = pdf_page.render(scale=dpi / 72)
        img = bitmap.to_pil().convert('RGB')
    finally:
        if bitmap is not None:
            bitmap.close()
        pdf_page.close()
    original_size = img.size
    if max(img.size) > max_side:
        s = max_side / max(img.size)
        from PIL import Image
        img = img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.Resampling.LANCZOS)
    img.info['render'] = {'dpi': dpi, 'uncapped_size': list(original_size), 'size': list(img.size), 'downsampled': img.size != original_size, 'max_side': max_side}
    return img

class Dummy:
    """Engine không cần model, dùng để test toàn bộ pipeline trên CPU.

    Trả về đủ chữ Hán để bước dịch không lọc mất file.
    """

    def __init__(self, a):
        self.a = a

    def run(self, img_path):
        from PIL import Image
        with Image.open(img_path) as im:
            w, h = im.size
        text = f'測試頁面 {w}x{h}\n' + '黃帝內經素問' * 10
        return (text, {'hit_max': False, 'dummy': True})

class Hunyuan:

    def __init__(self, a):
        started = time.monotonic()
        print('[model] Hunyuan: bắt đầu import torch/transformers', flush=True)
        import torch
        from transformers import AutoProcessor
        try:
            from transformers import HunYuanVLForConditionalGeneration as M
        except ImportError:
            from transformers import AutoModelForImageTextToText as M
        self.torch, self.max_new, self.rep = (torch, a.max_new, a.rep_penalty)
        self.prompt = a.prompt or HUNYUAN_PROMPT
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        self.dev = self._device(a.device)
        dtype = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}[a.dtype]
        print(f'[model] Hunyuan: bắt đầu tải/đọc model {a.model} (dùng cache nếu có)', flush=True)
        revision = {'revision': a.model_revision} if getattr(a, 'model_revision', '') else {}
        self.proc = AutoProcessor.from_pretrained(a.model, trust_remote_code=True, use_fast=False, **revision)
        self.processor_default_size = dict(self.proc.image_processor.size)
        self.model = M.from_pretrained(a.model, dtype=dtype, trust_remote_code=True, **revision).to(self.dev).eval()
        print(f'[model] Hunyuan: model sẵn sàng trên {self.dev}, sau {time.monotonic() - started:.1f}s', flush=True)

    @staticmethod
    def _device(requested: str) -> str:
        import torch
        if requested == 'auto':
            return 'cuda' if torch.cuda.is_available() else 'cpu'
        return requested

    def run(self, img_path):
        torch = self.torch
        processor = self.proc.image_processor
        size = dict(self.processor_default_size)
        if self.model_min_pixels is not None:
            size['shortest_edge'] = self.model_min_pixels
        if self.model_max_pixels is not None:
            size['longest_edge'] = self.model_max_pixels
        processor.size = size
        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img_path}, {'type': 'text', 'text': self.prompt}]}]
        inputs = self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to(self.dev)
        grid = inputs.get('image_grid_thw')
        grids = grid.detach().cpu().tolist() if grid is not None else []
        patch = getattr(processor, 'patch_size', 16)
        model_input = {'processor': type(processor).__name__, 'effective_size': size, 'image_grid_thw': grids, 'processed_sizes_wh': [[int(row[2]) * patch, int(row[1]) * patch] for row in grids], 'pixel_values_shape': list(inputs['pixel_values'].shape) if 'pixel_values' in inputs else None, 'observation': 'processor_tensor' if grids else 'unavailable'}
        for k, v in list(inputs.items()):
            if torch.is_tensor(v) and v.is_floating_point():
                inputs[k] = v.to(self.model.dtype)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new, do_sample=False, repetition_penalty=self.rep)
        gen = out[:, inputs['input_ids'].shape[1]:]
        text = self.proc.batch_decode(gen, skip_special_tokens=True)[0]
        hit_max = int(gen.shape[1]) >= self.max_new
        return (text, {'hit_max': hit_max, 'model_input': model_input, 'quality_issues': ocr_quality_issues(text, {'hit_max': hit_max})})

class Paddle:

    def __init__(self, a):
        started = time.monotonic()
        print('[model] Paddle: bắt đầu import paddleocr', flush=True)
        from paddleocr import PaddleOCRVL
        print(f'[model] Paddle: bắt đầu tải/khởi tạo PaddleOCR-VL {a.paddle_version} (dùng cache nếu có)', flush=True)
        self.pipe = PaddleOCRVL(pipeline_version=a.paddle_version, **json.loads(a.paddle_args))
        self.max_new = a.max_new
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        print(f'[model] Paddle: model sẵn sàng, sau {time.monotonic() - started:.1f}s', flush=True)

    def run(self, img_path):
        kwargs = {'max_new_tokens': self.max_new}
        for key, value in (('min_pixels', self.model_min_pixels), ('max_pixels', self.model_max_pixels)):
            if value is not None:
                kwargs[key] = value
        res = list(self.pipe.predict(img_path, **kwargs))[0]
        text, extra = self.parse(res.json)
        return (text, {**extra, 'model_input': {'requested': kwargs, 'observation': 'native_internal_unavailable'}})

    @staticmethod
    def parse(j):
        """res.json của PaddleOCR-VL -> (text, {"blocks": vùng đã đọc chữ, "boxes": mọi khung layout})."""
        j = j.get('res', j) if isinstance(j, dict) else {}
        blocks = [{'label': b.get('block_label'), 'box': [round(x) for x in b.get('block_bbox', [0, 0, 0, 0])], 'order': b.get('block_order'), 'text': b.get('block_content') or ''} for b in j.get('parsing_res_list', []) if b.get('block_label') not in PADDLE_IGNORE]
        blocks = [b for b in blocks if b['text']]
        text = '\n'.join(page_lines({'blocks': blocks}))
        boxes = [{'label': b['label'], 'box': [round(x) for x in b['coordinate']], 'order': b.get('order'), 'score': round(b.get('score', 0), 2)} for b in (j.get('layout_det_res') or {}).get('boxes', [])]
        return (text, {'blocks': blocks, 'boxes': boxes, 'quality_issues': ocr_quality_issues(text, {})})

class PPOCR:
    """PP-OCRv6 medium, retaining genuine single-character word boxes only."""

    def __init__(self, a):
        from paddleocr import PaddleOCR
        options = dict(ocr_version='PP-OCRv6', text_detection_model_name='PP-OCRv6_medium_det', text_recognition_model_name='PP-OCRv6_medium_rec', use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
        options.update(json.loads(a.paddle_args))
        for key in ('use_doc_orientation_classify', 'use_doc_unwarping', 'use_textline_orientation'):
            if options.get(key):
                raise ValueError(f'{key} must be False for source-image character coordinates')
        self.pipe = PaddleOCR(**options)

    def run(self, img_path):
        kwargs = {'return_word_box': True}
        if getattr(self, 'preserve_input_resolution', False):
            from PIL import Image
            with Image.open(img_path) as image:
                kwargs.update(text_det_limit_side_len=(max(image.size) + 31) // 32 * 32, text_det_limit_type='max')
        results = list(self.pipe.predict(img_path, **kwargs))
        if len(results) != 1:
            raise ValueError('Expected exactly one PP-OCR page result')
        text, extra = self.parse(results[0].json)
        return (text, {**extra, 'model_input': {'requested': kwargs, 'observation': 'native_internal_unavailable'}})

    @staticmethod
    def parse(j):
        j = j.get('res', j)
        texts = j.get('rec_texts', [])
        polygons = j.get('rec_polys', [])
        words = j.get('text_word', [])
        regions = j.get('text_word_region', [])
        rectangles = j.get('text_word_boxes', [])
        characters, lines, offset = ([], [], 0)
        for index, text in enumerate(texts):
            filtered = comparison_text(text)
            line_words = words[index] if index < len(words) else []
            line_boxes = regions[index] if index < len(regions) else rectangles[index] if index < len(rectangles) else []
            if len(line_words) == len(line_boxes) and comparison_text(''.join(line_words)) == filtered:
                local_offset = 0
                for word, polygon in zip(line_words, line_boxes):
                    word = comparison_text(word)
                    if len(word) == 1:
                        characters.append({'offset': offset + local_offset, 'text': word, 'polygon': polygon, 'granularity': 'character', 'method': 'ppocr_word_box', 'estimated': True})
                    local_offset += len(word)
            lines.append({'text': text, 'polygon': polygons[index] if index < len(polygons) else None})
            offset += len(filtered)
        return ('\n'.join(texts), {'character_boxes': characters, 'text_lines': lines, 'coordinate_frame': 'input_image', 'hit_max': False})

class TongGu:
    """Use the official TongGu layout/block pipeline in its configured environment."""

    def __init__(self, a):
        import sys
        root = Path(a.tonggu_repo).expanduser().resolve() if a.tonggu_repo else None
        if root is None or not (root / 'tongguocr' / 'pipeline.py').is_file():
            raise ValueError('Set tonggu_repo to the official TongGuOCR checkout with installed dependencies/weights')
        sys.path.insert(0, str(root))
        from tongguocr.config import load_config
        from tongguocr.pipeline import run_pipeline_on_image
        self.predict = run_pipeline_on_image
        self.config = load_config(a.tonggu_config or None)
        self.default_image_limits = {key: self.config['ocr']['inference'].get(key, default) for key, default in (('image_min_pixels', 1024), ('image_max_pixels', 1048576))}
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        for group, key in (('layout_inference', 'detector_checkpoint_path'), ('layout_inference', 'detector_config_path')):
            path = Path(self.config[group][key])
            if not path.is_absolute():
                self.config[group][key] = str(root / path)
        model_path = Path(self.config['ocr']['transformers']['model_path'])
        if not model_path.is_absolute():
            self.config['ocr']['transformers']['model_path'] = str(root / model_path)
        self.max_new = a.max_new
        self.temporary = tempfile.TemporaryDirectory(prefix='vtm-tonggu-')
        self.prepare_models()

    def prepare_models(self):
        """Fill the pinned upstream caches before the per-page deadline starts."""
        from tongguocr.layout.extractor import LineBBoxExtractor, _EXTRACTOR_CACHE
        started = time.monotonic()
        print('[model] TongGu: bắt đầu nạp detector layout', flush=True)
        layout = self.config['layout_inference']
        key = '|'.join((str(layout[field]) for field in ('detector_config_path', 'detector_checkpoint_path', 'device', 'detector_score_threshold')))
        if key not in _EXTRACTOR_CACHE:
            _EXTRACTOR_CACHE[key] = LineBBoxExtractor(self.config)
        if self.config['ocr']['backend'] == 'transformers':
            from tongguocr.ocr.transformers_backend import _load_transformers_runtime
            print('[model] TongGu: bắt đầu nạp model OCR', flush=True)
            backend = self.config['ocr']['transformers']
            _load_transformers_runtime(backend['model_path'], backend.get('device', 'cuda:0'))
        print(f'[model] TongGu: model sẵn sàng, sau {time.monotonic() - started:.1f}s; thời gian nạp không tính vào timeout trang.', flush=True)

    def run(self, img_path):
        self.config['ocr']['inference']['max_tokens'] = self.max_new
        for key, value in (('image_min_pixels', getattr(self, 'model_min_pixels', None)), ('image_max_pixels', getattr(self, 'model_max_pixels', None))):
            if value is not None or hasattr(self, 'default_image_limits'):
                self.config['ocr']['inference'][key] = value if value is not None else self.default_image_limits[key]
        output = Path(self.temporary.name) / str(time.time_ns())
        result = self.predict(img_path, output, self.config, visualize=False, crop_blocks=True)
        if result.status != 'ok':
            raise RuntimeError(f'TongGuOCR returned status={result.status}')
        return (result.ocr_text, {'tonggu_layout': result.line_bboxes, 'tonggu_blocks': result.blocks, 'hit_max': None, 'character_boxes': [], 'model_input': {'requested': {key: self.config['ocr']['inference'].get(key) for key in ('image_min_pixels', 'image_max_pixels')}, 'observation': 'native_internal_unavailable'}, 'quality_issues': ocr_quality_issues(result.ocr_text, {})})
ENGINES = {'hunyuan': Hunyuan, 'paddle': Paddle, 'ppocr': PPOCR, 'tonggu': TongGu, 'dummy': Dummy}
PAGE_TIMEOUT_EXIT = 124

@contextmanager
def page_deadline(seconds: int, on_hard_timeout):
    """Unix: interrupt the page; Windows: stop the worker so CUDA cannot keep running."""
    if seconds <= 0:
        yield
        return
    if hasattr(signal, 'SIGALRM'):

        def expire(signum, frame):
            raise TimeoutError(f'quá {seconds}s')
        previous = signal.signal(signal.SIGALRM, expire)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, previous)
    else:

        def expire():
            try:
                on_hard_timeout()
            finally:
                os._exit(PAGE_TIMEOUT_EXIT)
        timer = threading.Timer(seconds, expire)
        timer.daemon = True
        timer.start()
        try:
            yield
        finally:
            timer.cancel()
            timer.join()

def result_file(out: Path, book: str, name: str, shard: int) -> Path:
    return out / book / f'{name}_shard{shard}.jsonl'

def load_results(out: Path, book: str, name: str) -> dict[int, dict]:
    """{trang: bản ghi} của 1 sách, gộp mọi shard (dòng hỏng thì bỏ)."""
    recs = {}
    for f in sorted((out / book).glob(f'{name}_shard*.jsonl')):
        with open(f, encoding='utf-8') as stream:
            for line in stream:
                try:
                    r = json.loads(line)
                    recs[r['page']] = r
                except (json.JSONDecodeError, KeyError):
                    pass
    return recs


worker_ENGINES = ENGINES
worker__DOCS = _DOCS
worker_load_results = load_results
worker_page_deadline = page_deadline
worker_page_lines = page_lines
worker_render = render
worker_result_file = result_file

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from fnmatch import fnmatch
from datetime import datetime, timezone
from pathlib import Path
from tqdm.auto import tqdm

def export_ocr(run_dir: str | Path, books: dict, name: str) -> dict:
    run_dir = Path(run_dir)
    raw_dir, txt_dir = (run_dir / 'ocr_raw', run_dir / 'txt')
    txt_dir.mkdir(parents=True, exist_ok=True)
    total_ok = total = total_missing = 0
    for b, (_, sel) in books.items():
        recs = worker_load_results(raw_dir, b, name)
        pages = ['\n'.join(worker_page_lines(recs[p])) for p in sel if p in recs]
        (txt_dir / f'{b}.txt').write_text('\n\n'.join((p for p in pages if p)) + '\n', encoding='utf-8')
        miss = len(sel) - len(pages)
        total_ok += len(pages)
        total += len(sel)
        total_missing += miss
        print(f'{b:<50} {len(pages)}/{len(sel)} trang' + (f'  -- THIẾU {miss} trang' if miss else ''))
    return {'pages': total, 'done': total_ok, 'missing': total_missing}
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import uuid
from tqdm.auto import tqdm

def now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)
import os
from pathlib import Path, PureWindowsPath
import re

def to_wsl_path(value: str | Path) -> str:
    value = str(value)
    if value.startswith('/'):
        return value
    path = PureWindowsPath(value)
    if not re.fullmatch('[A-Za-z]:', path.drive) or not path.is_absolute():
        raise ValueError(f'WSL requires an absolute drive path or Linux path: {value}')
    return '/mnt/' + path.drive[0].lower() + '/' + '/'.join(path.parts[1:])

def runtime_path(value: str | Path) -> Path:
    value = str(value)
    return Path(to_wsl_path(value)) if os.name != 'nt' and re.match('^[A-Za-z]:[\\\\/]', value) else Path(value)
import argparse
import gc
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import time
import traceback
from types import SimpleNamespace
import uuid

def create_engine(engine: str, cfg: OCRConfig):
    args = SimpleNamespace(model=cfg.model, dtype=cfg.dtype, device=cfg.device, model_revision=cfg.model_revision, tonggu_repo=cfg.tonggu_repo, tonggu_config=cfg.tonggu_config, max_new=cfg.max_new_tokens, rep_penalty=cfg.repetition_penalty, prompt=cfg.prompt, paddle_version=cfg.paddle_version, model_min_pixels=cfg.model_min_pixels, model_max_pixels=cfg.model_max_pixels, paddle_args=json.dumps(cfg.paddle_args))
    return worker_ENGINES[engine](args)

def append_record(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        with path.open('r+b') as stream:
            stream.seek(-1, os.SEEK_END)
            if stream.read(1) != b'\n':
                stream.seek(0)
                contents = stream.read()
                boundary = contents.rfind(b'\n') + 1
                tail = contents[boundary:]
                try:
                    json.loads(tail)
                except (json.JSONDecodeError, UnicodeDecodeError):
                    path.with_suffix(f'.interrupted-{time.time_ns()}.log').write_bytes(tail)
                    stream.truncate(boundary)
                else:
                    stream.seek(0, os.SEEK_END)
                    stream.write(b'\n')
    with path.open('ab') as stream:
        stream.write((json.dumps(record, ensure_ascii=False) + '\n').encode('utf-8'))
        stream.flush()
        os.fsync(stream.fileno())

def corpus_book_plan(pair, engine, attempt):
    directory = runtime_path(pair[engine])
    manifest_file = directory / 'run_manifest.json'
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    cfg = OCRConfig(**manifest['config'])
    name = cfg.run_name or engine
    raw = directory / 'ocr_raw'
    raw_book = raw / pair['book']
    raw_book.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(line) for line in (directory / 'tasks.jsonl').read_text(encoding='utf-8').splitlines() if line]
    done = worker_load_results(raw, pair['book'], name)
    failed = set()
    errors = raw_book / f'{name}_errors.jsonl'
    if errors.exists():
        for line in errors.read_text(encoding='utf-8').splitlines():
            try:
                error = json.loads(line)
                if error.get('attempt') == attempt:
                    failed.add(int(error['page']))
            except (json.JSONDecodeError, KeyError, ValueError):
                pass
    todo = [task for task in tasks if task['page'] not in done and task['page'] not in failed]
    return dict(pair=pair, directory=directory, manifest_file=manifest_file, manifest=manifest, cfg=cfg, name=name, raw=raw, errors=errors, tasks=tasks, done=done, todo=todo)

def corpus_page_image(campaign, manifest, book, task, cfg, render, temporary):
    """Share lossless renders across models; a damaged cache is rendered again."""
    from PIL import Image
    source_sha = manifest.get('books', {}).get(book, {}).get('sha256')
    if cfg.image_dataset:
        prepared = dataset_ocr_page(cfg, source_sha, task['page'])
        if prepared is not None:
            image, path = prepared
            return (image, path, True)
    key = {'source_sha256': source_sha, 'page': task['page'], 'dpi': cfg.dpi, 'max_side': cfg.max_side, 'renderer': 'pdfium-rgb-lanczos-v2'}
    cached = bool(campaign.get('cache_renders') and source_sha)
    if cached:
        cache_root = runtime_path(campaign['render_cache']).resolve()
        if not cache_root.is_relative_to(runtime_path(cfg.workdir).resolve()):
            raise ValueError('Render cache must stay within the campaign workdir')
        path = cache_root / source_sha / f'{cfg.dpi}_{cfg.max_side}' / f"p{task['page'] + 1:05d}.png"
        metadata = path.with_suffix('.json')
        try:
            if path.is_symlink() or metadata.is_symlink():
                raise ValueError('Cache symlink')
            info = json.loads(metadata.read_text(encoding='utf-8'))
            data = path.read_bytes()
            if info['key'] != key or hashlib.sha256(data).hexdigest() != info['sha256']:
                raise ValueError('Render cache mismatch')
            with Image.open(io.BytesIO(data)) as loaded:
                image = loaded.convert('RGB')
            if list(image.size) != info['size']:
                raise ValueError('Render cache size mismatch')
            image.info['render'] = info.get('render', {})
            return (image, path, True)
        except (OSError, ValueError, KeyError):
            pass
    else:
        path = temporary
    image = render(str(runtime_path(task['src'])), task['page'], cfg.dpi, cfg.max_side)
    path.parent.mkdir(parents=True, exist_ok=True)
    destination = path.with_suffix('.png.tmp')
    image.save(destination, format='PNG', compress_level=1)
    destination.replace(path)
    if cached:
        write_json(metadata, {'key': key, 'size': list(image.size), 'render': image.info.get('render', {}), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    return (image, path, False)

def process_corpus(manifest_path: str | Path, engine: str, *, engine_factory=None, renderer=None, attempt_id=None):
    factory = engine_factory or create_engine
    render = renderer or worker_render
    campaign = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    attempt = attempt_id or os.environ.get('VTM_OCR_ATTEMPT') or uuid.uuid4().hex
    plans = [corpus_book_plan(pair, engine, attempt) for pair in campaign['pairs']]
    completed = sum((len({task['page'] for task in plan['tasks']} & plan['done'].keys()) for plan in plans))
    progress_file = runtime_path(Path(manifest_path).parent) / f'{engine}_progress.json'

    def update_progress():
        write_json(progress_file, {'engine': engine, 'attempt': attempt, 'pages': campaign['total_pages'], 'done': completed, 'missing': campaign['total_pages'] - completed, 'updated': now()})
    update_progress()
    backend = None
    try:
        with tempfile.TemporaryDirectory(prefix='vtm-corpus-') as temporary:
            image_path = Path(temporary) / 'page.png'
            for plan in plans:
                pair, directory, manifest_file, manifest, cfg, name, raw, errors, tasks, done, todo = (plan[key] for key in ('pair', 'directory', 'manifest_file', 'manifest', 'cfg', 'name', 'raw', 'errors', 'tasks', 'done', 'todo'))
                if todo and backend is None:
                    backend = factory(engine, cfg)
                    print(f'[corpus] {engine}: model đã tải; dùng chung cho toàn bộ sách đã chọn.', flush=True)
                if backend is not None:
                    backend.max_new = cfg.max_new_tokens
                    backend.model_min_pixels, backend.model_max_pixels = (cfg.model_min_pixels, cfg.model_max_pixels)
                    backend.preserve_input_resolution = cfg.segmentation != 'none' or bool(cfg.image_dataset)
                    if engine == 'hunyuan':
                        backend.rep, backend.prompt = (cfg.repetition_penalty, cfg.prompt or HUNYUAN_PROMPT)
                manifest.update(status='running', updated=now())
                write_json(manifest_file, manifest)
                print(f"[corpus] {engine} / {pair['book']}: {len(tasks)} trang được chọn, còn {len(todo)} trang cần OCR.", flush=True)
                for task in todo:
                    page, started = (task['page'], time.monotonic())
                    print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: bắt đầu (timeout={cfg.page_timeout}s).", flush=True)

                    def on_timeout():
                        append_record(errors, {'page': page, 'error': 'TimeoutError', 'message': f'quá {cfg.page_timeout}s; worker phải khởi động lại', 'attempt': attempt, 'timeout_seconds': cfg.page_timeout, 'seconds': round(time.monotonic() - started, 1), 'updated': now()})
                    try:
                        with worker_page_deadline(cfg.page_timeout, on_timeout):
                            image, rendered_path, cache_hit = corpus_page_image(campaign, manifest, pair['book'], task, cfg, render, image_path)
                            rendered_at = time.monotonic()
                            text, extra = run_prepared_page(backend, image, rendered_path, cfg, page=page)
                        record = {'page': page, 'text': text, 'w': image.width, 'h': image.height, 'sec': round(time.monotonic() - started, 1), **extra, 'render_cache_hit': cache_hit, 'render_sec': round(rendered_at - started, 3), 'inference_sec': round(time.monotonic() - rendered_at, 3), 'image_input': 'prepared' if image.info.get('prepared_dataset') else 'legacy_missing_page_fallback' if cfg.image_dataset else 'legacy'}
                        preparation = extra.get('image_preparation')
                        if preparation:
                            regions = preparation['regions']
                            dimensions = ', '.join(('x'.join(map(str, r['input_size'])) for r in regions))
                            observed = [r.get('processor_size') for r in regions]
                            processor = ', '.join(('x'.join(map(str, size)) for size in observed)) if all(observed) else 'unobserved native tensors'
                            print(f'[image] {image.width}x{image.height} -> {len(regions)} input(s): {dimensions}; processor: {processor}', flush=True)
                        append_record(worker_result_file(raw, pair['book'], name, 0), record)
                        done[page] = record
                        completed += 1
                        update_progress()
                        print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: xong sau {record['sec']}s.", flush=True)
                    except Exception as exc:
                        append_record(errors, {'page': page, 'error': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc(), 'attempt': attempt, 'timeout_seconds': cfg.page_timeout, 'seconds': round(time.monotonic() - started, 1), 'updated': now()})
                        detail = str(exc).replace('\n', ' ')[:300]
                        print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: lỗi {type(exc).__name__}: {detail}; log: {errors}", flush=True)
                wanted = {t['page'] for t in tasks}
                got = len(wanted & done.keys())
                counts = {'pages': len(tasks), 'done': got, 'missing': len(tasks) - got}
                if cfg.export:
                    export_ocr(directory, {pair['book']: (tasks[0]['src'], sorted(wanted))}, name)
                manifest.update(status='done' if got == len(tasks) else 'partial', counts=counts, updated=now())
                write_json(manifest_file, manifest)
                print(f"[corpus] {engine} / {pair['book']}: xong {got}/{len(tasks)} trang.", flush=True)
    finally:
        if backend is not None:
            del backend
        for document in worker__DOCS.values():
            document.close()
        worker__DOCS.clear()
        gc.collect()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="One isolated OCR model per process")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--engine", choices=["ppocr", "tonggu", "paddle", "hunyuan"], required=True)
    parser.add_argument("--token")
    parser.add_argument("--pid-file")
    parser.add_argument("--attempt")
    args = parser.parse_args()
    if args.pid_file:
        if os.getpgrp() != os.getpid():
            os.setsid()
        marker = Path(args.pid_file)
        write_json(marker, {"token": args.token, "pgid": os.getpgrp()})
    process_corpus(args.manifest, args.engine, attempt_id=args.attempt)


## 3. Setup và kiểm tra

Kiểm tra đường dẫn trước khi cài dependency. Windows dùng `.venv` hiện tại;
Kaggle chuẩn bị runtime Paddle/TongGu từ packages offline. Runtime khỏe được dùng lại.
Đọc checkpoint chỉ cần hỗ trợ PDFium, không cài/build model hay chạy GPU.


In [ ]:
if RUN_OCR and not MODELS_DIR:
    raise ValueError("Chưa điền MODELS_DIR ở bước 1: đường dẫn Dataset models.")

if EXPORTED_LOG_DIR:
    if RUN_OCR:
        raise ValueError("Đặt RUN_OCR=False để so sánh log đã có, hoặc để EXPORTED_LOG_DIR rỗng khi chạy OCR.")
    imported_manifest = import_ocr_logs(EXPORTED_LOG_DIR, WORK_DIR,
        input_dir=DATA_DIR or None, books=BOOKS, max_books=MAX_BOOKS, pages=PAGES, limit=LIMIT)
    ADVANCED = {**ADVANCED, "existing_manifest": str(imported_manifest)}

session = setup_ocr(
    packages_dir=PACKAGES_DIR, input_dir=DATA_DIR, weights=WEIGHTS,
    books=BOOKS, max_books=MAX_BOOKS, pages=PAGES, limit=LIMIT,
    run_ocr=RUN_OCR, work_dir=WORK_DIR,
    advanced={**ADVANCED, "ocr": {**IMAGE_PREPARATION,
        "image_dataset": str(PREPARED_IMAGES), "image_variant": PREPARED_VARIANT,
        "image_crop_type": PREPARED_CROP_TYPE, "image_dataset_fallback": PREPARED_MISSING_FALLBACK,
        **ADVANCED.get("ocr", {})}},
)


## 4. Chạy OCR

Bốn model chạy lần lượt trên cùng tập trang. Checkpoint được lưu sau mỗi trang;
chạy lại cùng cấu hình dùng kết quả đã có.
Trang lỗi được thử lại trong lượt chạy mới; trang thành công được giữ nguyên.
TongGu nạp model trước timeout trang; mặc định giới hạn 1.800 giây/trang.
Ảnh PNG được cache lossless giữa các model. Không dựng HTML/crop trong lúc chạy model.
Crop lấy từ ảnh render gốc, không chồng lặp. Hunyuan đọc từng nhóm cột; auto giữ
layout/block gốc của PaddleOCR-VL và TongGu. PP-OCR yêu cầu detector giữ độ phân giải crop.
Checkpoint có `image_preparation`: crop/input pixels, scale, văn bản và raw từng vùng.
Hunyuan ghi kích thước tensor sau processor; native engine ghi rõ khi không quan sát được tensor.
Đổi layout/pixel budget tạo run mới. `resolution_strict=True` dừng trang cần thu nhỏ;
chế độ False giữ output cùng cảnh báo trong checkpoint. Không tự loại chữ trùng.


In [ ]:
stage_results = run_ocr_models(session)


## 5. Xem kết quả

Xem bảng theo trang, dự đoán bốn model và ảnh cắt từng cụm bất đồng, kèm ngữ cảnh có khung đỏ.
**Page-level consensus chạy trước Character-level consensus**, tự bật khi so checkpoint.
Cột **Dấu hiệu cấp trang** chỉ model nghi thiếu/lặp/dư vùng hoặc sai thứ tự đọc.
Mốc chữ khớp được tìm độc lập với cách xuống dòng; đa số cần 3/4 model.
Thiếu/dư cần thêm bằng chứng mốc hai đầu; đoạn đọc khác không tự coi là sinh thêm văn bản.
Trường hợp hòa, thiếu bằng chứng hoặc vượt giới hạn phân tích giữ trạng thái chưa xác định.
Output có lỗi cấu trúc chưa sửa và được đa số đối chứng sẽ không bỏ phiếu ký tự.
Output gốc được giữ; PP-OCR đã sửa thứ tự vẫn được ghi lỗi ở luồng gốc và có thể bỏ phiếu sau sửa.
PP-OCR tự nhận diện trang có cột dọc và xếp phải sang trái, trong cột từ trên xuống.
Offset chữ và khung ảnh được chuyển cùng nhau; output gốc vẫn được lưu riêng.
Căn chỉnh dùng cụm chữ khớp duy nhất trong từng cột làm mốc khi đủ bằng chứng; nếu không thì dùng căn chỉnh chuỗi toàn trang.
Các mốc này chỉ là bằng chứng văn bản, chưa xác nhận cùng vị trí ảnh giữa các model.
Output lặp, nghi thiếu bất thường hoặc chạm giới hạn token được đánh dấu **không bỏ phiếu**.
Ngưỡng đa số vẫn tính trên cả bốn model. Số bất đồng không phải số chữ OCR sai.
Khung dòng chỉ dùng làm ngữ cảnh khi chưa định vị được chữ; không suy ra khung chữ từ cả dòng.
Đổi `REVIEW_BOOK`, `REVIEW_PAGE`, `REVIEW_VIEW`, `MAX_REVIEW_ROWS` ở bước 1 để chọn phần cần xem.


In [ ]:
result = compare_ocr_results(session)
if PREVIEW_CONFLICTS:
    review_character_conflicts(session, book=REVIEW_BOOK, page=REVIEW_PAGE, max_characters=MAX_REVIEW_ROWS, view=REVIEW_VIEW)


## 6. Xuất HTML, JSONL, ảnh bất đồng và ZIP

Mỗi sách có một thư mục `OUTPUT_DIR/<tên_sách>/`, mở `index.html` để đọc offline.
ZIP nằm bên cạnh thư mục; ảnh PNG giữ nguyên, văn bản nén nhẹ. Trang/model thiếu output
được ghi rõ trong HTML và manifest. Thư mục checkpoint và cache ảnh được giữ lại.
Xuất ảnh chỉ render các trang có vùng đã định vị được, mỗi trang một lần.
HTML có mục **Đồng thuận cấp trang**: model, dấu hiệu, vùng và model đối chứng.
Nếu có tọa độ, ảnh tổng quan đánh số vùng; mốc chữ chưa có tọa độ không được vẽ khung.
`comparison.jsonl` lưu đầy đủ `page_consensus` để rà soát lại mà không chạy model.


In [ ]:
exports = export_ocr_results(session, output_dir=OUTPUT_DIR,
    include_crops=EXPORT_CROPS, create_zip=CREATE_ZIP) if EXPORT_RESULTS else []
for artifact in exports:
    if artifact["zip_path"]:
        display(FileLink(os.path.relpath(artifact["zip_path"])))


Đồng thuận là bằng chứng để rà soát, không phải độ chính xác hay bản OCR đã chốt.
Chỗ thiếu tọa độ đáng tin cậy được ghi chưa xác định. Không tự dịch chữ còn tranh chấp.
Models/packages/PDF hiện có dùng lại; sửa logic chỉ cần cập nhật notebook.
Ảnh nhúng PDF 1 vốn chỉ 600 × 450 pixel; tăng DPI không khôi phục nét chữ đã mất.
`ADVANCED["reading_order"]`: `"auto"` (mặc định), `"vertical_rtl"` hoặc `"detection"`.
